<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/03_GES_future_instability_outcomes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# STAGE 5A — STEP 1
# VERIFY FROZEN OUTCOME-CONSTRUCTION INPUTS
#
# Purpose:
#   1. Mount Google Drive.
#   2. Locate the accepted T0 cohort, accepted T1 cohort,
#      and frozen Stage 3H linkage artifacts.
#   3. Recalculate and verify their SHA-256 checksums.
#   4. Confirm expected Parquet row and column counts.
#
# Leakage boundary:
#   - No Stage 4 GES score table is loaded.
#   - No future-instability outcome is created.
#   - No classification mapping is changed.
#   - No temporal model performance is examined.
#   - No file is written or overwritten.
# ==================================================================================================

from pathlib import Path
import hashlib

import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Mount Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise RuntimeError(
        "Google Drive could not be mounted."
    )


# --------------------------------------------------------------------------------------------------
# 2. Frozen project locations
# --------------------------------------------------------------------------------------------------

BASE_DIR = (
    DRIVE_ROOT
    / "GES_RAG_Temporal_Study"
)

T0_PATH = (
    BASE_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    BASE_DIR
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3H_LINKAGE_PATH = (
    BASE_DIR
    / "data_processed"
    / "stage3_crosswalk"
    / "stage3h_final_t0_t1_linkage_v1.parquet"
)

STAGE3H_POLICY_PATH = (
    BASE_DIR
    / "configs"
    / "stage3_crosswalk"
    / "stage3h_linkage_and_censoring_policy_v1.json"
)

STAGE3H_FREEZE_MANIFEST_PATH = (
    BASE_DIR
    / "configs"
    / "stage3_crosswalk"
    / "stage3h_final_linkage_freeze_manifest_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen accepted SHA-256 values
# --------------------------------------------------------------------------------------------------

EXPECTED_SHA256 = {
    "T0 accepted cohort": (
        "f6b6760b2ad6e4352e3bdecdeaf89827"
        "e8a514b031abf2373bb17568d999466d"
    ),
    "T1 accepted cohort": (
        "5713a11bdbf4804758cc011f9b2f302a"
        "fc91fa1f88c1b178d675c28bb277d37c"
    ),
    "Stage 3H final linkage": (
        "77d0522af5ec3ac0938a03aecdb9ebd2"
        "30d6cafae8e8ded7a8de1fd462cfbe75"
    ),
    "Stage 3H linkage policy": (
        "1da800feb912afa94a5e9eee2e7a23795"
        "c398e640575b1bbf3be68066f39e099"
    ),
    "Stage 3H freeze manifest": (
        "5b8dd32aa03456eb5a340c4ae5ff7bc1"
        "583752151dd555f969accb91d673bca0"
    ),
}

INPUT_PATHS = {
    "T0 accepted cohort": T0_PATH,
    "T1 accepted cohort": T1_PATH,
    "Stage 3H final linkage": STAGE3H_LINKAGE_PATH,
    "Stage 3H linkage policy": STAGE3H_POLICY_PATH,
    "Stage 3H freeze manifest": STAGE3H_FREEZE_MANIFEST_PATH,
}


# --------------------------------------------------------------------------------------------------
# 4. SHA-256 helper
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading the complete file
    into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 5. Verify existence and checksums
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("STAGE 5A — FROZEN INPUT VERIFICATION")
print("=" * 100)

observed_hashes = {}

for artifact_name, artifact_path in INPUT_PATHS.items():

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Required frozen artifact was not found:\n"
            f"{artifact_path}"
        )

    observed_sha256 = calculate_sha256(
        artifact_path
    )

    expected_sha256 = EXPECTED_SHA256[
        artifact_name
    ]

    status = (
        "PASS"
        if observed_sha256 == expected_sha256
        else "FAIL"
    )

    observed_hashes[artifact_name] = (
        observed_sha256
    )

    print(f"\n{status} — {artifact_name}")
    print("Path:    ", artifact_path)
    print("Expected:", expected_sha256)
    print("Observed:", observed_sha256)

    if observed_sha256 != expected_sha256:
        raise RuntimeError(
            f"Checksum mismatch for {artifact_name}.\n"
            "Stop Stage 5 and investigate the artifact."
        )


# --------------------------------------------------------------------------------------------------
# 6. Verify Parquet metadata
# --------------------------------------------------------------------------------------------------

EXPECTED_PARQUET_METADATA = {
    "T0 accepted cohort": {
        "path": T0_PATH,
        "rows": 71_659,
        "columns": 34,
    },
    "T1 accepted cohort": {
        "path": T1_PATH,
        "rows": 100_920,
        "columns": 36,
    },
    "Stage 3H final linkage": {
        "path": STAGE3H_LINKAGE_PATH,
        "rows": 71_659,
        "columns": None,
    },
}

print("\n" + "=" * 100)
print("PARQUET METADATA VERIFICATION")
print("=" * 100)

for artifact_name, specification in (
    EXPECTED_PARQUET_METADATA.items()
):

    parquet_file = pq.ParquetFile(
        specification["path"]
    )

    observed_rows = (
        parquet_file.metadata.num_rows
    )

    observed_columns = (
        parquet_file.metadata.num_columns
    )

    expected_rows = specification["rows"]
    expected_columns = specification["columns"]

    row_status = (
        "PASS"
        if observed_rows == expected_rows
        else "FAIL"
    )

    column_status = (
        "NOT FIXED"
        if expected_columns is None
        else (
            "PASS"
            if observed_columns == expected_columns
            else "FAIL"
        )
    )

    print(f"\n{artifact_name}")
    print(
        f"Rows:    {observed_rows:,} "
        f"| Expected: {expected_rows:,} "
        f"| {row_status}"
    )

    print(
        f"Columns: {observed_columns} "
        f"| Expected: "
        f"{expected_columns if expected_columns is not None else 'not prespecified here'} "
        f"| {column_status}"
    )

    if observed_rows != expected_rows:
        raise RuntimeError(
            f"Unexpected row count for {artifact_name}."
        )

    if (
        expected_columns is not None
        and observed_columns != expected_columns
    ):
        raise RuntimeError(
            f"Unexpected column count for {artifact_name}."
        )


# --------------------------------------------------------------------------------------------------
# 7. Final Stage 5A Step 1 decision
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("STAGE 5A STEP 1 DECISION")
print("=" * 100)

print("PASS_STAGE5A_FROZEN_INPUTS_VERIFIED")
print()
print("T0 rows verified:              71,659")
print("T1 rows verified:              100,920")
print("Stage 3H linkage rows:         71,659")
print("Expected accepted links:       70,583")
print("Expected censored records:      1,076")
print()
print("Stage 4 GES scores loaded:     NO")
print("Future outcomes created:       NO")
print("Temporal performance examined: NO")
print("Files written or modified:     NO")

Mounted at /content/drive
STAGE 5A — FROZEN INPUT VERIFICATION

PASS — T0 accepted cohort
Path:     /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_2.parquet
Expected: f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d
Observed: f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d

PASS — T1 accepted cohort
Path:     /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t1_rcv_target_genes_harmonized_v1.parquet
Expected: 5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
Observed: 5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c

PASS — Stage 3H final linkage
Path:     /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage3_crosswalk/stage3h_final_t0_t1_linkage_v1.parquet
Expected: 77d0522af5ec3ac0938a03aecdb9ebd230d6cafae8e8ded7a8de1fd462cfbe75
Observed: 77d0522af5ec3ac0938a03aecdb9ebd230d6cafae8e8ded7a8de1fd462cfbe75

PASS — Stage 3H linkage policy
Path:     /content/d

In [2]:
# ==================================================================================================
# STAGE 5A — STEP 2
# VERIFY THE FROZEN LINKAGE-ELIGIBILITY AND CENSORING BOUNDARY
#
# Purpose:
#   1. Inspect the Stage 3H final-linkage schema.
#   2. Confirm the exact linkage-decision accounting.
#   3. Verify that only accepted one-to-one links are eligible
#      for future-outcome construction.
#   4. Confirm that censored records have no attached T1 record.
#   5. Confirm that no future-instability outcome already exists.
#
# Leakage boundary:
#   - No Stage 4 feature, weak-label, model, or GES-score file is loaded.
#   - No classification transitions are calculated.
#   - No future-instability label is assigned.
#   - No temporal performance is examined.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage artifact
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Inspect the complete schema without loading the complete dataset
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(
    STAGE3H_LINKAGE_PATH
)

schema_columns = parquet_file.schema.names

print("=" * 108)
print("STAGE 5A STEP 2 — FINAL LINKAGE SCHEMA AND ELIGIBILITY AUDIT")
print("=" * 108)

print("\nPARQUET METADATA")
print("-" * 108)
print(
    f"Rows:       "
    f"{parquet_file.metadata.num_rows:,}"
)
print(
    f"Columns:    "
    f"{parquet_file.metadata.num_columns:,}"
)
print(
    f"Row groups: "
    f"{parquet_file.metadata.num_row_groups:,}"
)


# --------------------------------------------------------------------------------------------------
# 3. Confirm the columns required for Stage 5
# --------------------------------------------------------------------------------------------------

required_control_columns = [
    "t0_rcv_accession",
    "linked_t1_rcv_accession",
    "t1_rcv_accession",
    "linkage_status",
    "linkage_method",
    "linkage_decision_category",
    "censoring_disposition",
    "temporal_outcome_eligible",
    "future_instability_outcome_created",
    "future_instability_label",
    "stage3_freeze_version",
]

required_t0_semantic_columns = [
    "t0_target_genes_json",
    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",
]

required_t1_semantic_columns = [
    "t1_target_genes_json",
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classification_axis",
    "t1_aggregate_classifications_json",
]

required_columns = (
    required_control_columns
    + required_t0_semantic_columns
    + required_t1_semantic_columns
)

missing_required_columns = sorted(
    set(required_columns).difference(
        schema_columns
    )
)

print("\nREQUIRED STAGE 5 COLUMN CHECK")
print("-" * 108)

for column in required_columns:
    status = (
        "PRESENT"
        if column in schema_columns
        else "MISSING"
    )

    print(
        f"{status:8s}  {column}"
    )

if missing_required_columns:
    raise RuntimeError(
        "The frozen linkage is missing Stage 5 fields:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_required_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. Print a structured column inventory
# --------------------------------------------------------------------------------------------------

t0_columns = [
    column
    for column in schema_columns
    if column.startswith("t0_")
]

t1_columns = [
    column
    for column in schema_columns
    if column.startswith("t1_")
]

linkage_control_columns = [
    column
    for column in schema_columns
    if (
        not column.startswith("t0_")
        and not column.startswith("t1_")
    )
]

print("\nCOLUMN-FAMILY INVENTORY")
print("-" * 108)
print(
    f"T0-prefixed fields:       "
    f"{len(t0_columns):,}"
)
print(
    f"T1-prefixed fields:       "
    f"{len(t1_columns):,}"
)
print(
    f"Linkage/control fields:   "
    f"{len(linkage_control_columns):,}"
)
print(
    f"Total fields:             "
    f"{len(schema_columns):,}"
)

print("\nLINKAGE/CONTROL COLUMNS")
print("-" * 108)

for number, column in enumerate(
    linkage_control_columns,
    start=1,
):
    print(f"{number:02d}. {column}")


# --------------------------------------------------------------------------------------------------
# 5. Load only the control fields needed for this audit
# --------------------------------------------------------------------------------------------------

control_columns_to_load = [
    "t0_rcv_accession",
    "linked_t1_rcv_accession",
    "t1_rcv_accession",
    "linkage_status",
    "linkage_method",
    "linkage_decision_category",
    "censoring_disposition",
    "temporal_outcome_eligible",
    "future_instability_outcome_created",
    "future_instability_label",
    "stage3_freeze_version",
]

linkage = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=control_columns_to_load,
)


# --------------------------------------------------------------------------------------------------
# 6. Expected frozen linkage decisions
# --------------------------------------------------------------------------------------------------

EXPECTED_ROWS = 71_659

EXPECTED_DECISION_COUNTS = {
    "ACCEPTED_EXACT_RCV": 70_413,
    "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE": 170,
    "CENSORED_COMPLEX_STRONG_CANDIDATE": 38,
    (
        "CENSORED_VARIANT_CONTINUITY_"
        "CONDITION_ASSOCIATION_UNRESOLVED"
    ): 416,
    "CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE": 622,
}

EXPECTED_ELIGIBLE = 70_583
EXPECTED_CENSORED = 1_076


# --------------------------------------------------------------------------------------------------
# 7. Normalize Boolean eligibility without altering the source
# --------------------------------------------------------------------------------------------------

def normalize_boolean(value):
    """
    Normalize common Boolean representations.
    Invalid representations remain visibly invalid.
    """

    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True
        if value == 0:
            return False

    normalized = str(value).strip().lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        normalized,
        "INVALID",
    )


eligibility_normalized = (
    linkage["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

outcome_created_normalized = (
    linkage[
        "future_instability_outcome_created"
    ]
    .apply(normalize_boolean)
)

invalid_eligibility_values = int(
    eligibility_normalized.eq(
        "INVALID"
    ).sum()
)

invalid_outcome_created_values = int(
    outcome_created_normalized.eq(
        "INVALID"
    ).sum()
)

eligible_mask = (
    eligibility_normalized.eq(True)
)

censored_mask = (
    eligibility_normalized.eq(False)
)


# --------------------------------------------------------------------------------------------------
# 8. Core row and key integrity
# --------------------------------------------------------------------------------------------------

row_count = len(linkage)

unique_t0_rcvs = (
    linkage["t0_rcv_accession"]
    .nunique(dropna=True)
)

duplicate_t0_rows = int(
    linkage["t0_rcv_accession"]
    .duplicated(keep=False)
    .sum()
)

missing_t0_rcvs = int(
    linkage["t0_rcv_accession"]
    .isna()
    .sum()
)

print("\nROW AND T0-KEY INTEGRITY")
print("-" * 108)
print(
    f"Rows loaded:                         "
    f"{row_count:,}"
)
print(
    f"Expected rows:                       "
    f"{EXPECTED_ROWS:,}"
)
print(
    f"Unique T0 RCV accessions:            "
    f"{unique_t0_rcvs:,}"
)
print(
    f"Missing T0 RCV accessions:           "
    f"{missing_t0_rcvs:,}"
)
print(
    f"Rows in duplicated T0 RCVs:          "
    f"{duplicate_t0_rows:,}"
)


# --------------------------------------------------------------------------------------------------
# 9. Linkage-decision accounting
# --------------------------------------------------------------------------------------------------

observed_decision_counts = (
    linkage[
        "linkage_decision_category"
    ]
    .value_counts(dropna=False)
    .to_dict()
)

print("\nFROZEN LINKAGE-DECISION ACCOUNTING")
print("-" * 108)

for category, expected_count in (
    EXPECTED_DECISION_COUNTS.items()
):

    observed_count = int(
        observed_decision_counts.get(
            category,
            0,
        )
    )

    status = (
        "PASS"
        if observed_count == expected_count
        else "FAIL"
    )

    print(
        f"{status:4s}  "
        f"{category:68s} "
        f"observed={observed_count:>7,}  "
        f"expected={expected_count:>7,}"
    )

unexpected_decision_categories = sorted(
    set(observed_decision_counts).difference(
        EXPECTED_DECISION_COUNTS
    ),
    key=str,
)

if unexpected_decision_categories:
    print("\nUnexpected decision categories:")

    for category in unexpected_decision_categories:
        print(
            " -",
            repr(category),
            observed_decision_counts[
                category
            ],
        )


# --------------------------------------------------------------------------------------------------
# 10. Eligibility and attached-T1 checks
# --------------------------------------------------------------------------------------------------

eligible_count = int(
    eligible_mask.sum()
)

censored_count = int(
    censored_mask.sum()
)

missing_eligibility_count = int(
    eligibility_normalized.isna().sum()
)

eligible_missing_link_key = int(
    linkage.loc[
        eligible_mask,
        "linked_t1_rcv_accession",
    ]
    .isna()
    .sum()
)

eligible_missing_t1_record = int(
    linkage.loc[
        eligible_mask,
        "t1_rcv_accession",
    ]
    .isna()
    .sum()
)

censored_with_link_key = int(
    linkage.loc[
        censored_mask,
        "linked_t1_rcv_accession",
    ]
    .notna()
    .sum()
)

censored_with_t1_record = int(
    linkage.loc[
        censored_mask,
        "t1_rcv_accession",
    ]
    .notna()
    .sum()
)

eligible_t1_duplicate_rows = int(
    linkage.loc[
        eligible_mask,
        "t1_rcv_accession",
    ]
    .duplicated(keep=False)
    .sum()
)

print("\nTEMPORAL-OUTCOME ELIGIBILITY BOUNDARY")
print("-" * 108)
print(
    f"Outcome-eligible linked records:     "
    f"{eligible_count:,}"
)
print(
    f"Expected eligible records:           "
    f"{EXPECTED_ELIGIBLE:,}"
)
print(
    f"Unresolved/censored records:         "
    f"{censored_count:,}"
)
print(
    f"Expected censored records:           "
    f"{EXPECTED_CENSORED:,}"
)
print(
    f"Missing eligibility values:          "
    f"{missing_eligibility_count:,}"
)
print(
    f"Invalid eligibility values:          "
    f"{invalid_eligibility_values:,}"
)
print(
    f"Eligible rows missing link key:      "
    f"{eligible_missing_link_key:,}"
)
print(
    f"Eligible rows missing T1 record:     "
    f"{eligible_missing_t1_record:,}"
)
print(
    f"Censored rows containing link key:   "
    f"{censored_with_link_key:,}"
)
print(
    f"Censored rows containing T1 record:  "
    f"{censored_with_t1_record:,}"
)
print(
    f"Rows in duplicated eligible T1 RCVs: "
    f"{eligible_t1_duplicate_rows:,}"
)


# --------------------------------------------------------------------------------------------------
# 11. Profile linkage status, method, censoring disposition, and freeze version
# --------------------------------------------------------------------------------------------------

profile_columns = [
    "linkage_status",
    "linkage_method",
    "censoring_disposition",
    "stage3_freeze_version",
]

for column in profile_columns:

    print(
        f"\n{column.upper()} DISTRIBUTION"
    )
    print("-" * 108)

    print(
        linkage[column]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 12. Confirm that Stage 3 did not create an outcome
# --------------------------------------------------------------------------------------------------

outcome_created_true = int(
    outcome_created_normalized.eq(
        True
    ).sum()
)

outcome_created_false = int(
    outcome_created_normalized.eq(
        False
    ).sum()
)

outcome_created_missing = int(
    outcome_created_normalized.isna().sum()
)

nonmissing_future_labels = int(
    linkage[
        "future_instability_label"
    ]
    .notna()
    .sum()
)

print("\nSCIENTIFIC-BOUNDARY CHECKS")
print("-" * 108)
print(
    f"future_instability_outcome_created=True:  "
    f"{outcome_created_true:,}"
)
print(
    f"future_instability_outcome_created=False: "
    f"{outcome_created_false:,}"
)
print(
    f"Missing outcome-created flags:            "
    f"{outcome_created_missing:,}"
)
print(
    f"Invalid outcome-created flags:            "
    f"{invalid_outcome_created_values:,}"
)
print(
    f"Nonmissing future-instability labels:     "
    f"{nonmissing_future_labels:,}"
)


# --------------------------------------------------------------------------------------------------
# 13. Final acceptance decision
# --------------------------------------------------------------------------------------------------

critical_failures = {
    "unexpected linkage row count":
        row_count != EXPECTED_ROWS,

    "missing T0 RCV keys":
        missing_t0_rcvs > 0,

    "duplicated T0 RCV keys":
        duplicate_t0_rows > 0,

    "incorrect linkage-decision accounting":
        observed_decision_counts
        != EXPECTED_DECISION_COUNTS,

    "missing or invalid eligibility values":
        (
            missing_eligibility_count > 0
            or invalid_eligibility_values > 0
        ),

    "incorrect eligible count":
        eligible_count != EXPECTED_ELIGIBLE,

    "incorrect censored count":
        censored_count != EXPECTED_CENSORED,

    "eligible rows missing T1 linkage":
        (
            eligible_missing_link_key > 0
            or eligible_missing_t1_record > 0
        ),

    "censored rows contain attached T1 evidence":
        (
            censored_with_link_key > 0
            or censored_with_t1_record > 0
        ),

    "eligible T1 records reused":
        eligible_t1_duplicate_rows > 0,

    "future outcome already created":
        outcome_created_true > 0,

    "missing or invalid outcome-created flags":
        (
            outcome_created_missing > 0
            or invalid_outcome_created_values > 0
        ),

    "future-instability labels already populated":
        nonmissing_future_labels > 0,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\n" + "=" * 108)
print("STAGE 5A STEP 2 DECISION")
print("=" * 108)

if failed_checks:

    print(
        "FAIL_STAGE5A_LINKAGE_BOUNDARY_REVIEW_REQUIRED"
    )

    print("\nFailed checks:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Stage 5A Step 2 failed. "
        "Do not proceed to classification profiling."
    )

print(
    "PASS_STAGE5A_LINKAGE_ELIGIBILITY_BOUNDARY_VERIFIED"
)

print()
print(
    "Outcome-eligible linked records: "
    f"{eligible_count:,}"
)
print(
    "Unresolved/censored records:     "
    f"{censored_count:,}"
)
print()
print(
    "Stage 4 GES scores loaded:       NO"
)
print(
    "Classification changes computed: NO"
)
print(
    "Future outcomes created:         NO"
)
print(
    "Temporal performance examined:   NO"
)
print(
    "Files written or modified:       NO"
)

STAGE 5A STEP 2 — FINAL LINKAGE SCHEMA AND ELIGIBILITY AUDIT

PARQUET METADATA
------------------------------------------------------------------------------------------------------------
Rows:       71,659
Columns:    81
Row groups: 1

REQUIRED STAGE 5 COLUMN CHECK
------------------------------------------------------------------------------------------------------------
PRESENT   t0_rcv_accession
PRESENT   linked_t1_rcv_accession
PRESENT   t1_rcv_accession
PRESENT   linkage_status
PRESENT   linkage_method
PRESENT   linkage_decision_category
PRESENT   censoring_disposition
PRESENT   temporal_outcome_eligible
PRESENT   future_instability_outcome_created
PRESENT   future_instability_label
PRESENT   stage3_freeze_version
PRESENT   t0_target_genes_json
PRESENT   t0_aggregate_classification
PRESENT   t0_aggregate_classification_group
PRESENT   t0_aggregate_review_status
PRESENT   t0_aggregate_review_stars
PRESENT   t0_aggregate_conflict_flag
PRESENT   t0_scv_group_disagreement_flag
PRESEN

In [5]:
# ==================================================================================================
# STAGE 5A — STEP 3
# INVENTORY T0/T1 CLASSIFICATIONS, GROUPS, AXES, REVIEW FIELDS, AND MISSINGNESS
#
# Purpose:
#   1. Restrict semantic profiling to the 70,583 frozen outcome-eligible links.
#   2. Confirm target-gene continuity for accepted links.
#   3. Inventory exact T0 and T1 aggregate classification values.
#   4. Inventory normalized classification groups.
#   5. Inventory T1 classification axes overall and by gene.
#   6. Audit review statuses, review stars, conflict flags, and SCV disagreement flags.
#   7. Validate the structure of aggregate_classifications_json.
#
# Scientific boundary:
#   - No T0-to-T1 classification transition is assigned.
#   - No future-instability outcome is created.
#   - No group or axis mapping is frozen in this cell.
#   - No Stage 4 feature, weak-label, model, or GES-score artifact is loaded.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
import json

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage artifact
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load only Stage 5A semantic-precheck fields
# --------------------------------------------------------------------------------------------------

SEMANTIC_COLUMNS = [
    # Linkage controls
    "t0_rcv_accession",
    "t1_rcv_accession",
    "linkage_decision_category",
    "temporal_outcome_eligible",

    # Gene continuity
    "t0_target_genes_json",
    "t1_target_genes_json",

    # T0 aggregate evidence
    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",

    # T1 aggregate evidence
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classification_axis",
    "t1_aggregate_classifications_json",
]

linkage = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=SEMANTIC_COLUMNS,
)

EXPECTED_TOTAL_ROWS = 71_659
EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_CENSORED_ROWS = 1_076


# --------------------------------------------------------------------------------------------------
# 3. Helper functions
# --------------------------------------------------------------------------------------------------

def is_missing_scalar(value) -> bool:
    """
    Safely identify scalar missing values.
    """

    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except Exception:
        pass

    return False


def normalize_boolean(value):
    """
    Normalize common Boolean representations.
    Invalid values remain visibly marked as INVALID.
    """

    if is_missing_scalar(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True
        if value == 0:
            return False

    text = str(value).strip().lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        text,
        "INVALID",
    )


def normalize_text_series(
    series: pd.Series,
    lowercase: bool = False,
) -> pd.Series:
    """
    Normalize whitespace and convert blank/null-like strings
    to pandas missing values.
    """

    normalized = (
        series.astype("string")
        .str.strip()
    )

    null_like = (
        normalized.str.lower()
        .isin(
            {
                "",
                "none",
                "null",
                "nan",
                "nat",
            }
        )
    )

    normalized = normalized.mask(
        null_like,
        pd.NA,
    )

    if lowercase:
        normalized = normalized.str.lower()

    return normalized


def parse_gene_list(value):
    """
    Parse a target-gene JSON field and return a sorted,
    deduplicated list of uppercase symbols.

    Returns None for malformed or structurally invalid values.
    """

    if is_missing_scalar(value):
        return None

    if isinstance(value, list):
        parsed = value
    else:
        try:
            parsed = json.loads(
                str(value)
            )
        except (
            json.JSONDecodeError,
            TypeError,
            ValueError,
        ):
            return None

    if not isinstance(parsed, list):
        return None

    normalized_genes = []

    for gene in parsed:

        if gene is None:
            continue

        gene_text = (
            str(gene)
            .strip()
            .upper()
        )

        if gene_text:
            normalized_genes.append(
                gene_text
            )

    return sorted(
        set(normalized_genes)
    )


def profile_json_value(value):
    """
    Parse one JSON field and return:
      - structural type;
      - number of top-level entries.

    The original data are not modified.
    """

    if is_missing_scalar(value):
        return {
            "status": "MISSING",
            "type": None,
            "entry_count": None,
        }

    try:
        parsed = (
            json.loads(value)
            if isinstance(value, str)
            else value
        )
    except (
        json.JSONDecodeError,
        TypeError,
        ValueError,
    ):
        return {
            "status": "PARSE_ERROR",
            "type": None,
            "entry_count": None,
        }

    parsed_type = type(parsed).__name__

    if isinstance(
        parsed,
        (list, dict, tuple, set),
    ):
        entry_count = len(parsed)
    else:
        entry_count = None

    return {
        "status": "PARSED",
        "type": parsed_type,
        "entry_count": entry_count,
    }


def print_distribution(
    title: str,
    series: pd.Series,
):
    """
    Print a complete value distribution, including missingness.
    """

    print(f"\n{title}")
    print("-" * 112)

    display_series = (
        series.astype("string")
        .fillna("<MISSING>")
    )

    counts = display_series.value_counts(
        dropna=False
    )

    print(
        f"Unique displayed values: "
        f"{len(counts):,}"
    )

    print(
        counts.to_string()
    )


def audit_star_field(
    frame: pd.DataFrame,
    column: str,
):
    """
    Profile review stars and return structural error counts.
    """

    original = frame[column]

    numeric = pd.to_numeric(
        original,
        errors="coerce",
    )

    original_nonmissing = (
        normalize_text_series(
            original
        )
        .notna()
    )

    malformed_mask = (
        original_nonmissing
        & numeric.isna()
    )

    outside_range_mask = (
        numeric.notna()
        & (
            (numeric < 0)
            | (numeric > 4)
            | (numeric % 1 != 0)
        )
    )

    return {
        "numeric": numeric,
        "missing": int(
            numeric.isna().sum()
        ),
        "malformed": int(
            malformed_mask.sum()
        ),
        "outside_range": int(
            outside_range_mask.sum()
        ),
    }


# --------------------------------------------------------------------------------------------------
# 4. Confirm eligibility boundary again before profiling
# --------------------------------------------------------------------------------------------------

eligibility = (
    linkage["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

invalid_eligibility = int(
    eligibility.eq("INVALID").sum()
)

missing_eligibility = int(
    eligibility.isna().sum()
)

eligible_mask = eligibility.eq(True)
censored_mask = eligibility.eq(False)

eligible = (
    linkage.loc[eligible_mask]
    .copy()
    .reset_index(drop=True)
)

censored = (
    linkage.loc[censored_mask]
    .copy()
    .reset_index(drop=True)
)

print("=" * 112)
print("STAGE 5A STEP 3 — CLASSIFICATION AND AXIS INVENTORY")
print("=" * 112)

print("\nLINKAGE BOUNDARY")
print("-" * 112)
print(
    f"Total Stage 3H rows:              "
    f"{len(linkage):,}"
)
print(
    f"Outcome-eligible rows:            "
    f"{len(eligible):,}"
)
print(
    f"Unresolved/censored rows:         "
    f"{len(censored):,}"
)
print(
    f"Missing eligibility values:       "
    f"{missing_eligibility:,}"
)
print(
    f"Invalid eligibility values:       "
    f"{invalid_eligibility:,}"
)


# --------------------------------------------------------------------------------------------------
# 5. Confirm censored records carry no T1 semantic evidence
# --------------------------------------------------------------------------------------------------

T1_SEMANTIC_FIELDS = [
    "t1_rcv_accession",
    "t1_target_genes_json",
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classification_axis",
    "t1_aggregate_classifications_json",
]

censored_t1_nonmissing_counts = (
    censored[T1_SEMANTIC_FIELDS]
    .notna()
    .sum()
    .astype(int)
)

censored_rows_with_any_t1_semantic_value = int(
    censored[
        T1_SEMANTIC_FIELDS
    ]
    .notna()
    .any(axis=1)
    .sum()
)

print("\nCENSORED-RECORD T1 SEMANTIC CHECK")
print("-" * 112)
print(
    f"Censored rows containing any T1 semantic value: "
    f"{censored_rows_with_any_t1_semantic_value:,}"
)

if censored_rows_with_any_t1_semantic_value > 0:
    print(
        censored_t1_nonmissing_counts[
            censored_t1_nonmissing_counts > 0
        ].to_string()
    )


# --------------------------------------------------------------------------------------------------
# 6. Parse and validate target genes
# --------------------------------------------------------------------------------------------------

eligible["_t0_genes"] = (
    eligible["t0_target_genes_json"]
    .apply(parse_gene_list)
)

eligible["_t1_genes"] = (
    eligible["t1_target_genes_json"]
    .apply(parse_gene_list)
)

t0_gene_parse_failure = int(
    eligible["_t0_genes"]
    .isna()
    .sum()
)

t1_gene_parse_failure = int(
    eligible["_t1_genes"]
    .isna()
    .sum()
)

t0_not_single_gene = int(
    eligible["_t0_genes"]
    .apply(
        lambda genes:
            not (
                isinstance(genes, list)
                and len(genes) == 1
            )
    )
    .sum()
)

t1_not_single_gene = int(
    eligible["_t1_genes"]
    .apply(
        lambda genes:
            not (
                isinstance(genes, list)
                and len(genes) == 1
            )
    )
    .sum()
)

eligible["_t0_gene"] = (
    eligible["_t0_genes"]
    .apply(
        lambda genes:
            genes[0]
            if (
                isinstance(genes, list)
                and len(genes) == 1
            )
            else pd.NA
    )
)

eligible["_t1_gene"] = (
    eligible["_t1_genes"]
    .apply(
        lambda genes:
            genes[0]
            if (
                isinstance(genes, list)
                and len(genes) == 1
            )
            else pd.NA
    )
)

gene_mismatch_mask = (
    eligible["_t0_gene"].notna()
    & eligible["_t1_gene"].notna()
    & eligible["_t0_gene"].ne(
        eligible["_t1_gene"]
    )
)

gene_mismatch_count = int(
    gene_mismatch_mask.sum()
)

print("\nTARGET-GENE CONTINUITY")
print("-" * 112)
print(
    f"T0 gene JSON parsing failures:     "
    f"{t0_gene_parse_failure:,}"
)
print(
    f"T1 gene JSON parsing failures:     "
    f"{t1_gene_parse_failure:,}"
)
print(
    f"T0 rows not containing one gene:   "
    f"{t0_not_single_gene:,}"
)
print(
    f"T1 rows not containing one gene:   "
    f"{t1_not_single_gene:,}"
)
print(
    f"T0/T1 gene mismatches:             "
    f"{gene_mismatch_count:,}"
)

print_distribution(
    "OUTCOME-ELIGIBLE RECORDS BY TARGET GENE",
    eligible["_t0_gene"],
)

print_distribution(
    "OUTCOME-ELIGIBLE RECORDS BY LINKAGE DECISION",
    eligible[
        "linkage_decision_category"
    ],
)


# --------------------------------------------------------------------------------------------------
# 7. Normalize exact classification, group, review, and axis fields
# --------------------------------------------------------------------------------------------------

TEXT_FIELDS = [
    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_classification_axis",
]

for column in TEXT_FIELDS:
    eligible[f"_{column}_normalized"] = (
        normalize_text_series(
            eligible[column]
        )
    )

t0_classification = eligible[
    "_t0_aggregate_classification_normalized"
]

t1_classification = eligible[
    "_t1_aggregate_classification_normalized"
]

t0_group = eligible[
    "_t0_aggregate_classification_group_normalized"
]

t1_group = eligible[
    "_t1_aggregate_classification_group_normalized"
]

t0_review_status = eligible[
    "_t0_aggregate_review_status_normalized"
]

t1_review_status = eligible[
    "_t1_aggregate_review_status_normalized"
]

t1_axis = eligible[
    "_t1_aggregate_classification_axis_normalized"
]


# --------------------------------------------------------------------------------------------------
# 8. Exact classification inventory
# --------------------------------------------------------------------------------------------------

print_distribution(
    "T0 EXACT AGGREGATE CLASSIFICATION VALUES",
    t0_classification,
)

print_distribution(
    "T1 EXACT AGGREGATE CLASSIFICATION VALUES",
    t1_classification,
)

print("\nCLASSIFICATION MISSINGNESS")
print("-" * 112)
print(
    f"T0 missing aggregate classifications: "
    f"{t0_classification.isna().sum():,}"
)
print(
    f"T1 missing aggregate classifications: "
    f"{t1_classification.isna().sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 9. Normalized classification-group inventory
# --------------------------------------------------------------------------------------------------

print_distribution(
    "T0 NORMALIZED CLASSIFICATION GROUPS",
    t0_group,
)

print_distribution(
    "T1 NORMALIZED CLASSIFICATION GROUPS",
    t1_group,
)

ALLOWED_CLASSIFICATION_GROUPS = {
    "benign/likely benign",
    "pathogenic/likely pathogenic",
    "vus",
    "conflicting",
    "mixed",
    "other",
    "missing",
}

t0_group_lower = (
    t0_group.str.lower()
)

t1_group_lower = (
    t1_group.str.lower()
)

unexpected_t0_group_mask = (
    t0_group_lower.notna()
    & ~t0_group_lower.isin(
        ALLOWED_CLASSIFICATION_GROUPS
    )
)

unexpected_t1_group_mask = (
    t1_group_lower.notna()
    & ~t1_group_lower.isin(
        ALLOWED_CLASSIFICATION_GROUPS
    )
)

unexpected_t0_group_count = int(
    unexpected_t0_group_mask.sum()
)

unexpected_t1_group_count = int(
    unexpected_t1_group_mask.sum()
)

print("\nCLASSIFICATION-GROUP STRUCTURAL CHECKS")
print("-" * 112)
print(
    f"T0 missing group values:           "
    f"{t0_group.isna().sum():,}"
)
print(
    f"T1 missing group values:           "
    f"{t1_group.isna().sum():,}"
)
print(
    f"T0 unexpected group values:        "
    f"{unexpected_t0_group_count:,}"
)
print(
    f"T1 unexpected group values:        "
    f"{unexpected_t1_group_count:,}"
)

if unexpected_t0_group_count > 0:
    print("\nUnexpected T0 group values:")

    print(
        t0_group.loc[
            unexpected_t0_group_mask
        ]
        .value_counts()
        .to_string()
    )

if unexpected_t1_group_count > 0:
    print("\nUnexpected T1 group values:")

    print(
        t1_group.loc[
            unexpected_t1_group_mask
        ]
        .value_counts()
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 10. T1 classification-axis inventory
# --------------------------------------------------------------------------------------------------

print_distribution(
    "T1 SELECTED CLASSIFICATION AXES",
    t1_axis,
)

ALLOWED_T1_AXES = {
    "germlineclassification",
    "oncogenicityclassification",
    "somaticclinicalimpact",
    "noclassification",
}

t1_axis_lower = (
    t1_axis.str.lower()
)

unexpected_axis_mask = (
    t1_axis_lower.notna()
    & ~t1_axis_lower.isin(
        ALLOWED_T1_AXES
    )
)

unexpected_axis_count = int(
    unexpected_axis_mask.sum()
)

missing_axis_count = int(
    t1_axis.isna().sum()
)

axis_by_gene = pd.crosstab(
    eligible["_t0_gene"]
    .fillna("<GENE_PARSE_ERROR>"),
    t1_axis.fillna("<MISSING>"),
    dropna=False,
)

print("\nT1 CLASSIFICATION AXIS BY TARGET GENE")
print("-" * 112)
print(
    axis_by_gene.to_string()
)

print("\nT1 AXIS STRUCTURAL CHECKS")
print("-" * 112)
print(
    f"Missing selected T1 axes:           "
    f"{missing_axis_count:,}"
)
print(
    f"Unexpected selected T1 axes:        "
    f"{unexpected_axis_count:,}"
)

if unexpected_axis_count > 0:
    print(
        t1_axis.loc[
            unexpected_axis_mask
        ]
        .value_counts()
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 11. Check classification presence against NoClassification axis
# --------------------------------------------------------------------------------------------------

no_classification_axis_mask = (
    t1_axis_lower.eq(
        "noclassification"
    )
)

classification_missing_outside_no_classification = int(
    (
        t1_classification.isna()
        & ~no_classification_axis_mask
    ).sum()
)

classification_present_inside_no_classification = int(
    (
        t1_classification.notna()
        & no_classification_axis_mask
    ).sum()
)

group_missing_label_inside_no_classification = int(
    (
        no_classification_axis_mask
        & t1_group_lower.eq("missing")
    ).sum()
)

print("\nT1 AXIS/CLASSIFICATION CONSISTENCY")
print("-" * 112)
print(
    f"NoClassification-axis records:                     "
    f"{no_classification_axis_mask.sum():,}"
)
print(
    f"Missing classification outside NoClassification:   "
    f"{classification_missing_outside_no_classification:,}"
)
print(
    f"Present classification inside NoClassification:    "
    f"{classification_present_inside_no_classification:,}"
)
print(
    f"NoClassification records with group='Missing':     "
    f"{group_missing_label_inside_no_classification:,}"
)


# --------------------------------------------------------------------------------------------------
# 12. Review-status and review-star inventory
# --------------------------------------------------------------------------------------------------

print_distribution(
    "T0 AGGREGATE REVIEW STATUS",
    t0_review_status,
)

print_distribution(
    "T1 AGGREGATE REVIEW STATUS",
    t1_review_status,
)

t0_star_audit = audit_star_field(
    eligible,
    "t0_aggregate_review_stars",
)

t1_star_audit = audit_star_field(
    eligible,
    "t1_aggregate_review_stars",
)

print_distribution(
    "T0 AGGREGATE REVIEW STARS",
    t0_star_audit["numeric"],
)

print_distribution(
    "T1 AGGREGATE REVIEW STARS",
    t1_star_audit["numeric"],
)

print("\nREVIEW-STAR STRUCTURAL CHECKS")
print("-" * 112)
print(
    f"T0 missing review stars:            "
    f"{t0_star_audit['missing']:,}"
)
print(
    f"T0 malformed review stars:          "
    f"{t0_star_audit['malformed']:,}"
)
print(
    f"T0 stars outside integer 0-4:       "
    f"{t0_star_audit['outside_range']:,}"
)
print(
    f"T1 missing review stars:            "
    f"{t1_star_audit['missing']:,}"
)
print(
    f"T1 malformed review stars:          "
    f"{t1_star_audit['malformed']:,}"
)
print(
    f"T1 stars outside integer 0-4:       "
    f"{t1_star_audit['outside_range']:,}"
)


# --------------------------------------------------------------------------------------------------
# 13. Conflict and SCV-disagreement field inventory
# --------------------------------------------------------------------------------------------------

BOOLEAN_FIELDS = [
    "t0_aggregate_conflict_flag",
    "t1_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",
    "t1_scv_group_disagreement_flag",
]

boolean_audit = {}

for column in BOOLEAN_FIELDS:

    normalized_boolean = (
        eligible[column]
        .apply(normalize_boolean)
    )

    boolean_audit[column] = {
        "normalized": normalized_boolean,
        "missing": int(
            normalized_boolean.isna().sum()
        ),
        "invalid": int(
            normalized_boolean.eq(
                "INVALID"
            ).sum()
        ),
    }

    print_distribution(
        column.upper(),
        normalized_boolean,
    )

print("\nBOOLEAN-FIELD STRUCTURAL CHECKS")
print("-" * 112)

for column in BOOLEAN_FIELDS:
    print(
        f"{column:42s} "
        f"missing={boolean_audit[column]['missing']:>6,} | "
        f"invalid={boolean_audit[column]['invalid']:>6,}"
    )


# --------------------------------------------------------------------------------------------------
# 14. Validate aggregate_classifications_json structure
# --------------------------------------------------------------------------------------------------

classification_json_profiles = (
    eligible[
        "t1_aggregate_classifications_json"
    ]
    .apply(profile_json_value)
)

classification_json_status = (
    classification_json_profiles
    .apply(
        lambda item:
            item["status"]
    )
)

classification_json_type = (
    classification_json_profiles
    .apply(
        lambda item:
            item["type"]
    )
)

classification_json_entry_count = pd.to_numeric(
    classification_json_profiles.apply(
        lambda item:
            item["entry_count"]
    ),
    errors="coerce",
)

json_parse_errors = int(
    classification_json_status.eq(
        "PARSE_ERROR"
    ).sum()
)

json_missing = int(
    classification_json_status.eq(
        "MISSING"
    ).sum()
)

print_distribution(
    "T1 aggregate_classifications_json PARSE STATUS",
    classification_json_status,
)

print_distribution(
    "T1 aggregate_classifications_json TOP-LEVEL TYPE",
    classification_json_type,
)

print("\nT1 aggregate_classifications_json ENTRY COUNTS")
print("-" * 112)

if classification_json_entry_count.notna().any():

    print(
        classification_json_entry_count
        .describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
        .to_string()
    )

else:
    print(
        "No numeric top-level entry counts were available."
    )

print("\nJSON STRUCTURAL CHECKS")
print("-" * 112)
print(
    f"Missing aggregate classification JSON values: "
    f"{json_missing:,}"
)
print(
    f"JSON parsing failures:                       "
    f"{json_parse_errors:,}"
)


# --------------------------------------------------------------------------------------------------
# 15. Final structural decision
# --------------------------------------------------------------------------------------------------

critical_failures = {
    "unexpected total linkage row count":
        len(linkage) != EXPECTED_TOTAL_ROWS,

    "unexpected eligible row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "unexpected censored row count":
        len(censored) != EXPECTED_CENSORED_ROWS,

    "missing or invalid eligibility values":
        (
            missing_eligibility > 0
            or invalid_eligibility > 0
        ),

    "censored records contain T1 semantic evidence":
        censored_rows_with_any_t1_semantic_value > 0,

    "T0 gene parsing or structure failure":
        (
            t0_gene_parse_failure > 0
            or t0_not_single_gene > 0
        ),

    "T1 gene parsing or structure failure":
        (
            t1_gene_parse_failure > 0
            or t1_not_single_gene > 0
        ),

    "accepted-link target-gene mismatch":
        gene_mismatch_count > 0,

    "missing T0 classification groups":
        int(t0_group.isna().sum()) > 0,

    "missing T1 classification groups":
        int(t1_group.isna().sum()) > 0,

    "unexpected T0 classification groups":
        unexpected_t0_group_count > 0,

    "unexpected T1 classification groups":
        unexpected_t1_group_count > 0,

    "missing T1 classification axes":
        missing_axis_count > 0,

    "unexpected T1 classification axes":
        unexpected_axis_count > 0,

    "T1 classification missing outside NoClassification":
        classification_missing_outside_no_classification > 0,

    "T1 classification present inside NoClassification":
        classification_present_inside_no_classification > 0,

    "invalid T0 review-star values":
        (
            t0_star_audit["malformed"] > 0
            or t0_star_audit[
                "outside_range"
            ] > 0
        ),

    "invalid T1 review-star values":
        (
            t1_star_audit["malformed"] > 0
            or t1_star_audit[
                "outside_range"
            ] > 0
        ),

    "invalid or missing Boolean evidence fields":
        any(
            result["missing"] > 0
            or result["invalid"] > 0
            for result in boolean_audit.values()
        ),

    "T1 classification JSON parsing failure":
        json_parse_errors > 0,

    "missing T1 classification JSON":
        json_missing > 0,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\n" + "=" * 112)
print("STAGE 5A STEP 3 DECISION")
print("=" * 112)

if failed_checks:

    print(
        "REVIEW_STAGE5A_CLASSIFICATION_AXIS_INVENTORY"
    )

    print("\nChecks requiring review:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Stage 5A Step 3 found a structural issue. "
        "Do not construct classification transitions or outcomes."
    )

print(
    "PASS_STAGE5A_CLASSIFICATION_AXIS_INVENTORY_COMPLETE"
)

print()
print(
    f"Outcome-eligible records profiled: "
    f"{len(eligible):,}"
)
print(
    f"Censored records excluded:         "
    f"{len(censored):,}"
)

print()
print(
    "T0/T1 transitions assigned:        NO"
)
print(
    "Classification mapping frozen:     NO"
)
print(
    "Future outcomes created:           NO"
)
print(
    "Stage 4 GES scores loaded:         NO"
)
print(
    "Temporal performance examined:     NO"
)
print(
    "Files written or modified:         NO"
)

STAGE 5A STEP 3 — CLASSIFICATION AND AXIS INVENTORY

LINKAGE BOUNDARY
----------------------------------------------------------------------------------------------------------------
Total Stage 3H rows:              71,659
Outcome-eligible rows:            70,583
Unresolved/censored rows:         1,076
Missing eligibility values:       0
Invalid eligibility values:       0

CENSORED-RECORD T1 SEMANTIC CHECK
----------------------------------------------------------------------------------------------------------------
Censored rows containing any T1 semantic value: 0

TARGET-GENE CONTINUITY
----------------------------------------------------------------------------------------------------------------
T0 gene JSON parsing failures:     0
T1 gene JSON parsing failures:     0
T0 rows not containing one gene:   0
T1 rows not containing one gene:   0
T0/T1 gene mismatches:             0

OUTCOME-ELIGIBLE RECORDS BY TARGET GENE
--------------------------------------------------------------

RuntimeError: Stage 5A Step 3 found a structural issue. Do not construct classification transitions or outcomes.

In [6]:
# ==================================================================================================
# STAGE 5A — STEP 3A
# DIAGNOSE T1 NoClassification RECORDS AND THE evidence_only SENTINEL
#
# Purpose:
#   1. Isolate all outcome-eligible T1 NoClassification records.
#   2. Confirm whether "evidence_only" is a uniform harmonization sentinel.
#   3. Inspect review status, stars, conflict status, genes, T0 classifications,
#      and nested aggregate-classification JSON.
#   4. Determine whether this is an expected semantic state rather than
#      a malformed T1 classification.
#
# Scientific boundary:
#   - No future-instability outcome is assigned.
#   - No record is labeled stable or unstable.
#   - No censoring rule is frozen.
#   - No GES score or Stage 4 model artifact is loaded.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
from collections import Counter, defaultdict
import json

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Frozen linkage artifact
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load only fields required for this focused diagnostic
# --------------------------------------------------------------------------------------------------

columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "linkage_decision_category",
    "temporal_outcome_eligible",

    "t0_target_genes_json",
    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",

    "t1_target_genes_json",
    "t1_aggregate_classification_axis",
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classifications_json",
]

df = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=columns,
)


# --------------------------------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------------------------------

def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


def normalize_boolean(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().lower(),
        "INVALID",
    )


def parse_gene(value):
    if pd.isna(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    return "|".join(genes) if genes else "<EMPTY>"


def parse_json_list(value):
    if pd.isna(value):
        return None, "MISSING"

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return None, "PARSE_ERROR"

    if not isinstance(parsed, list):
        return parsed, "WRONG_TOP_LEVEL_TYPE"

    return parsed, "PASS"


def flatten_json(value, prefix="root"):
    """
    Flatten nested JSON while using [] for list members so that
    structurally identical records share the same path.
    """

    leaves = []

    if isinstance(value, dict):

        if not value:
            leaves.append(
                (prefix, "<EMPTY_DICT>")
            )

        for key, item in value.items():
            child_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            leaves.extend(
                flatten_json(
                    item,
                    child_prefix,
                )
            )

    elif isinstance(value, list):

        if not value:
            leaves.append(
                (f"{prefix}[]", "<EMPTY_LIST>")
            )

        for item in value:
            leaves.extend(
                flatten_json(
                    item,
                    f"{prefix}[]",
                )
            )

    else:
        safe_value = (
            "<NULL>"
            if value is None
            else str(value)
        )

        leaves.append(
            (prefix, safe_value)
        )

    return leaves


def print_counts(title, series):
    print(f"\n{title}")
    print("-" * 110)

    print(
        series.astype("string")
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 4. Restrict to accepted outcome-eligible links
# --------------------------------------------------------------------------------------------------

eligibility = (
    df["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

invalid_eligibility = int(
    eligibility.eq("INVALID").sum()
)

eligible = (
    df.loc[
        eligibility.eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

eligible["_t0_gene"] = (
    eligible["t0_target_genes_json"]
    .apply(parse_gene)
)

eligible["_t1_gene"] = (
    eligible["t1_target_genes_json"]
    .apply(parse_gene)
)

eligible["_t1_axis"] = (
    normalize_text(
        eligible[
            "t1_aggregate_classification_axis"
        ]
    )
)

eligible["_t1_classification"] = (
    normalize_text(
        eligible[
            "t1_aggregate_classification"
        ]
    )
)

eligible["_t1_group"] = (
    normalize_text(
        eligible[
            "t1_aggregate_classification_group"
        ]
    )
)

eligible["_t1_review_status"] = (
    normalize_text(
        eligible[
            "t1_aggregate_review_status"
        ]
    )
)

eligible["_t1_conflict"] = (
    eligible[
        "t1_aggregate_conflict_flag"
    ]
    .apply(normalize_boolean)
)

eligible["_t1_scv_disagreement"] = (
    eligible[
        "t1_scv_group_disagreement_flag"
    ]
    .apply(normalize_boolean)
)


# --------------------------------------------------------------------------------------------------
# 5. Isolate NoClassification records
# --------------------------------------------------------------------------------------------------

no_classification = (
    eligible.loc[
        eligible["_t1_axis"]
        .str.casefold()
        .eq("noclassification")
    ]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_NO_CLASSIFICATION_ROWS = 3_092

print("=" * 110)
print("STAGE 5A STEP 3A — T1 NoClassification / evidence_only DIAGNOSTIC")
print("=" * 110)

print("\nBASIC ACCOUNTING")
print("-" * 110)
print(
    f"Stage 3H rows loaded:                    "
    f"{len(df):,}"
)
print(
    f"Outcome-eligible rows:                   "
    f"{len(eligible):,}"
)
print(
    f"Invalid eligibility values:              "
    f"{invalid_eligibility:,}"
)
print(
    f"T1 NoClassification rows:                "
    f"{len(no_classification):,}"
)
print(
    f"Expected T1 NoClassification rows:       "
    f"{EXPECTED_NO_CLASSIFICATION_ROWS:,}"
)


# --------------------------------------------------------------------------------------------------
# 6. Profile the harmonized sentinel and supporting fields
# --------------------------------------------------------------------------------------------------

print_counts(
    "NoClassification — T1 AGGREGATE CLASSIFICATION",
    no_classification["_t1_classification"],
)

print_counts(
    "NoClassification — T1 CLASSIFICATION GROUP",
    no_classification["_t1_group"],
)

print_counts(
    "NoClassification — T1 REVIEW STATUS",
    no_classification["_t1_review_status"],
)

print_counts(
    "NoClassification — T1 REVIEW STARS",
    no_classification[
        "t1_aggregate_review_stars"
    ],
)

print_counts(
    "NoClassification — T1 CONFLICT FLAG",
    no_classification["_t1_conflict"],
)

print_counts(
    "NoClassification — T1 SCV DISAGREEMENT FLAG",
    no_classification[
        "_t1_scv_disagreement"
    ],
)

print_counts(
    "NoClassification — TARGET GENE",
    no_classification["_t0_gene"],
)

print_counts(
    "NoClassification — LINKAGE DECISION",
    no_classification[
        "linkage_decision_category"
    ],
)


# --------------------------------------------------------------------------------------------------
# 7. Profile the corresponding T0 evidence
# --------------------------------------------------------------------------------------------------

print_counts(
    "CORRESPONDING T0 CLASSIFICATION GROUPS",
    normalize_text(
        no_classification[
            "t0_aggregate_classification_group"
        ]
    ),
)

print_counts(
    "CORRESPONDING T0 EXACT CLASSIFICATIONS",
    normalize_text(
        no_classification[
            "t0_aggregate_classification"
        ]
    ),
)

print_counts(
    "CORRESPONDING T0 REVIEW STATUS",
    normalize_text(
        no_classification[
            "t0_aggregate_review_status"
        ]
    ),
)

print_counts(
    "CORRESPONDING T0 CONFLICT FLAG",
    no_classification[
        "t0_aggregate_conflict_flag"
    ]
    .apply(normalize_boolean),
)


# --------------------------------------------------------------------------------------------------
# 8. Parse and inspect aggregate_classifications_json
# --------------------------------------------------------------------------------------------------

parsed_json = []
json_statuses = []
json_lengths = []

path_presence = Counter()
path_value_counts = defaultdict(Counter)

for value in no_classification[
    "t1_aggregate_classifications_json"
]:

    parsed, status = parse_json_list(
        value
    )

    parsed_json.append(parsed)
    json_statuses.append(status)

    if status == "PASS":
        json_lengths.append(
            len(parsed)
        )

        record_paths = set()

        for path, leaf_value in flatten_json(
            parsed
        ):
            record_paths.add(path)
            path_value_counts[path][
                leaf_value
            ] += 1

        for path in record_paths:
            path_presence[path] += 1

    else:
        json_lengths.append(
            pd.NA
        )

no_classification[
    "_parsed_classifications_json"
] = parsed_json

no_classification[
    "_classification_json_status"
] = json_statuses

no_classification[
    "_classification_json_length"
] = json_lengths

print_counts(
    "aggregate_classifications_json PARSE STATUS",
    no_classification[
        "_classification_json_status"
    ],
)

print_counts(
    "aggregate_classifications_json LIST LENGTH",
    no_classification[
        "_classification_json_length"
    ],
)

print("\nNESTED JSON LEAF-PATH INVENTORY")
print("-" * 110)

for path, count in sorted(
    path_presence.items(),
    key=lambda item: (
        -item[1],
        item[0],
    ),
):
    print(
        f"{path:70s} "
        f"present_in={count:>5,} records"
    )

    top_values = (
        path_value_counts[path]
        .most_common(10)
    )

    for value, value_count in top_values:
        print(
            f"    {value!r}: "
            f"{value_count:,}"
        )


# --------------------------------------------------------------------------------------------------
# 9. Display a few complete examples
# --------------------------------------------------------------------------------------------------

sample_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "_t0_gene",
    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t1_aggregate_classification_axis",
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
]

print("\nFIRST 10 NoClassification RECORDS")
print("-" * 110)

print(
    no_classification[
        sample_columns
    ]
    .head(10)
    .to_string(index=False)
)

print("\nFIRST 3 PARSED aggregate_classifications_json EXAMPLES")
print("-" * 110)

for index, parsed in enumerate(
    no_classification[
        "_parsed_classifications_json"
    ].head(3),
    start=1,
):
    print(f"\nExample {index}")

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. Focused consistency checks
# --------------------------------------------------------------------------------------------------

sentinel_mask = (
    no_classification[
        "_t1_classification"
    ]
    .str.casefold()
    .eq("evidence_only")
)

other_group_mask = (
    no_classification[
        "_t1_group"
    ]
    .str.casefold()
    .eq("other")
)

json_pass_mask = (
    no_classification[
        "_classification_json_status"
    ]
    .eq("PASS")
)

single_json_entry_mask = (
    pd.to_numeric(
        no_classification[
            "_classification_json_length"
        ],
        errors="coerce",
    )
    .eq(1)
)

conflict_positive_mask = (
    no_classification[
        "_t1_conflict"
    ]
    .eq(True)
)

gene_mismatch_mask = (
    no_classification[
        "_t0_gene"
    ]
    .ne(
        no_classification[
            "_t1_gene"
        ]
    )
)

print("\nFOCUSED CONSISTENCY RESULTS")
print("-" * 110)
print(
    f"Rows with exact evidence_only sentinel:     "
    f"{sentinel_mask.sum():,}"
)
print(
    f"Rows with a different sentinel/value:       "
    f"{(~sentinel_mask).sum():,}"
)
print(
    f"Rows normalized to group Other:             "
    f"{other_group_mask.sum():,}"
)
print(
    f"Rows with a different classification group: "
    f"{(~other_group_mask).sum():,}"
)
print(
    f"JSON values parsed successfully:            "
    f"{json_pass_mask.sum():,}"
)
print(
    f"JSON values with exactly one entry:         "
    f"{single_json_entry_mask.sum():,}"
)
print(
    f"T1 conflict-positive NoClassification rows: "
    f"{conflict_positive_mask.sum():,}"
)
print(
    f"T0/T1 target-gene mismatches:               "
    f"{gene_mismatch_mask.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 11. Diagnostic decision
# --------------------------------------------------------------------------------------------------

failures = {
    "unexpected eligible-row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "unexpected NoClassification-row count":
        len(no_classification)
        != EXPECTED_NO_CLASSIFICATION_ROWS,

    "NoClassification values are not uniformly evidence_only":
        not sentinel_mask.all(),

    "NoClassification rows are not uniformly group Other":
        not other_group_mask.all(),

    "aggregate classification JSON parsing failure":
        not json_pass_mask.all(),

    "aggregate classification JSON does not contain one entry per row":
        not single_json_entry_mask.all(),

    "NoClassification records are conflict-positive":
        conflict_positive_mask.any(),

    "target-gene mismatch":
        gene_mismatch_mask.any(),
}

failed_checks = [
    name
    for name, failed in failures.items()
    if failed
]

print("\n" + "=" * 110)
print("STAGE 5A STEP 3A DECISION")
print("=" * 110)

if failed_checks:

    print(
        "REVIEW_NOCLASSIFICATION_SENTINEL_SEMANTICS"
    )

    print("\nChecks requiring review:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "The NoClassification records are not yet "
        "semantically uniform enough to define an outcome policy."
    )

print(
    "PASS_NOCLASSIFICATION_EVIDENCE_ONLY_SENTINEL_CONFIRMED"
)

print()
print(
    "Interpretation: evidence_only is a harmonization sentinel "
    "for a T1 record without a selected clinical classification."
)
print(
    "It must not be interpreted as Pathogenic, VUS, Benign, "
    "Conflicting, stable, or unstable."
)
print()
print(
    "Primary-outcome policy frozen:      NO"
)
print(
    "Records censored for Stage 5:       NO — policy pending"
)
print(
    "Future outcomes created:            NO"
)
print(
    "GES scores loaded:                  NO"
)
print(
    "Files written or modified:          NO"
)

STAGE 5A STEP 3A — T1 NoClassification / evidence_only DIAGNOSTIC

BASIC ACCOUNTING
--------------------------------------------------------------------------------------------------------------
Stage 3H rows loaded:                    71,659
Outcome-eligible rows:                   70,583
Invalid eligibility values:              0
T1 NoClassification rows:                3,092
Expected T1 NoClassification rows:       3,092

NoClassification — T1 AGGREGATE CLASSIFICATION
--------------------------------------------------------------------------------------------------------------
_t1_classification
evidence_only    3092

NoClassification — T1 CLASSIFICATION GROUP
--------------------------------------------------------------------------------------------------------------
_t1_group
Other    3092

NoClassification — T1 REVIEW STATUS
--------------------------------------------------------------------------------------------------------------
_t1_review_status
no classification provided 

In [7]:
# ==================================================================================================
# STAGE 5A — STEP 3B
# INVENTORY CLASSIFICATION-GROUP AND CONFLICT TRANSITIONS
#
# Purpose:
#   1. Profile T0-to-T1 classification-group transitions.
#   2. Keep Germline, NoClassification, Oncogenicity, and
#      SomaticClinicalImpact axes separate.
#   3. Identify candidate clinically meaningful category crossings.
#   4. Profile new, persistent, resolved, and absent conflict states.
#   5. Compare conflict flags with normalized Conflicting groups.
#
# Important:
#   - Candidate transitions are NOT final outcomes.
#   - No outcome policy is frozen.
#   - No GES score or Stage 4 model artifact is loaded.
#   - No performance metric is calculated.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
import json

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage artifact
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load only the fields required for this semantic inventory
# --------------------------------------------------------------------------------------------------

columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "linkage_decision_category",
    "temporal_outcome_eligible",

    "t0_target_genes_json",
    "t1_target_genes_json",

    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",

    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_classification_axis",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
]

df = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=columns,
)


# --------------------------------------------------------------------------------------------------
# 3. Normalization helpers
# --------------------------------------------------------------------------------------------------

def normalize_boolean(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True
        if value == 0:
            return False

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().lower(),
        "INVALID",
    )


def parse_single_gene(value):
    if pd.isna(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    if len(genes) == 0:
        return "<EMPTY>"

    return "|".join(genes)


def canonical_group(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().casefold()

    mapping = {
        "benign/likely benign": "BLB",
        "vus": "VUS",
        "pathogenic/likely pathogenic": "PLP",
        "conflicting": "CONFLICTING",
        "other": "OTHER",
        "missing": "MISSING",
        "mixed": "MIXED",
    }

    return mapping.get(
        text,
        f"UNEXPECTED:{text}",
    )


def canonical_axis(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().casefold()

    mapping = {
        "germlineclassification": "GERMLINE",
        "noclassification": "NO_CLASSIFICATION",
        "oncogenicityclassification": "ONCOGENICITY",
        "somaticclinicalimpact": "SOMATIC_CLINICAL_IMPACT",
    }

    return mapping.get(
        text,
        f"UNEXPECTED:{text}",
    )


def print_counts(title, series):
    print(f"\n{title}")
    print("-" * 112)

    print(
        series.astype("string")
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 4. Restrict to the frozen outcome-eligible linkage set
# --------------------------------------------------------------------------------------------------

eligibility = (
    df["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

invalid_eligibility = int(
    eligibility.eq("INVALID").sum()
)

missing_eligibility = int(
    eligibility.isna().sum()
)

eligible = (
    df.loc[
        eligibility.eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_TOTAL_ROWS = 71_659
EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_NO_CLASSIFICATION_ROWS = 3_092
EXPECTED_NON_GERMLINE_AXIS_ROWS = 2


# --------------------------------------------------------------------------------------------------
# 5. Normalize genes, groups, axes, and conflict fields
# --------------------------------------------------------------------------------------------------

eligible["_t0_gene"] = (
    eligible["t0_target_genes_json"]
    .apply(parse_single_gene)
)

eligible["_t1_gene"] = (
    eligible["t1_target_genes_json"]
    .apply(parse_single_gene)
)

eligible["_t0_group"] = (
    eligible[
        "t0_aggregate_classification_group"
    ]
    .apply(canonical_group)
)

eligible["_t1_group"] = (
    eligible[
        "t1_aggregate_classification_group"
    ]
    .apply(canonical_group)
)

eligible["_t1_axis"] = (
    eligible[
        "t1_aggregate_classification_axis"
    ]
    .apply(canonical_axis)
)

eligible["_t0_conflict"] = (
    eligible[
        "t0_aggregate_conflict_flag"
    ]
    .apply(normalize_boolean)
)

eligible["_t1_conflict"] = (
    eligible[
        "t1_aggregate_conflict_flag"
    ]
    .apply(normalize_boolean)
)

eligible["_t0_scv_disagreement"] = (
    eligible[
        "t0_scv_group_disagreement_flag"
    ]
    .apply(normalize_boolean)
)

eligible["_t1_scv_disagreement"] = (
    eligible[
        "t1_scv_group_disagreement_flag"
    ]
    .apply(normalize_boolean)
)


# --------------------------------------------------------------------------------------------------
# 6. Basic accounting
# --------------------------------------------------------------------------------------------------

print("=" * 112)
print("STAGE 5A STEP 3B — CLASSIFICATION-GROUP AND CONFLICT TRANSITION INVENTORY")
print("=" * 112)

print("\nBASIC ACCOUNTING")
print("-" * 112)
print(
    f"Complete Stage 3H rows:              "
    f"{len(df):,}"
)
print(
    f"Outcome-eligible linked rows:        "
    f"{len(eligible):,}"
)
print(
    f"Missing eligibility values:          "
    f"{missing_eligibility:,}"
)
print(
    f"Invalid eligibility values:          "
    f"{invalid_eligibility:,}"
)


# --------------------------------------------------------------------------------------------------
# 7. Gene and axis integrity
# --------------------------------------------------------------------------------------------------

gene_mismatch_mask = (
    eligible["_t0_gene"]
    .ne(
        eligible["_t1_gene"]
    )
)

gene_mismatch_count = int(
    gene_mismatch_mask.sum()
)

unexpected_group_mask = (
    eligible["_t0_group"]
    .astype("string")
    .str.startswith(
        "UNEXPECTED:",
        na=False,
    )
    |
    eligible["_t1_group"]
    .astype("string")
    .str.startswith(
        "UNEXPECTED:",
        na=False,
    )
)

unexpected_axis_mask = (
    eligible["_t1_axis"]
    .astype("string")
    .str.startswith(
        "UNEXPECTED:",
        na=False,
    )
)

no_classification_mask = (
    eligible["_t1_axis"]
    .eq("NO_CLASSIFICATION")
)

non_germline_axis_mask = (
    eligible["_t1_axis"]
    .isin(
        {
            "ONCOGENICITY",
            "SOMATIC_CLINICAL_IMPACT",
        }
    )
)

germline_mask = (
    eligible["_t1_axis"]
    .eq("GERMLINE")
)

print("\nGENE AND AXIS INTEGRITY")
print("-" * 112)
print(
    f"T0/T1 target-gene mismatches:       "
    f"{gene_mismatch_count:,}"
)
print(
    f"Unexpected classification groups:   "
    f"{unexpected_group_mask.sum():,}"
)
print(
    f"Unexpected classification axes:     "
    f"{unexpected_axis_mask.sum():,}"
)
print(
    f"Germline-axis rows:                  "
    f"{germline_mask.sum():,}"
)
print(
    f"NoClassification rows:              "
    f"{no_classification_mask.sum():,}"
)
print(
    f"Non-germline selected-axis rows:     "
    f"{non_germline_axis_mask.sum():,}"
)

axis_by_gene = pd.crosstab(
    eligible["_t0_gene"],
    eligible["_t1_axis"],
    dropna=False,
)

print("\nT1 SELECTED AXIS BY TARGET GENE")
print("-" * 112)
print(
    axis_by_gene.to_string()
)


# --------------------------------------------------------------------------------------------------
# 8. Complete classification-group transition tables
# --------------------------------------------------------------------------------------------------

all_group_transition_table = pd.crosstab(
    eligible["_t0_group"],
    eligible["_t1_group"],
    margins=True,
    dropna=False,
)

germline_group_transition_table = pd.crosstab(
    eligible.loc[
        germline_mask,
        "_t0_group",
    ],
    eligible.loc[
        germline_mask,
        "_t1_group",
    ],
    margins=True,
    dropna=False,
)

print("\nALL ELIGIBLE T0 × T1 CLASSIFICATION-GROUP TRANSITIONS")
print("-" * 112)
print(
    all_group_transition_table.to_string()
)

print("\nGERMLINE-AXIS T0 × T1 CLASSIFICATION-GROUP TRANSITIONS")
print("-" * 112)
print(
    germline_group_transition_table.to_string()
)


# --------------------------------------------------------------------------------------------------
# 9. Create mutually exclusive semantic strata
# --------------------------------------------------------------------------------------------------

CLINICAL_TRIAD = {
    "BLB",
    "VUS",
    "PLP",
}


def assign_semantic_stratum(row):
    axis = row["_t1_axis"]
    t0_group = row["_t0_group"]
    t1_group = row["_t1_group"]

    if axis == "NO_CLASSIFICATION":
        return "NO_CLASSIFICATION_SENTINEL"

    if axis in {
        "ONCOGENICITY",
        "SOMATIC_CLINICAL_IMPACT",
    }:
        return "NON_GERMLINE_SELECTED_AXIS"

    if axis != "GERMLINE":
        return "UNEXPECTED_AXIS"

    if (
        t0_group in CLINICAL_TRIAD
        and t1_group in CLINICAL_TRIAD
    ):
        return "GERMLINE_TRIAD_TO_TRIAD"

    if (
        t0_group in CLINICAL_TRIAD
        and t1_group == "CONFLICTING"
    ):
        return "GERMLINE_TRIAD_TO_CONFLICTING"

    if (
        t0_group == "CONFLICTING"
        and t1_group in CLINICAL_TRIAD
    ):
        return "GERMLINE_CONFLICTING_TO_TRIAD"

    if (
        t0_group == "CONFLICTING"
        and t1_group == "CONFLICTING"
    ):
        return "GERMLINE_CONFLICTING_TO_CONFLICTING"

    return "GERMLINE_OTHER_OR_MISSING"


eligible["_semantic_stratum"] = (
    eligible.apply(
        assign_semantic_stratum,
        axis=1,
    )
)

print_counts(
    "MUTUALLY EXCLUSIVE SEMANTIC STRATA",
    eligible["_semantic_stratum"],
)

stratum_by_gene = pd.crosstab(
    eligible["_t0_gene"],
    eligible["_semantic_stratum"],
    dropna=False,
)

print("\nSEMANTIC STRATUM BY TARGET GENE")
print("-" * 112)
print(
    stratum_by_gene.to_string()
)


# --------------------------------------------------------------------------------------------------
# 10. Candidate clinically meaningful triad crossings
# --------------------------------------------------------------------------------------------------

triad_to_triad_mask = (
    germline_mask
    & eligible["_t0_group"].isin(
        CLINICAL_TRIAD
    )
    & eligible["_t1_group"].isin(
        CLINICAL_TRIAD
    )
)

triad_change_candidate_mask = (
    triad_to_triad_mask
    & eligible["_t0_group"]
    .ne(
        eligible["_t1_group"]
    )
)

triad_unchanged_mask = (
    triad_to_triad_mask
    & eligible["_t0_group"]
    .eq(
        eligible["_t1_group"]
    )
)

changed_triad_transitions = (
    eligible.loc[
        triad_change_candidate_mask,
        [
            "_t0_group",
            "_t1_group",
        ],
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .sort_values(
        [
            "_t0_group",
            "_t1_group",
        ]
    )
)

print("\nCLINICALLY MEANINGFUL TRIAD-TO-TRIAD INVENTORY")
print("-" * 112)
print(
    f"Triad-to-triad comparable records:     "
    f"{triad_to_triad_mask.sum():,}"
)
print(
    f"Same broad clinical group:             "
    f"{triad_unchanged_mask.sum():,}"
)
print(
    f"Different broad clinical group:        "
    f"{triad_change_candidate_mask.sum():,}"
)

print("\nCHANGED TRIAD TRANSITIONS")
print("-" * 112)

if len(changed_triad_transitions) > 0:
    print(
        changed_triad_transitions.to_string(
            index=False
        )
    )
else:
    print("None")


# --------------------------------------------------------------------------------------------------
# 11. Conflict-state transition inventory
# --------------------------------------------------------------------------------------------------

invalid_conflict_values = int(
    eligible["_t0_conflict"]
    .eq("INVALID")
    .sum()
    +
    eligible["_t1_conflict"]
    .eq("INVALID")
    .sum()
)

missing_conflict_values = int(
    eligible["_t0_conflict"]
    .isna()
    .sum()
    +
    eligible["_t1_conflict"]
    .isna()
    .sum()
)


def conflict_transition(row):
    t0_conflict = row["_t0_conflict"]
    t1_conflict = row["_t1_conflict"]

    if t0_conflict is False and t1_conflict is False:
        return "NO_CONFLICT_AT_EITHER_TIME"

    if t0_conflict is False and t1_conflict is True:
        return "NEW_CONFLICT_AT_T1"

    if t0_conflict is True and t1_conflict is True:
        return "CONFLICT_PERSISTED"

    if t0_conflict is True and t1_conflict is False:
        return "CONFLICT_RESOLVED_BY_T1"

    return "INVALID_OR_MISSING"


eligible["_conflict_transition"] = (
    eligible.apply(
        conflict_transition,
        axis=1,
    )
)

print_counts(
    "AGGREGATE CONFLICT-FLAG TRANSITIONS",
    eligible["_conflict_transition"],
)

conflict_transition_by_gene = pd.crosstab(
    eligible["_t0_gene"],
    eligible["_conflict_transition"],
    dropna=False,
)

print("\nCONFLICT TRANSITION BY TARGET GENE")
print("-" * 112)
print(
    conflict_transition_by_gene.to_string()
)


# --------------------------------------------------------------------------------------------------
# 12. Compare conflict flags with normalized Conflicting groups
# --------------------------------------------------------------------------------------------------

t0_group_conflicting = (
    eligible["_t0_group"]
    .eq("CONFLICTING")
)

t1_group_conflicting = (
    eligible["_t1_group"]
    .eq("CONFLICTING")
)

t0_flag_conflicting = (
    eligible["_t0_conflict"]
    .eq(True)
)

t1_flag_conflicting = (
    eligible["_t1_conflict"]
    .eq(True)
)

t0_group_flag_mismatch = (
    t0_group_conflicting
    .ne(
        t0_flag_conflicting
    )
)

t1_group_flag_mismatch = (
    t1_group_conflicting
    .ne(
        t1_flag_conflicting
    )
)

print("\nCONFLICT GROUP/FIELD CONSISTENCY")
print("-" * 112)
print(
    f"T0 group/flag mismatches:            "
    f"{t0_group_flag_mismatch.sum():,}"
)
print(
    f"T1 group/flag mismatches:            "
    f"{t1_group_flag_mismatch.sum():,}"
)

mismatch_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "_t0_gene",
    "t0_aggregate_classification",
    "_t0_group",
    "_t0_conflict",
    "t1_aggregate_classification",
    "_t1_group",
    "_t1_conflict",
    "_t1_axis",
]

mismatch_sample_mask = (
    t0_group_flag_mismatch
    | t1_group_flag_mismatch
)

print("\nFIRST 25 GROUP/FLAG MISMATCH RECORDS")
print("-" * 112)

if mismatch_sample_mask.any():
    print(
        eligible.loc[
            mismatch_sample_mask,
            mismatch_columns,
        ]
        .head(25)
        .to_string(index=False)
    )
else:
    print("None")


# --------------------------------------------------------------------------------------------------
# 13. Candidate new-conflict and conflict-resolution inventories
# --------------------------------------------------------------------------------------------------

new_conflict_flag_candidate = (
    germline_mask
    & eligible["_t0_conflict"].eq(False)
    & eligible["_t1_conflict"].eq(True)
)

new_conflict_group_candidate = (
    germline_mask
    & eligible["_t0_group"].ne(
        "CONFLICTING"
    )
    & eligible["_t1_group"].eq(
        "CONFLICTING"
    )
)

resolved_conflict_flag_candidate = (
    germline_mask
    & eligible["_t0_conflict"].eq(True)
    & eligible["_t1_conflict"].eq(False)
    & eligible["_t1_group"].isin(
        CLINICAL_TRIAD
    )
)

resolved_conflict_group_candidate = (
    germline_mask
    & eligible["_t0_group"].eq(
        "CONFLICTING"
    )
    & eligible["_t1_group"].isin(
        CLINICAL_TRIAD
    )
)

print("\nCANDIDATE CONFLICT-OUTCOME COMPONENTS")
print("-" * 112)
print(
    f"New conflict using Boolean flags:        "
    f"{new_conflict_flag_candidate.sum():,}"
)
print(
    f"New conflict using normalized groups:    "
    f"{new_conflict_group_candidate.sum():,}"
)
print(
    f"Resolved prior conflict using flags:     "
    f"{resolved_conflict_flag_candidate.sum():,}"
)
print(
    f"Resolved prior conflict using groups:    "
    f"{resolved_conflict_group_candidate.sum():,}"
)

print("\nT1 GROUP AFTER FLAG-BASED PRIOR-CONFLICT RESOLUTION")
print("-" * 112)

print(
    eligible.loc[
        resolved_conflict_flag_candidate,
        "_t1_group",
    ]
    .value_counts(dropna=False)
    .to_string()
)


# --------------------------------------------------------------------------------------------------
# 14. Structural decision
# --------------------------------------------------------------------------------------------------

critical_failures = {
    "unexpected total Stage 3H row count":
        len(df) != EXPECTED_TOTAL_ROWS,

    "unexpected eligible-row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "missing or invalid eligibility values":
        (
            missing_eligibility > 0
            or invalid_eligibility > 0
        ),

    "target-gene mismatch":
        gene_mismatch_count > 0,

    "unexpected classification group":
        unexpected_group_mask.any(),

    "unexpected classification axis":
        unexpected_axis_mask.any(),

    "unexpected NoClassification count":
        int(no_classification_mask.sum())
        != EXPECTED_NO_CLASSIFICATION_ROWS,

    "unexpected non-germline-axis count":
        int(non_germline_axis_mask.sum())
        != EXPECTED_NON_GERMLINE_AXIS_ROWS,

    "missing or invalid conflict values":
        (
            missing_conflict_values > 0
            or invalid_conflict_values > 0
        ),
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\n" + "=" * 112)
print("STAGE 5A STEP 3B DECISION")
print("=" * 112)

if failed_checks:
    print(
        "FAIL_STAGE5A_TRANSITION_INVENTORY"
    )

    print("\nFailed structural checks:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Stage 5A Step 3B found a structural failure. "
        "Do not define or freeze the outcome."
    )

print(
    "PASS_STAGE5A_CLASSIFICATION_AND_CONFLICT_TRANSITION_INVENTORY"
)

print()
print(
    "Important: all transition and conflict counts above "
    "remain diagnostic candidate counts."
)
print(
    "Conflict group/flag discrepancies, if present, must be "
    "examined before the outcome definition is frozen."
)
print()
print(
    "Future-instability outcome assigned: NO"
)
print(
    "Outcome policy frozen:               NO"
)
print(
    "GES scores loaded:                   NO"
)
print(
    "Temporal performance examined:       NO"
)
print(
    "Files written or modified:           NO"
)

STAGE 5A STEP 3B — CLASSIFICATION-GROUP AND CONFLICT TRANSITION INVENTORY

BASIC ACCOUNTING
----------------------------------------------------------------------------------------------------------------
Complete Stage 3H rows:              71,659
Outcome-eligible linked rows:        70,583
Missing eligibility values:          0
Invalid eligibility values:          0

GENE AND AXIS INTEGRITY
----------------------------------------------------------------------------------------------------------------
T0/T1 target-gene mismatches:       0
Unexpected classification groups:   0
Unexpected classification axes:     0
Germline-axis rows:                  67,489
NoClassification rows:              3,092
Non-germline selected-axis rows:     2

T1 SELECTED AXIS BY TARGET GENE
----------------------------------------------------------------------------------------------------------------
_t1_axis  GERMLINE  NO_CLASSIFICATION  ONCOGENICITY  SOMATIC_CLINICAL_IMPACT
_t0_gene                     

In [8]:
# ==================================================================================================
# STAGE 5A — STEP 3C
# AUDIT CLASSIFICATION-GROUP / AGGREGATE-CONFLICT-FLAG DIFFERENCES
#
# Purpose:
#   1. Examine every T0 and T1 record where the normalized classification
#      group and aggregate conflict flag do not agree.
#   2. Determine whether the conflict flag is explained by an explicitly
#      conflicting ClinVar review status.
#   3. Verify the frozen semantic conflict rule:
#
#         conflict =
#             classification group is Conflicting
#             OR review status explicitly reports conflicting interpretations/classifications
#
#   4. Establish which frozen field should control the later conflict-event
#      component without assigning any future outcome in this step.
#
# Scientific boundary:
#   - No future-instability outcome is assigned.
#   - No outcome rule is frozen.
#   - No GES score or Stage 4 model artifact is loaded.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Confirm required fields and load optional supporting evidence when available
# --------------------------------------------------------------------------------------------------

schema_columns = set(
    pq.ParquetFile(
        STAGE3H_LINKAGE_PATH
    ).schema.names
)

required_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "temporal_outcome_eligible",

    "t0_target_genes_json",
    "t1_target_genes_json",

    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",

    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classification_axis",
]

optional_columns = [
    "t0_aggregate_explanation",
    "t1_aggregate_explanation",

    "t0_scv_group_counts_json",
    "t1_scv_group_counts_json",

    "t0_scv_classification_counts_json",
    "t1_scv_classification_counts_json",

    "t1_aggregate_classifications_json",
]

missing_required_columns = sorted(
    set(required_columns).difference(
        schema_columns
    )
)

if missing_required_columns:
    raise RuntimeError(
        "Required Stage 5A Step 3C fields are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_required_columns
        )
    )

available_optional_columns = [
    column
    for column in optional_columns
    if column in schema_columns
]

columns_to_load = (
    required_columns
    + available_optional_columns
)

df = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=columns_to_load,
)


# --------------------------------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------------------------------

def is_missing_scalar(value) -> bool:
    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except Exception:
        pass

    return False


def normalize_boolean(value):
    if is_missing_scalar(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True

        if value == 0:
            return False

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().casefold(),
        "INVALID",
    )


def normalize_text(value):
    if is_missing_scalar(value):
        return pd.NA

    text = str(value).strip()

    if text.casefold() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return pd.NA

    return text


def canonical_group(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "benign/likely benign": "BLB",
        "vus": "VUS",
        "pathogenic/likely pathogenic": "PLP",
        "conflicting": "CONFLICTING",
        "other": "OTHER",
        "missing": "MISSING",
        "mixed": "MIXED",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def parse_single_gene(value):
    if is_missing_scalar(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    if not genes:
        return "<EMPTY>"

    return "|".join(genes)


def compact_json_summary(value, maximum_length=180):
    """
    Produce a compact diagnostic representation of a JSON value.
    This does not modify the source data.
    """

    if is_missing_scalar(value):
        return "<MISSING>"

    try:
        parsed = (
            value
            if isinstance(
                value,
                (dict, list),
            )
            else json.loads(str(value))
        )
    except Exception:
        text = str(value).strip()

        return (
            text
            if len(text) <= maximum_length
            else text[:maximum_length] + "..."
        )

    if isinstance(parsed, dict):
        summary = "; ".join(
            f"{key}={parsed[key]}"
            for key in sorted(parsed)
        )

    elif isinstance(parsed, list):
        items = []

        for item in parsed:
            if isinstance(item, dict):
                selected = []

                for key in [
                    "axis",
                    "classification",
                    "classification_group",
                    "review_status",
                ]:
                    if key in item:
                        selected.append(
                            f"{key}={item[key]}"
                        )

                items.append(
                    "|".join(selected)
                    if selected
                    else str(item)
                )
            else:
                items.append(str(item))

        summary = "; ".join(items)

    else:
        summary = str(parsed)

    return (
        summary
        if len(summary) <= maximum_length
        else summary[:maximum_length] + "..."
    )


def mismatch_type(
    group_is_conflicting,
    stored_flag,
):
    if group_is_conflicting and stored_flag is False:
        return "GROUP_CONFLICTING_FLAG_FALSE"

    if not group_is_conflicting and stored_flag is True:
        return "GROUP_NONCONFLICTING_FLAG_TRUE"

    return "GROUP_AND_FLAG_AGREE"


# --------------------------------------------------------------------------------------------------
# 4. Restrict to frozen outcome-eligible records
# --------------------------------------------------------------------------------------------------

eligibility = (
    df["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

invalid_eligibility_count = int(
    eligibility.eq("INVALID").sum()
)

missing_eligibility_count = int(
    eligibility.isna().sum()
)

eligible = (
    df.loc[
        eligibility.eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_TOTAL_ROWS = 71_659
EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_T0_GROUP_FLAG_DIFFERENCES = 4
EXPECTED_T1_GROUP_FLAG_DIFFERENCES = 27


# --------------------------------------------------------------------------------------------------
# 5. Normalize T0 and T1 semantic fields
# --------------------------------------------------------------------------------------------------

for timepoint in ["t0", "t1"]:

    eligible[f"_{timepoint}_gene"] = (
        eligible[
            f"{timepoint}_target_genes_json"
        ]
        .apply(parse_single_gene)
    )

    eligible[f"_{timepoint}_classification"] = (
        eligible[
            f"{timepoint}_aggregate_classification"
        ]
        .apply(normalize_text)
    )

    eligible[f"_{timepoint}_group"] = (
        eligible[
            f"{timepoint}_aggregate_classification_group"
        ]
        .apply(canonical_group)
    )

    eligible[f"_{timepoint}_review_status"] = (
        eligible[
            f"{timepoint}_aggregate_review_status"
        ]
        .apply(normalize_text)
    )

    eligible[f"_{timepoint}_conflict"] = (
        eligible[
            f"{timepoint}_aggregate_conflict_flag"
        ]
        .apply(normalize_boolean)
    )

    eligible[f"_{timepoint}_scv_disagreement"] = (
        eligible[
            f"{timepoint}_scv_group_disagreement_flag"
        ]
        .apply(normalize_boolean)
    )


# --------------------------------------------------------------------------------------------------
# 6. Explicit accepted review-status semantics
# --------------------------------------------------------------------------------------------------

T0_CONFLICT_REVIEW_STATUSES = {
    "criteria provided, conflicting interpretations",
}

T1_CONFLICT_REVIEW_STATUSES = {
    "criteria provided, conflicting classifications",
}

eligible["_t0_review_status_conflicting"] = (
    eligible["_t0_review_status"]
    .astype("string")
    .str.casefold()
    .isin(T0_CONFLICT_REVIEW_STATUSES)
)

eligible["_t1_review_status_conflicting"] = (
    eligible["_t1_review_status"]
    .astype("string")
    .str.casefold()
    .isin(T1_CONFLICT_REVIEW_STATUSES)
)

eligible["_t0_group_conflicting"] = (
    eligible["_t0_group"]
    .eq("CONFLICTING")
)

eligible["_t1_group_conflicting"] = (
    eligible["_t1_group"]
    .eq("CONFLICTING")
)


# --------------------------------------------------------------------------------------------------
# 7. Reconstruct the semantic conflict rule independently
# --------------------------------------------------------------------------------------------------

eligible["_t0_expected_conflict"] = (
    eligible["_t0_group_conflicting"]
    | eligible[
        "_t0_review_status_conflicting"
    ]
)

eligible["_t1_expected_conflict"] = (
    eligible["_t1_group_conflicting"]
    | eligible[
        "_t1_review_status_conflicting"
    ]
)

eligible["_t0_group_flag_difference"] = (
    eligible["_t0_group_conflicting"]
    .ne(
        eligible["_t0_conflict"].eq(True)
    )
)

eligible["_t1_group_flag_difference"] = (
    eligible["_t1_group_conflicting"]
    .ne(
        eligible["_t1_conflict"].eq(True)
    )
)

eligible["_t0_expected_flag_difference"] = (
    eligible["_t0_expected_conflict"]
    .ne(
        eligible["_t0_conflict"].eq(True)
    )
)

eligible["_t1_expected_flag_difference"] = (
    eligible["_t1_expected_conflict"]
    .ne(
        eligible["_t1_conflict"].eq(True)
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Basic accounting
# --------------------------------------------------------------------------------------------------

t0_group_flag_difference_count = int(
    eligible[
        "_t0_group_flag_difference"
    ].sum()
)

t1_group_flag_difference_count = int(
    eligible[
        "_t1_group_flag_difference"
    ].sum()
)

t0_expected_flag_difference_count = int(
    eligible[
        "_t0_expected_flag_difference"
    ].sum()
)

t1_expected_flag_difference_count = int(
    eligible[
        "_t1_expected_flag_difference"
    ].sum()
)

print("=" * 118)
print("STAGE 5A STEP 3C — CONFLICT GROUP/FIELD SEMANTIC AUDIT")
print("=" * 118)

print("\nBASIC ACCOUNTING")
print("-" * 118)
print(
    f"Stage 3H rows loaded:                         "
    f"{len(df):,}"
)
print(
    f"Outcome-eligible rows:                        "
    f"{len(eligible):,}"
)
print(
    f"Missing eligibility values:                   "
    f"{missing_eligibility_count:,}"
)
print(
    f"Invalid eligibility values:                   "
    f"{invalid_eligibility_count:,}"
)
print(
    f"T0 group/flag differences:                    "
    f"{t0_group_flag_difference_count:,}"
)
print(
    f"T1 group/flag differences:                    "
    f"{t1_group_flag_difference_count:,}"
)
print(
    f"T0 stored flag vs semantic-rule differences:  "
    f"{t0_expected_flag_difference_count:,}"
)
print(
    f"T1 stored flag vs semantic-rule differences:  "
    f"{t1_expected_flag_difference_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 9. Build one long-form table containing all group/flag differences
# --------------------------------------------------------------------------------------------------

def build_difference_table(
    frame,
    timepoint,
):
    mask = frame[
        f"_{timepoint}_group_flag_difference"
    ]

    selected = (
        frame.loc[mask]
        .copy()
        .reset_index(drop=True)
    )

    result = pd.DataFrame(
        {
            "timepoint": timepoint.upper(),

            "rcv_accession": selected[
                f"{timepoint}_rcv_accession"
            ],

            "gene": selected[
                f"_{timepoint}_gene"
            ],

            "classification": selected[
                f"_{timepoint}_classification"
            ],

            "classification_group": selected[
                f"_{timepoint}_group"
            ],

            "review_status": selected[
                f"_{timepoint}_review_status"
            ],

            "review_stars": selected[
                f"{timepoint}_aggregate_review_stars"
            ],

            "stored_conflict_flag": selected[
                f"_{timepoint}_conflict"
            ],

            "group_is_conflicting": selected[
                f"_{timepoint}_group_conflicting"
            ],

            "review_status_is_conflicting": selected[
                f"_{timepoint}_review_status_conflicting"
            ],

            "semantic_expected_conflict": selected[
                f"_{timepoint}_expected_conflict"
            ],

            "scv_group_disagreement": selected[
                f"_{timepoint}_scv_disagreement"
            ],
        }
    )

    result["difference_type"] = [
        mismatch_type(
            group_is_conflicting,
            stored_flag,
        )
        for group_is_conflicting, stored_flag in zip(
            result["group_is_conflicting"],
            result["stored_conflict_flag"],
        )
    ]

    group_counts_column = (
        f"{timepoint}_scv_group_counts_json"
    )

    if group_counts_column in selected.columns:
        result["scv_group_counts"] = (
            selected[
                group_counts_column
            ]
            .apply(compact_json_summary)
        )
    else:
        result["scv_group_counts"] = (
            "<FIELD_NOT_AVAILABLE>"
        )

    classification_counts_column = (
        f"{timepoint}_scv_classification_counts_json"
    )

    if classification_counts_column in selected.columns:
        result["scv_classification_counts"] = (
            selected[
                classification_counts_column
            ]
            .apply(compact_json_summary)
        )
    else:
        result["scv_classification_counts"] = (
            "<FIELD_NOT_AVAILABLE>"
        )

    if (
        timepoint == "t1"
        and "t1_aggregate_classifications_json"
        in selected.columns
    ):
        result["aggregate_classifications"] = (
            selected[
                "t1_aggregate_classifications_json"
            ]
            .apply(compact_json_summary)
        )
    else:
        result["aggregate_classifications"] = (
            "<NOT_APPLICABLE>"
        )

    return result


t0_differences = build_difference_table(
    eligible,
    "t0",
)

t1_differences = build_difference_table(
    eligible,
    "t1",
)

all_differences = (
    pd.concat(
        [
            t0_differences,
            t1_differences,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "timepoint",
            "gene",
            "rcv_accession",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 10. Summarize all difference patterns
# --------------------------------------------------------------------------------------------------

print("\nDIFFERENCE TYPE DISTRIBUTION")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "difference_type",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)

print("\nREVIEW STATUS AMONG GROUP/FLAG DIFFERENCES")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "review_status",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)

print("\nCLASSIFICATION GROUP AMONG GROUP/FLAG DIFFERENCES")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "classification_group",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)

print("\nGENE DISTRIBUTION AMONG GROUP/FLAG DIFFERENCES")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "gene",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 11. Display every difference record
# --------------------------------------------------------------------------------------------------

display_columns = [
    "timepoint",
    "rcv_accession",
    "gene",
    "classification",
    "classification_group",
    "review_status",
    "review_stars",
    "stored_conflict_flag",
    "group_is_conflicting",
    "review_status_is_conflicting",
    "semantic_expected_conflict",
    "difference_type",
    "scv_group_disagreement",
    "scv_group_counts",
]

print("\nALL GROUP/FLAG DIFFERENCE RECORDS")
print("-" * 118)

if len(all_differences) == 0:
    print("None")
else:
    print(
        all_differences[
            display_columns
        ]
        .to_string(index=False)
    )


# --------------------------------------------------------------------------------------------------
# 12. Focused direction and explanation checks
# --------------------------------------------------------------------------------------------------

t0_group_conflicting_flag_false = int(
    (
        eligible["_t0_group_conflicting"]
        & eligible["_t0_conflict"].eq(False)
    ).sum()
)

t1_group_conflicting_flag_false = int(
    (
        eligible["_t1_group_conflicting"]
        & eligible["_t1_conflict"].eq(False)
    ).sum()
)

t0_nonconflicting_group_flag_true = int(
    (
        ~eligible["_t0_group_conflicting"]
        & eligible["_t0_conflict"].eq(True)
    ).sum()
)

t1_nonconflicting_group_flag_true = int(
    (
        ~eligible["_t1_group_conflicting"]
        & eligible["_t1_conflict"].eq(True)
    ).sum()
)

all_t0_differences_explained_by_review_status = bool(
    eligible.loc[
        eligible[
            "_t0_group_flag_difference"
        ],
        "_t0_review_status_conflicting",
    ].all()
)

all_t1_differences_explained_by_review_status = bool(
    eligible.loc[
        eligible[
            "_t1_group_flag_difference"
        ],
        "_t1_review_status_conflicting",
    ].all()
)

print("\nFOCUSED SEMANTIC RESULTS")
print("-" * 118)
print(
    f"T0 group=Conflicting but flag=False:             "
    f"{t0_group_conflicting_flag_false:,}"
)
print(
    f"T1 group=Conflicting but flag=False:             "
    f"{t1_group_conflicting_flag_false:,}"
)
print(
    f"T0 group not Conflicting but flag=True:          "
    f"{t0_nonconflicting_group_flag_true:,}"
)
print(
    f"T1 group not Conflicting but flag=True:          "
    f"{t1_nonconflicting_group_flag_true:,}"
)
print(
    f"All T0 differences explained by review status:   "
    f"{all_t0_differences_explained_by_review_status}"
)
print(
    f"All T1 differences explained by review status:   "
    f"{all_t1_differences_explained_by_review_status}"
)


# --------------------------------------------------------------------------------------------------
# 13. Final diagnostic decision
# --------------------------------------------------------------------------------------------------

invalid_conflict_values = int(
    eligible["_t0_conflict"]
    .eq("INVALID")
    .sum()
    +
    eligible["_t1_conflict"]
    .eq("INVALID")
    .sum()
)

missing_conflict_values = int(
    eligible["_t0_conflict"]
    .isna()
    .sum()
    +
    eligible["_t1_conflict"]
    .isna()
    .sum()
)

critical_failures = {
    "unexpected Stage 3H row count":
        len(df) != EXPECTED_TOTAL_ROWS,

    "unexpected eligible-row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "missing or invalid eligibility":
        (
            missing_eligibility_count > 0
            or invalid_eligibility_count > 0
        ),

    "missing or invalid conflict flags":
        (
            missing_conflict_values > 0
            or invalid_conflict_values > 0
        ),

    "unexpected T0 group/flag difference count":
        t0_group_flag_difference_count
        != EXPECTED_T0_GROUP_FLAG_DIFFERENCES,

    "unexpected T1 group/flag difference count":
        t1_group_flag_difference_count
        != EXPECTED_T1_GROUP_FLAG_DIFFERENCES,

    "T0 frozen conflict flag violates semantic rule":
        t0_expected_flag_difference_count > 0,

    "T1 frozen conflict flag violates semantic rule":
        t1_expected_flag_difference_count > 0,

    "T0 Conflicting group has a false conflict flag":
        t0_group_conflicting_flag_false > 0,

    "T1 Conflicting group has a false conflict flag":
        t1_group_conflicting_flag_false > 0,

    "T0 differences not explained by conflicting review status":
        not all_t0_differences_explained_by_review_status,

    "T1 differences not explained by conflicting review status":
        not all_t1_differences_explained_by_review_status,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\n" + "=" * 118)
print("STAGE 5A STEP 3C DECISION")
print("=" * 118)

if failed_checks:

    print(
        "REVIEW_CONFLICT_FIELD_SEMANTICS"
    )

    print("\nChecks requiring review:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Conflict semantics are not yet sufficiently "
        "resolved to define the future-conflict outcome."
    )

print(
    "PASS_FROZEN_AGGREGATE_CONFLICT_FLAG_SEMANTICS_CONFIRMED"
)

print()
print(
    "Interpretation:"
)
print(
    "The classification group represents the selected aggregate "
    "clinical category."
)
print(
    "The aggregate conflict flag additionally captures records whose "
    "review status explicitly reports conflicting submissions."
)
print(
    "Therefore, the later new-conflict and conflict-resolution components "
    "should use the frozen aggregate conflict flag rather than relying "
    "only on classification_group='Conflicting'."
)

print()
print(
    "Outcome conflict rule frozen:       NO — diagnostic confirmation only"
)
print(
    "Future-instability outcome created: NO"
)
print(
    "GES scores loaded:                  NO"
)
print(
    "Temporal performance examined:      NO"
)
print(
    "Files written or modified:          NO"
)

STAGE 5A STEP 3C — CONFLICT GROUP/FIELD SEMANTIC AUDIT

BASIC ACCOUNTING
----------------------------------------------------------------------------------------------------------------------
Stage 3H rows loaded:                         71,659
Outcome-eligible rows:                        70,583
Missing eligibility values:                   0
Invalid eligibility values:                   0
T0 group/flag differences:                    4
T1 group/flag differences:                    27
T0 stored flag vs semantic-rule differences:  0
T1 stored flag vs semantic-rule differences:  0

DIFFERENCE TYPE DISTRIBUTION
----------------------------------------------------------------------------------------------------------------------
timepoint                difference_type  record_count
       T1 GROUP_NONCONFLICTING_FLAG_TRUE            27
       T0 GROUP_NONCONFLICTING_FLAG_TRUE             4

REVIEW STATUS AMONG GROUP/FLAG DIFFERENCES
-------------------------------------------------------

In [9]:
# ==================================================================================================
# STAGE 5A — STEP 3C
# AUDIT CLASSIFICATION-GROUP / AGGREGATE-CONFLICT-FLAG DIFFERENCES
#
# Purpose:
#   1. Examine every T0 and T1 record where the normalized classification
#      group and aggregate conflict flag do not agree.
#   2. Determine whether the conflict flag is explained by an explicitly
#      conflicting ClinVar review status.
#   3. Verify the frozen semantic conflict rule:
#
#         conflict =
#             classification group is Conflicting
#             OR review status explicitly reports conflicting interpretations/classifications
#
#   4. Establish which frozen field should control the later conflict-event
#      component without assigning any future outcome in this step.
#
# Scientific boundary:
#   - No future-instability outcome is assigned.
#   - No outcome rule is frozen.
#   - No GES score or Stage 4 model artifact is loaded.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Confirm required fields and load optional supporting evidence when available
# --------------------------------------------------------------------------------------------------

schema_columns = set(
    pq.ParquetFile(
        STAGE3H_LINKAGE_PATH
    ).schema.names
)

required_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "temporal_outcome_eligible",

    "t0_target_genes_json",
    "t1_target_genes_json",

    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",

    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classification_axis",
]

optional_columns = [
    "t0_aggregate_explanation",
    "t1_aggregate_explanation",

    "t0_scv_group_counts_json",
    "t1_scv_group_counts_json",

    "t0_scv_classification_counts_json",
    "t1_scv_classification_counts_json",

    "t1_aggregate_classifications_json",
]

missing_required_columns = sorted(
    set(required_columns).difference(
        schema_columns
    )
)

if missing_required_columns:
    raise RuntimeError(
        "Required Stage 5A Step 3C fields are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_required_columns
        )
    )

available_optional_columns = [
    column
    for column in optional_columns
    if column in schema_columns
]

columns_to_load = (
    required_columns
    + available_optional_columns
)

df = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=columns_to_load,
)


# --------------------------------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------------------------------

def is_missing_scalar(value) -> bool:
    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except Exception:
        pass

    return False


def normalize_boolean(value):
    if is_missing_scalar(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True

        if value == 0:
            return False

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().casefold(),
        "INVALID",
    )


def normalize_text(value):
    if is_missing_scalar(value):
        return pd.NA

    text = str(value).strip()

    if text.casefold() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return pd.NA

    return text


def canonical_group(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "benign/likely benign": "BLB",
        "vus": "VUS",
        "pathogenic/likely pathogenic": "PLP",
        "conflicting": "CONFLICTING",
        "other": "OTHER",
        "missing": "MISSING",
        "mixed": "MIXED",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def parse_single_gene(value):
    if is_missing_scalar(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    if not genes:
        return "<EMPTY>"

    return "|".join(genes)


def compact_json_summary(value, maximum_length=180):
    """
    Produce a compact diagnostic representation of a JSON value.
    This does not modify the source data.
    """

    if is_missing_scalar(value):
        return "<MISSING>"

    try:
        parsed = (
            value
            if isinstance(
                value,
                (dict, list),
            )
            else json.loads(str(value))
        )
    except Exception:
        text = str(value).strip()

        return (
            text
            if len(text) <= maximum_length
            else text[:maximum_length] + "..."
        )

    if isinstance(parsed, dict):
        summary = "; ".join(
            f"{key}={parsed[key]}"
            for key in sorted(parsed)
        )

    elif isinstance(parsed, list):
        items = []

        for item in parsed:
            if isinstance(item, dict):
                selected = []

                for key in [
                    "axis",
                    "classification",
                    "classification_group",
                    "review_status",
                ]:
                    if key in item:
                        selected.append(
                            f"{key}={item[key]}"
                        )

                items.append(
                    "|".join(selected)
                    if selected
                    else str(item)
                )
            else:
                items.append(str(item))

        summary = "; ".join(items)

    else:
        summary = str(parsed)

    return (
        summary
        if len(summary) <= maximum_length
        else summary[:maximum_length] + "..."
    )


def mismatch_type(
    group_is_conflicting,
    stored_flag,
):
    if group_is_conflicting and stored_flag is False:
        return "GROUP_CONFLICTING_FLAG_FALSE"

    if not group_is_conflicting and stored_flag is True:
        return "GROUP_NONCONFLICTING_FLAG_TRUE"

    return "GROUP_AND_FLAG_AGREE"


# --------------------------------------------------------------------------------------------------
# 4. Restrict to frozen outcome-eligible records
# --------------------------------------------------------------------------------------------------

eligibility = (
    df["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

invalid_eligibility_count = int(
    eligibility.eq("INVALID").sum()
)

missing_eligibility_count = int(
    eligibility.isna().sum()
)

eligible = (
    df.loc[
        eligibility.eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_TOTAL_ROWS = 71_659
EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_T0_GROUP_FLAG_DIFFERENCES = 4
EXPECTED_T1_GROUP_FLAG_DIFFERENCES = 27


# --------------------------------------------------------------------------------------------------
# 5. Normalize T0 and T1 semantic fields
# --------------------------------------------------------------------------------------------------

for timepoint in ["t0", "t1"]:

    eligible[f"_{timepoint}_gene"] = (
        eligible[
            f"{timepoint}_target_genes_json"
        ]
        .apply(parse_single_gene)
    )

    eligible[f"_{timepoint}_classification"] = (
        eligible[
            f"{timepoint}_aggregate_classification"
        ]
        .apply(normalize_text)
    )

    eligible[f"_{timepoint}_group"] = (
        eligible[
            f"{timepoint}_aggregate_classification_group"
        ]
        .apply(canonical_group)
    )

    eligible[f"_{timepoint}_review_status"] = (
        eligible[
            f"{timepoint}_aggregate_review_status"
        ]
        .apply(normalize_text)
    )

    eligible[f"_{timepoint}_conflict"] = (
        eligible[
            f"{timepoint}_aggregate_conflict_flag"
        ]
        .apply(normalize_boolean)
    )

    eligible[f"_{timepoint}_scv_disagreement"] = (
        eligible[
            f"{timepoint}_scv_group_disagreement_flag"
        ]
        .apply(normalize_boolean)
    )


# --------------------------------------------------------------------------------------------------
# 6. Explicit accepted review-status semantics
# --------------------------------------------------------------------------------------------------

T0_CONFLICT_REVIEW_STATUSES = {
    "criteria provided, conflicting interpretations",
}

T1_CONFLICT_REVIEW_STATUSES = {
    "criteria provided, conflicting classifications",
}

eligible["_t0_review_status_conflicting"] = (
    eligible["_t0_review_status"]
    .astype("string")
    .str.casefold()
    .isin(T0_CONFLICT_REVIEW_STATUSES)
)

eligible["_t1_review_status_conflicting"] = (
    eligible["_t1_review_status"]
    .astype("string")
    .str.casefold()
    .isin(T1_CONFLICT_REVIEW_STATUSES)
)

eligible["_t0_group_conflicting"] = (
    eligible["_t0_group"]
    .eq("CONFLICTING")
)

eligible["_t1_group_conflicting"] = (
    eligible["_t1_group"]
    .eq("CONFLICTING")
)


# --------------------------------------------------------------------------------------------------
# 7. Reconstruct the semantic conflict rule independently
# --------------------------------------------------------------------------------------------------

eligible["_t0_expected_conflict"] = (
    eligible["_t0_group_conflicting"]
    | eligible[
        "_t0_review_status_conflicting"
    ]
)

eligible["_t1_expected_conflict"] = (
    eligible["_t1_group_conflicting"]
    | eligible[
        "_t1_review_status_conflicting"
    ]
)

eligible["_t0_group_flag_difference"] = (
    eligible["_t0_group_conflicting"]
    .ne(
        eligible["_t0_conflict"].eq(True)
    )
)

eligible["_t1_group_flag_difference"] = (
    eligible["_t1_group_conflicting"]
    .ne(
        eligible["_t1_conflict"].eq(True)
    )
)

eligible["_t0_expected_flag_difference"] = (
    eligible["_t0_expected_conflict"]
    .ne(
        eligible["_t0_conflict"].eq(True)
    )
)

eligible["_t1_expected_flag_difference"] = (
    eligible["_t1_expected_conflict"]
    .ne(
        eligible["_t1_conflict"].eq(True)
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Basic accounting
# --------------------------------------------------------------------------------------------------

t0_group_flag_difference_count = int(
    eligible[
        "_t0_group_flag_difference"
    ].sum()
)

t1_group_flag_difference_count = int(
    eligible[
        "_t1_group_flag_difference"
    ].sum()
)

t0_expected_flag_difference_count = int(
    eligible[
        "_t0_expected_flag_difference"
    ].sum()
)

t1_expected_flag_difference_count = int(
    eligible[
        "_t1_expected_flag_difference"
    ].sum()
)

print("=" * 118)
print("STAGE 5A STEP 3C — CONFLICT GROUP/FIELD SEMANTIC AUDIT")
print("=" * 118)

print("\nBASIC ACCOUNTING")
print("-" * 118)
print(
    f"Stage 3H rows loaded:                         "
    f"{len(df):,}"
)
print(
    f"Outcome-eligible rows:                        "
    f"{len(eligible):,}"
)
print(
    f"Missing eligibility values:                   "
    f"{missing_eligibility_count:,}"
)
print(
    f"Invalid eligibility values:                   "
    f"{invalid_eligibility_count:,}"
)
print(
    f"T0 group/flag differences:                    "
    f"{t0_group_flag_difference_count:,}"
)
print(
    f"T1 group/flag differences:                    "
    f"{t1_group_flag_difference_count:,}"
)
print(
    f"T0 stored flag vs semantic-rule differences:  "
    f"{t0_expected_flag_difference_count:,}"
)
print(
    f"T1 stored flag vs semantic-rule differences:  "
    f"{t1_expected_flag_difference_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 9. Build one long-form table containing all group/flag differences
# --------------------------------------------------------------------------------------------------

def build_difference_table(
    frame,
    timepoint,
):
    mask = frame[
        f"_{timepoint}_group_flag_difference"
    ]

    selected = (
        frame.loc[mask]
        .copy()
        .reset_index(drop=True)
    )

    result = pd.DataFrame(
        {
            "timepoint": timepoint.upper(),

            "rcv_accession": selected[
                f"{timepoint}_rcv_accession"
            ],

            "gene": selected[
                f"_{timepoint}_gene"
            ],

            "classification": selected[
                f"_{timepoint}_classification"
            ],

            "classification_group": selected[
                f"_{timepoint}_group"
            ],

            "review_status": selected[
                f"_{timepoint}_review_status"
            ],

            "review_stars": selected[
                f"{timepoint}_aggregate_review_stars"
            ],

            "stored_conflict_flag": selected[
                f"_{timepoint}_conflict"
            ],

            "group_is_conflicting": selected[
                f"_{timepoint}_group_conflicting"
            ],

            "review_status_is_conflicting": selected[
                f"_{timepoint}_review_status_conflicting"
            ],

            "semantic_expected_conflict": selected[
                f"_{timepoint}_expected_conflict"
            ],

            "scv_group_disagreement": selected[
                f"_{timepoint}_scv_disagreement"
            ],
        }
    )

    result["difference_type"] = [
        mismatch_type(
            group_is_conflicting,
            stored_flag,
        )
        for group_is_conflicting, stored_flag in zip(
            result["group_is_conflicting"],
            result["stored_conflict_flag"],
        )
    ]

    group_counts_column = (
        f"{timepoint}_scv_group_counts_json"
    )

    if group_counts_column in selected.columns:
        result["scv_group_counts"] = (
            selected[
                group_counts_column
            ]
            .apply(compact_json_summary)
        )
    else:
        result["scv_group_counts"] = (
            "<FIELD_NOT_AVAILABLE>"
        )

    classification_counts_column = (
        f"{timepoint}_scv_classification_counts_json"
    )

    if classification_counts_column in selected.columns:
        result["scv_classification_counts"] = (
            selected[
                classification_counts_column
            ]
            .apply(compact_json_summary)
        )
    else:
        result["scv_classification_counts"] = (
            "<FIELD_NOT_AVAILABLE>"
        )

    if (
        timepoint == "t1"
        and "t1_aggregate_classifications_json"
        in selected.columns
    ):
        result["aggregate_classifications"] = (
            selected[
                "t1_aggregate_classifications_json"
            ]
            .apply(compact_json_summary)
        )
    else:
        result["aggregate_classifications"] = (
            "<NOT_APPLICABLE>"
        )

    return result


t0_differences = build_difference_table(
    eligible,
    "t0",
)

t1_differences = build_difference_table(
    eligible,
    "t1",
)

all_differences = (
    pd.concat(
        [
            t0_differences,
            t1_differences,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "timepoint",
            "gene",
            "rcv_accession",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 10. Summarize all difference patterns
# --------------------------------------------------------------------------------------------------

print("\nDIFFERENCE TYPE DISTRIBUTION")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "difference_type",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)

print("\nREVIEW STATUS AMONG GROUP/FLAG DIFFERENCES")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "review_status",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)

print("\nCLASSIFICATION GROUP AMONG GROUP/FLAG DIFFERENCES")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "classification_group",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)

print("\nGENE DISTRIBUTION AMONG GROUP/FLAG DIFFERENCES")
print("-" * 118)

print(
    all_differences[
        [
            "timepoint",
            "gene",
        ]
    ]
    .value_counts()
    .rename("record_count")
    .reset_index()
    .to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 11. Display every difference record
# --------------------------------------------------------------------------------------------------

display_columns = [
    "timepoint",
    "rcv_accession",
    "gene",
    "classification",
    "classification_group",
    "review_status",
    "review_stars",
    "stored_conflict_flag",
    "group_is_conflicting",
    "review_status_is_conflicting",
    "semantic_expected_conflict",
    "difference_type",
    "scv_group_disagreement",
    "scv_group_counts",
]

print("\nALL GROUP/FLAG DIFFERENCE RECORDS")
print("-" * 118)

if len(all_differences) == 0:
    print("None")
else:
    print(
        all_differences[
            display_columns
        ]
        .to_string(index=False)
    )


# --------------------------------------------------------------------------------------------------
# 12. Focused direction and explanation checks
# --------------------------------------------------------------------------------------------------

t0_group_conflicting_flag_false = int(
    (
        eligible["_t0_group_conflicting"]
        & eligible["_t0_conflict"].eq(False)
    ).sum()
)

t1_group_conflicting_flag_false = int(
    (
        eligible["_t1_group_conflicting"]
        & eligible["_t1_conflict"].eq(False)
    ).sum()
)

t0_nonconflicting_group_flag_true = int(
    (
        ~eligible["_t0_group_conflicting"]
        & eligible["_t0_conflict"].eq(True)
    ).sum()
)

t1_nonconflicting_group_flag_true = int(
    (
        ~eligible["_t1_group_conflicting"]
        & eligible["_t1_conflict"].eq(True)
    ).sum()
)

all_t0_differences_explained_by_review_status = bool(
    eligible.loc[
        eligible[
            "_t0_group_flag_difference"
        ],
        "_t0_review_status_conflicting",
    ].all()
)

all_t1_differences_explained_by_review_status = bool(
    eligible.loc[
        eligible[
            "_t1_group_flag_difference"
        ],
        "_t1_review_status_conflicting",
    ].all()
)

print("\nFOCUSED SEMANTIC RESULTS")
print("-" * 118)
print(
    f"T0 group=Conflicting but flag=False:             "
    f"{t0_group_conflicting_flag_false:,}"
)
print(
    f"T1 group=Conflicting but flag=False:             "
    f"{t1_group_conflicting_flag_false:,}"
)
print(
    f"T0 group not Conflicting but flag=True:          "
    f"{t0_nonconflicting_group_flag_true:,}"
)
print(
    f"T1 group not Conflicting but flag=True:          "
    f"{t1_nonconflicting_group_flag_true:,}"
)
print(
    f"All T0 differences explained by review status:   "
    f"{all_t0_differences_explained_by_review_status}"
)
print(
    f"All T1 differences explained by review status:   "
    f"{all_t1_differences_explained_by_review_status}"
)


# --------------------------------------------------------------------------------------------------
# 13. Final diagnostic decision
# --------------------------------------------------------------------------------------------------

invalid_conflict_values = int(
    eligible["_t0_conflict"]
    .eq("INVALID")
    .sum()
    +
    eligible["_t1_conflict"]
    .eq("INVALID")
    .sum()
)

missing_conflict_values = int(
    eligible["_t0_conflict"]
    .isna()
    .sum()
    +
    eligible["_t1_conflict"]
    .isna()
    .sum()
)

critical_failures = {
    "unexpected Stage 3H row count":
        len(df) != EXPECTED_TOTAL_ROWS,

    "unexpected eligible-row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "missing or invalid eligibility":
        (
            missing_eligibility_count > 0
            or invalid_eligibility_count > 0
        ),

    "missing or invalid conflict flags":
        (
            missing_conflict_values > 0
            or invalid_conflict_values > 0
        ),

    "unexpected T0 group/flag difference count":
        t0_group_flag_difference_count
        != EXPECTED_T0_GROUP_FLAG_DIFFERENCES,

    "unexpected T1 group/flag difference count":
        t1_group_flag_difference_count
        != EXPECTED_T1_GROUP_FLAG_DIFFERENCES,

    "T0 frozen conflict flag violates semantic rule":
        t0_expected_flag_difference_count > 0,

    "T1 frozen conflict flag violates semantic rule":
        t1_expected_flag_difference_count > 0,

    "T0 Conflicting group has a false conflict flag":
        t0_group_conflicting_flag_false > 0,

    "T1 Conflicting group has a false conflict flag":
        t1_group_conflicting_flag_false > 0,

    "T0 differences not explained by conflicting review status":
        not all_t0_differences_explained_by_review_status,

    "T1 differences not explained by conflicting review status":
        not all_t1_differences_explained_by_review_status,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\n" + "=" * 118)
print("STAGE 5A STEP 3C DECISION")
print("=" * 118)

if failed_checks:

    print(
        "REVIEW_CONFLICT_FIELD_SEMANTICS"
    )

    print("\nChecks requiring review:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Conflict semantics are not yet sufficiently "
        "resolved to define the future-conflict outcome."
    )

print(
    "PASS_FROZEN_AGGREGATE_CONFLICT_FLAG_SEMANTICS_CONFIRMED"
)

print()
print(
    "Interpretation:"
)
print(
    "The classification group represents the selected aggregate "
    "clinical category."
)
print(
    "The aggregate conflict flag additionally captures records whose "
    "review status explicitly reports conflicting submissions."
)
print(
    "Therefore, the later new-conflict and conflict-resolution components "
    "should use the frozen aggregate conflict flag rather than relying "
    "only on classification_group='Conflicting'."
)

print()
print(
    "Outcome conflict rule frozen:       NO — diagnostic confirmation only"
)
print(
    "Future-instability outcome created: NO"
)
print(
    "GES scores loaded:                  NO"
)
print(
    "Temporal performance examined:      NO"
)
print(
    "Files written or modified:          NO"
)

STAGE 5A STEP 3C — CONFLICT GROUP/FIELD SEMANTIC AUDIT

BASIC ACCOUNTING
----------------------------------------------------------------------------------------------------------------------
Stage 3H rows loaded:                         71,659
Outcome-eligible rows:                        70,583
Missing eligibility values:                   0
Invalid eligibility values:                   0
T0 group/flag differences:                    4
T1 group/flag differences:                    27
T0 stored flag vs semantic-rule differences:  0
T1 stored flag vs semantic-rule differences:  0

DIFFERENCE TYPE DISTRIBUTION
----------------------------------------------------------------------------------------------------------------------
timepoint                difference_type  record_count
       T1 GROUP_NONCONFLICTING_FLAG_TRUE            27
       T0 GROUP_NONCONFLICTING_FLAG_TRUE             4

REVIEW STATUS AMONG GROUP/FLAG DIFFERENCES
-------------------------------------------------------

In [10]:
# ==================================================================================================
# STAGE 5A — STEP 3D
# AUDIT PRIOR-CONFLICT RESOLUTION PATTERNS
#
# Purpose:
#   1. Isolate records that were conflicted at T0.
#   2. Distinguish persistent conflict from conflict resolved by T1.
#   3. Determine the T0 and T1 classification groups for resolved records.
#   4. Separate:
#        a. aggregate Conflicting -> BLB/VUS/PLP resolution;
#        b. flag-only conflict -> same clinical group;
#        c. flag-only conflict -> different clinical group;
#        d. nonclinical or non-germline resolution states.
#   5. Inspect whether the final T1 clinical group had supporting SCV
#      classifications already present at T0.
#
# Scientific boundary:
#   - Counts are diagnostic candidates only.
#   - No future-instability outcome is assigned.
#   - No outcome rule is frozen.
#   - No GES score or Stage 4 model artifact is loaded.
#   - No temporal performance is examined.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
from collections import Counter
import json

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Required and optional fields
# --------------------------------------------------------------------------------------------------

schema_columns = set(
    pq.ParquetFile(
        STAGE3H_LINKAGE_PATH
    ).schema.names
)

required_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "temporal_outcome_eligible",

    "t0_target_genes_json",
    "t1_target_genes_json",

    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_conflict_flag",

    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_conflict_flag",
    "t1_aggregate_classification_axis",
]

optional_columns = [
    "t0_scv_group_counts_json",
    "t1_scv_group_counts_json",
    "t0_scv_classification_counts_json",
    "t1_scv_classification_counts_json",
]

missing_required = sorted(
    set(required_columns).difference(
        schema_columns
    )
)

if missing_required:
    raise RuntimeError(
        "Required fields are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_required
        )
    )

available_optional = [
    column
    for column in optional_columns
    if column in schema_columns
]

df = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=(
        required_columns
        + available_optional
    ),
)


# --------------------------------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------------------------------

def is_missing(value):
    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except Exception:
        pass

    return False


def normalize_boolean(value):
    if is_missing(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True

        if value == 0:
            return False

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().casefold(),
        "INVALID",
    )


def normalize_text(value):
    if is_missing(value):
        return pd.NA

    text = str(value).strip()

    if text.casefold() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return pd.NA

    return text


def canonical_group(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "benign/likely benign": "BLB",
        "vus": "VUS",
        "pathogenic/likely pathogenic": "PLP",
        "conflicting": "CONFLICTING",
        "other": "OTHER",
        "missing": "MISSING",
        "mixed": "MIXED",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def canonical_axis(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "germlineclassification": "GERMLINE",
        "noclassification": "NO_CLASSIFICATION",
        "oncogenicityclassification": "ONCOGENICITY",
        "somaticclinicalimpact": "SOMATIC_CLINICAL_IMPACT",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def parse_single_gene(value):
    if is_missing(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    if not genes:
        return "<EMPTY>"

    return "|".join(genes)


def parse_json_dict(value):
    if is_missing(value):
        return None

    try:
        parsed = (
            value
            if isinstance(value, dict)
            else json.loads(str(value))
        )
    except Exception:
        return None

    return parsed if isinstance(parsed, dict) else None


def canonical_scv_group(value):
    text = str(value).strip().casefold()

    mapping = {
        "benign/likely benign": "BLB",
        "benign": "BLB",
        "likely benign": "BLB",

        "uncertain significance": "VUS",
        "vus": "VUS",

        "pathogenic/likely pathogenic": "PLP",
        "pathogenic": "PLP",
        "likely pathogenic": "PLP",

        "conflicting": "CONFLICTING",
        "other": "OTHER",
        "missing": "MISSING",
    }

    return mapping.get(
        text,
        str(value).strip(),
    )


def summarize_scv_groups(value):
    parsed = parse_json_dict(value)

    if parsed is None:
        return {
            "status": "MISSING_OR_INVALID",
            "groups": {},
            "clinical_groups_present": set(),
        }

    normalized_counts = Counter()

    for raw_group, raw_count in parsed.items():

        canonical = canonical_scv_group(
            raw_group
        )

        try:
            count = int(raw_count)
        except Exception:
            count = 0

        normalized_counts[canonical] += count

    clinical_groups_present = {
        group
        for group in {"BLB", "VUS", "PLP"}
        if normalized_counts.get(group, 0) > 0
    }

    return {
        "status": "PASS",
        "groups": dict(normalized_counts),
        "clinical_groups_present": clinical_groups_present,
    }


def print_counts(title, series):
    print(f"\n{title}")
    print("-" * 116)

    print(
        series.astype("string")
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 4. Restrict to accepted outcome-eligible links
# --------------------------------------------------------------------------------------------------

eligibility = (
    df["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

missing_eligibility = int(
    eligibility.isna().sum()
)

invalid_eligibility = int(
    eligibility.eq("INVALID").sum()
)

eligible = (
    df.loc[
        eligibility.eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_TOTAL_ROWS = 71_659
EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_T0_CONFLICT_POSITIVE = 1_481
EXPECTED_RESOLVED_CONFLICTS = 298
EXPECTED_PERSISTENT_CONFLICTS = 1_183


# --------------------------------------------------------------------------------------------------
# 5. Normalize evidence fields
# --------------------------------------------------------------------------------------------------

eligible["_t0_gene"] = (
    eligible["t0_target_genes_json"]
    .apply(parse_single_gene)
)

eligible["_t1_gene"] = (
    eligible["t1_target_genes_json"]
    .apply(parse_single_gene)
)

eligible["_t0_group"] = (
    eligible[
        "t0_aggregate_classification_group"
    ]
    .apply(canonical_group)
)

eligible["_t1_group"] = (
    eligible[
        "t1_aggregate_classification_group"
    ]
    .apply(canonical_group)
)

eligible["_t1_axis"] = (
    eligible[
        "t1_aggregate_classification_axis"
    ]
    .apply(canonical_axis)
)

eligible["_t0_conflict"] = (
    eligible[
        "t0_aggregate_conflict_flag"
    ]
    .apply(normalize_boolean)
)

eligible["_t1_conflict"] = (
    eligible[
        "t1_aggregate_conflict_flag"
    ]
    .apply(normalize_boolean)
)

eligible["_t0_classification"] = (
    eligible[
        "t0_aggregate_classification"
    ]
    .apply(normalize_text)
)

eligible["_t1_classification"] = (
    eligible[
        "t1_aggregate_classification"
    ]
    .apply(normalize_text)
)


# --------------------------------------------------------------------------------------------------
# 6. Identify T0-conflicted records and resolution states
# --------------------------------------------------------------------------------------------------

prior_conflict_mask = (
    eligible["_t0_conflict"].eq(True)
)

persistent_conflict_mask = (
    prior_conflict_mask
    & eligible["_t1_conflict"].eq(True)
)

resolved_conflict_mask = (
    prior_conflict_mask
    & eligible["_t1_conflict"].eq(False)
)

prior_conflict = (
    eligible.loc[
        prior_conflict_mask
    ]
    .copy()
)

resolved = (
    eligible.loc[
        resolved_conflict_mask
    ]
    .copy()
    .reset_index(drop=True)
)

persistent = (
    eligible.loc[
        persistent_conflict_mask
    ]
    .copy()
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 7. Basic accounting
# --------------------------------------------------------------------------------------------------

print("=" * 116)
print("STAGE 5A STEP 3D — PRIOR-CONFLICT RESOLUTION AUDIT")
print("=" * 116)

print("\nBASIC ACCOUNTING")
print("-" * 116)
print(
    f"Complete Stage 3H rows:                  "
    f"{len(df):,}"
)
print(
    f"Outcome-eligible linked rows:            "
    f"{len(eligible):,}"
)
print(
    f"Missing eligibility values:              "
    f"{missing_eligibility:,}"
)
print(
    f"Invalid eligibility values:              "
    f"{invalid_eligibility:,}"
)
print(
    f"T0 conflict-positive records:            "
    f"{len(prior_conflict):,}"
)
print(
    f"Conflict persisted at T1:                "
    f"{len(persistent):,}"
)
print(
    f"Conflict resolved by T1:                 "
    f"{len(resolved):,}"
)


# --------------------------------------------------------------------------------------------------
# 8. Profile resolved records
# --------------------------------------------------------------------------------------------------

print_counts(
    "RESOLVED RECORDS — T0 CLASSIFICATION GROUP",
    resolved["_t0_group"],
)

print_counts(
    "RESOLVED RECORDS — T1 CLASSIFICATION GROUP",
    resolved["_t1_group"],
)

print_counts(
    "RESOLVED RECORDS — T1 CLASSIFICATION AXIS",
    resolved["_t1_axis"],
)

print_counts(
    "RESOLVED RECORDS — TARGET GENE",
    resolved["_t0_gene"],
)


# --------------------------------------------------------------------------------------------------
# 9. Create explicit resolution-pattern categories
# --------------------------------------------------------------------------------------------------

CLINICAL_TRIAD = {
    "BLB",
    "VUS",
    "PLP",
}


def assign_resolution_pattern(row):
    t0_group = row["_t0_group"]
    t1_group = row["_t1_group"]
    t1_axis = row["_t1_axis"]

    if t1_axis != "GERMLINE":
        return (
            f"NON_GERMLINE_OR_UNCLASSIFIED:"
            f"{t1_axis}"
        )

    if (
        t0_group == "CONFLICTING"
        and t1_group in CLINICAL_TRIAD
    ):
        return (
            f"AGGREGATE_CONFLICTING_TO_{t1_group}"
        )

    if (
        t0_group in CLINICAL_TRIAD
        and t1_group == t0_group
    ):
        return (
            f"FLAG_ONLY_CONFLICT_RESOLVED_"
            f"SAME_GROUP_{t0_group}"
        )

    if (
        t0_group in CLINICAL_TRIAD
        and t1_group in CLINICAL_TRIAD
        and t1_group != t0_group
    ):
        return (
            f"FLAG_ONLY_CONFLICT_RESOLVED_"
            f"DIFFERENT_GROUP_{t0_group}_TO_{t1_group}"
        )

    return (
        f"OTHER_RESOLUTION_"
        f"{t0_group}_TO_{t1_group}"
    )


resolved["_resolution_pattern"] = (
    resolved.apply(
        assign_resolution_pattern,
        axis=1,
    )
)

print_counts(
    "MUTUALLY EXCLUSIVE PRIOR-CONFLICT RESOLUTION PATTERNS",
    resolved["_resolution_pattern"],
)


# --------------------------------------------------------------------------------------------------
# 10. Diagnostic material-change categories
# --------------------------------------------------------------------------------------------------

aggregate_conflicting_to_triad_mask = (
    resolved["_t0_group"].eq(
        "CONFLICTING"
    )
    & resolved["_t1_group"].isin(
        CLINICAL_TRIAD
    )
    & resolved["_t1_axis"].eq(
        "GERMLINE"
    )
)

flag_only_same_group_mask = (
    resolved["_t0_group"].isin(
        CLINICAL_TRIAD
    )
    & resolved["_t1_group"].eq(
        resolved["_t0_group"]
    )
    & resolved["_t1_axis"].eq(
        "GERMLINE"
    )
)

flag_only_different_group_mask = (
    resolved["_t0_group"].isin(
        CLINICAL_TRIAD
    )
    & resolved["_t1_group"].isin(
        CLINICAL_TRIAD
    )
    & resolved["_t1_group"].ne(
        resolved["_t0_group"]
    )
    & resolved["_t1_axis"].eq(
        "GERMLINE"
    )
)

nonclinical_resolution_mask = ~(
    aggregate_conflicting_to_triad_mask
    | flag_only_same_group_mask
    | flag_only_different_group_mask
)

diagnostic_material_resolution_mask = (
    aggregate_conflicting_to_triad_mask
    | flag_only_different_group_mask
)

print("\nDIAGNOSTIC MATERIAL-CHANGE ACCOUNTING")
print("-" * 116)
print(
    f"Aggregate Conflicting -> clinical triad:     "
    f"{aggregate_conflicting_to_triad_mask.sum():,}"
)
print(
    f"Flag-only conflict -> same clinical group:   "
    f"{flag_only_same_group_mask.sum():,}"
)
print(
    f"Flag-only conflict -> different group:       "
    f"{flag_only_different_group_mask.sum():,}"
)
print(
    f"Other/nonclinical resolution states:         "
    f"{nonclinical_resolution_mask.sum():,}"
)
print(
    f"Diagnostic material-resolution candidates:   "
    f"{diagnostic_material_resolution_mask.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 11. Inspect T0 SCV group support when available
# --------------------------------------------------------------------------------------------------

if "t0_scv_group_counts_json" in resolved.columns:

    scv_summaries = (
        resolved[
            "t0_scv_group_counts_json"
        ]
        .apply(summarize_scv_groups)
    )

    resolved["_t0_scv_json_status"] = (
        scv_summaries.apply(
            lambda item:
                item["status"]
        )
    )

    resolved["_t0_scv_groups_present"] = (
        scv_summaries.apply(
            lambda item:
                sorted(
                    item[
                        "clinical_groups_present"
                    ]
                )
        )
    )

    resolved["_t1_group_present_at_t0_scv"] = [
        (
            t1_group in set(groups)
            if isinstance(groups, list)
            else False
        )
        for t1_group, groups in zip(
            resolved["_t1_group"],
            resolved[
                "_t0_scv_groups_present"
            ],
        )
    ]

    resolved["_t0_number_clinical_scv_groups"] = (
        resolved[
            "_t0_scv_groups_present"
        ]
        .apply(
            lambda groups:
                len(groups)
                if isinstance(groups, list)
                else pd.NA
        )
    )

    print_counts(
        "T0 SCV-GROUP JSON STATUS FOR RESOLVED RECORDS",
        resolved["_t0_scv_json_status"],
    )

    print_counts(
        "NUMBER OF T0 CLINICAL SCV GROUPS PRESENT",
        resolved[
            "_t0_number_clinical_scv_groups"
        ],
    )

    print_counts(
        "WAS THE FINAL T1 GROUP ALREADY REPRESENTED AMONG T0 SCVS?",
        resolved[
            "_t1_group_present_at_t0_scv"
        ],
    )

else:

    print("\nT0 SCV-GROUP SUPPORT AUDIT")
    print("-" * 116)
    print(
        "t0_scv_group_counts_json is not available "
        "in the frozen Stage 3H linkage."
    )


# --------------------------------------------------------------------------------------------------
# 12. Cross-tabulations
# --------------------------------------------------------------------------------------------------

resolution_cross_tab = pd.crosstab(
    resolved["_t0_group"],
    resolved["_t1_group"],
    margins=True,
    dropna=False,
)

print("\nRESOLVED PRIOR-CONFLICT T0 × T1 GROUP TABLE")
print("-" * 116)
print(
    resolution_cross_tab.to_string()
)

resolution_by_gene = pd.crosstab(
    resolved["_t0_gene"],
    resolved["_resolution_pattern"],
    dropna=False,
)

print("\nRESOLUTION PATTERN BY TARGET GENE")
print("-" * 116)
print(
    resolution_by_gene.to_string()
)


# --------------------------------------------------------------------------------------------------
# 13. Display all unusual/nonstandard resolution records
# --------------------------------------------------------------------------------------------------

unusual_mask = (
    flag_only_same_group_mask
    | flag_only_different_group_mask
    | nonclinical_resolution_mask
)

display_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "_t0_gene",
    "_t0_classification",
    "_t0_group",
    "t0_aggregate_review_status",
    "_t1_classification",
    "_t1_group",
    "t1_aggregate_review_status",
    "_t1_axis",
    "_resolution_pattern",
]

if (
    "t0_scv_group_counts_json"
    in resolved.columns
):
    display_columns.extend(
        [
            "t0_scv_group_counts_json",
            "_t0_scv_groups_present",
            "_t1_group_present_at_t0_scv",
        ]
    )

print("\nFLAG-ONLY OR NONSTANDARD RESOLUTION RECORDS")
print("-" * 116)

if unusual_mask.any():

    print(
        resolved.loc[
            unusual_mask,
            display_columns,
        ]
        .to_string(index=False)
    )

else:
    print("None")


# --------------------------------------------------------------------------------------------------
# 14. Structural checks
# --------------------------------------------------------------------------------------------------

gene_mismatch_count = int(
    eligible["_t0_gene"]
    .ne(
        eligible["_t1_gene"]
    )
    .sum()
)

invalid_conflict_values = int(
    eligible["_t0_conflict"]
    .eq("INVALID")
    .sum()
    +
    eligible["_t1_conflict"]
    .eq("INVALID")
    .sum()
)

missing_conflict_values = int(
    eligible["_t0_conflict"]
    .isna()
    .sum()
    +
    eligible["_t1_conflict"]
    .isna()
    .sum()
)

resolution_accounting_total = int(
    aggregate_conflicting_to_triad_mask.sum()
    + flag_only_same_group_mask.sum()
    + flag_only_different_group_mask.sum()
    + nonclinical_resolution_mask.sum()
)

critical_failures = {
    "unexpected Stage 3H row count":
        len(df) != EXPECTED_TOTAL_ROWS,

    "unexpected eligible-row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "missing or invalid eligibility":
        (
            missing_eligibility > 0
            or invalid_eligibility > 0
        ),

    "target-gene mismatch":
        gene_mismatch_count > 0,

    "missing or invalid conflict values":
        (
            missing_conflict_values > 0
            or invalid_conflict_values > 0
        ),

    "unexpected T0 conflict-positive count":
        len(prior_conflict)
        != EXPECTED_T0_CONFLICT_POSITIVE,

    "unexpected persistent-conflict count":
        len(persistent)
        != EXPECTED_PERSISTENT_CONFLICTS,

    "unexpected resolved-conflict count":
        len(resolved)
        != EXPECTED_RESOLVED_CONFLICTS,

    "resolution categories do not reconcile":
        resolution_accounting_total
        != len(resolved),
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]


# --------------------------------------------------------------------------------------------------
# 15. Diagnostic decision
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 116)
print("STAGE 5A STEP 3D DECISION")
print("=" * 116)

if failed_checks:

    print(
        "FAIL_STAGE5A_PRIOR_CONFLICT_RESOLUTION_AUDIT"
    )

    print("\nFailed checks:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Prior-conflict resolution patterns are not "
        "sufficiently reconciled to proceed."
    )

print(
    "PASS_STAGE5A_PRIOR_CONFLICT_RESOLUTION_PATTERNS_AUDITED"
)

print()
print(
    "The material-resolution counts above remain "
    "diagnostic candidates only."
)
print(
    "No record has yet been assigned a stable or "
    "unstable temporal outcome."
)
print()
print(
    "Prior-conflict outcome rule frozen: NO"
)
print(
    "Future-instability outcome created: NO"
)
print(
    "GES scores loaded:                  NO"
)
print(
    "Temporal performance examined:      NO"
)
print(
    "Files written or modified:          NO"
)

STAGE 5A STEP 3D — PRIOR-CONFLICT RESOLUTION AUDIT

BASIC ACCOUNTING
--------------------------------------------------------------------------------------------------------------------
Complete Stage 3H rows:                  71,659
Outcome-eligible linked rows:            70,583
Missing eligibility values:              0
Invalid eligibility values:              0
T0 conflict-positive records:            1,481
Conflict persisted at T1:                1,183
Conflict resolved by T1:                 298

RESOLVED RECORDS — T0 CLASSIFICATION GROUP
--------------------------------------------------------------------------------------------------------------------
_t0_group
CONFLICTING    297
BLB              1

RESOLVED RECORDS — T1 CLASSIFICATION GROUP
--------------------------------------------------------------------------------------------------------------------
_t1_group
BLB    207
VUS     53
PLP     38

RESOLVED RECORDS — T1 CLASSIFICATION AXIS
-------------------------------------

In [11]:
# ==================================================================================================
# STAGE 5A — STEP 3E
# AUDIT REVIEW-STATUS, REVIEW-STAR, AND EXPERT-PANEL DRIFT
#
# Purpose:
#   1. Inventory T0-to-T1 review-star transitions.
#   2. Inventory exact review-status transitions.
#   3. Identify new, persistent, lost, and absent expert-panel review.
#   4. Quantify the same-star cohort for later prespecified analysis.
#   5. Keep review drift separate from the primary future-instability outcome.
#
# Scientific boundary:
#   - Review-star change alone is NOT treated as primary instability.
#   - No future-instability outcome is assigned.
#   - No outcome policy is frozen.
#   - No GES score or Stage 4 model artifact is loaded.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
import json

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 3H linkage artifact
# --------------------------------------------------------------------------------------------------

STAGE3H_LINKAGE_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage3_crosswalk/"
    "stage3h_final_t0_t1_linkage_v1.parquet"
)

if not STAGE3H_LINKAGE_PATH.exists():
    raise FileNotFoundError(
        "Frozen Stage 3H linkage artifact was not found:\n"
        f"{STAGE3H_LINKAGE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load only fields required for the review-drift audit
# --------------------------------------------------------------------------------------------------

columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "temporal_outcome_eligible",
    "linkage_decision_category",

    "t0_target_genes_json",
    "t1_target_genes_json",

    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",

    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_aggregate_classification_axis",
]

df = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=columns,
)


# --------------------------------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------------------------------

def is_missing(value):
    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except Exception:
        pass

    return False


def normalize_boolean(value):
    if is_missing(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True

        if value == 0:
            return False

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().casefold(),
        "INVALID",
    )


def normalize_text(value):
    if is_missing(value):
        return pd.NA

    text = str(value).strip()

    if text.casefold() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return pd.NA

    return text


def canonical_axis(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "germlineclassification": "GERMLINE",
        "noclassification": "NO_CLASSIFICATION",
        "oncogenicityclassification": "ONCOGENICITY",
        "somaticclinicalimpact": "SOMATIC_CLINICAL_IMPACT",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def parse_single_gene(value):
    if is_missing(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    if not genes:
        return "<EMPTY>"

    return "|".join(genes)


def print_counts(title, series):
    print(f"\n{title}")
    print("-" * 116)

    print(
        series.astype("string")
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )


# --------------------------------------------------------------------------------------------------
# 4. Restrict to frozen outcome-eligible links
# --------------------------------------------------------------------------------------------------

eligibility = (
    df["temporal_outcome_eligible"]
    .apply(normalize_boolean)
)

missing_eligibility = int(
    eligibility.isna().sum()
)

invalid_eligibility = int(
    eligibility.eq("INVALID").sum()
)

eligible = (
    df.loc[
        eligibility.eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

EXPECTED_TOTAL_ROWS = 71_659
EXPECTED_ELIGIBLE_ROWS = 70_583
EXPECTED_NO_CLASSIFICATION_ROWS = 3_092
EXPECTED_NON_GERMLINE_ROWS = 2


# --------------------------------------------------------------------------------------------------
# 5. Normalize genes, axes, review statuses, and star values
# --------------------------------------------------------------------------------------------------

eligible["_t0_gene"] = (
    eligible["t0_target_genes_json"]
    .apply(parse_single_gene)
)

eligible["_t1_gene"] = (
    eligible["t1_target_genes_json"]
    .apply(parse_single_gene)
)

eligible["_t1_axis"] = (
    eligible[
        "t1_aggregate_classification_axis"
    ]
    .apply(canonical_axis)
)

eligible["_t0_review_status"] = (
    eligible[
        "t0_aggregate_review_status"
    ]
    .apply(normalize_text)
)

eligible["_t1_review_status"] = (
    eligible[
        "t1_aggregate_review_status"
    ]
    .apply(normalize_text)
)

eligible["_t0_star_numeric"] = pd.to_numeric(
    eligible[
        "t0_aggregate_review_stars"
    ],
    errors="coerce",
)

eligible["_t1_star_numeric"] = pd.to_numeric(
    eligible[
        "t1_aggregate_review_stars"
    ],
    errors="coerce",
)


# --------------------------------------------------------------------------------------------------
# 6. Validate review-star structure
# --------------------------------------------------------------------------------------------------

t0_original_nonmissing = (
    eligible[
        "t0_aggregate_review_stars"
    ]
    .apply(normalize_text)
    .notna()
)

t1_original_nonmissing = (
    eligible[
        "t1_aggregate_review_stars"
    ]
    .apply(normalize_text)
    .notna()
)

t0_malformed_stars = int(
    (
        t0_original_nonmissing
        & eligible[
            "_t0_star_numeric"
        ].isna()
    ).sum()
)

t1_malformed_stars = int(
    (
        t1_original_nonmissing
        & eligible[
            "_t1_star_numeric"
        ].isna()
    ).sum()
)

t0_invalid_range = int(
    (
        eligible[
            "_t0_star_numeric"
        ].notna()
        & (
            (
                eligible[
                    "_t0_star_numeric"
                ] < 0
            )
            |
            (
                eligible[
                    "_t0_star_numeric"
                ] > 4
            )
            |
            (
                eligible[
                    "_t0_star_numeric"
                ] % 1 != 0
            )
        )
    ).sum()
)

t1_invalid_range = int(
    (
        eligible[
            "_t1_star_numeric"
        ].notna()
        & (
            (
                eligible[
                    "_t1_star_numeric"
                ] < 0
            )
            |
            (
                eligible[
                    "_t1_star_numeric"
                ] > 4
            )
            |
            (
                eligible[
                    "_t1_star_numeric"
                ] % 1 != 0
            )
        )
    ).sum()
)

eligible["_t0_star"] = (
    eligible[
        "_t0_star_numeric"
    ]
    .astype("Int64")
)

eligible["_t1_star"] = (
    eligible[
        "_t1_star_numeric"
    ]
    .astype("Int64")
)


# --------------------------------------------------------------------------------------------------
# 7. Basic accounting
# --------------------------------------------------------------------------------------------------

print("=" * 116)
print("STAGE 5A STEP 3E — REVIEW-STATUS, STAR, AND EXPERT-PANEL DRIFT AUDIT")
print("=" * 116)

print("\nBASIC ACCOUNTING")
print("-" * 116)
print(
    f"Complete Stage 3H rows:                  "
    f"{len(df):,}"
)
print(
    f"Outcome-eligible linked rows:            "
    f"{len(eligible):,}"
)
print(
    f"Missing eligibility values:              "
    f"{missing_eligibility:,}"
)
print(
    f"Invalid eligibility values:              "
    f"{invalid_eligibility:,}"
)

print("\nREVIEW-STAR STRUCTURAL CHECKS")
print("-" * 116)
print(
    f"T0 missing review-star values:           "
    f"{eligible['_t0_star'].isna().sum():,}"
)
print(
    f"T0 malformed review-star values:         "
    f"{t0_malformed_stars:,}"
)
print(
    f"T0 review stars outside integer 0-4:     "
    f"{t0_invalid_range:,}"
)
print(
    f"T1 missing review-star values:           "
    f"{eligible['_t1_star'].isna().sum():,}"
)
print(
    f"T1 malformed review-star values:         "
    f"{t1_malformed_stars:,}"
)
print(
    f"T1 review stars outside integer 0-4:     "
    f"{t1_invalid_range:,}"
)


# --------------------------------------------------------------------------------------------------
# 8. Review-star distributions and transition matrix
# --------------------------------------------------------------------------------------------------

print_counts(
    "T0 REVIEW-STAR DISTRIBUTION",
    eligible["_t0_star"],
)

print_counts(
    "T1 REVIEW-STAR DISTRIBUTION",
    eligible["_t1_star"],
)

star_transition_table = pd.crosstab(
    eligible["_t0_star"],
    eligible["_t1_star"],
    margins=True,
    dropna=False,
)

print("\nT0 × T1 REVIEW-STAR TRANSITION TABLE")
print("-" * 116)
print(
    star_transition_table.to_string()
)


# --------------------------------------------------------------------------------------------------
# 9. Star-change direction and magnitude
# --------------------------------------------------------------------------------------------------

eligible["_review_star_delta"] = (
    eligible["_t1_star"]
    - eligible["_t0_star"]
)

eligible["_review_star_transition"] = pd.NA

eligible.loc[
    eligible[
        "_review_star_delta"
    ].eq(0),
    "_review_star_transition",
] = "UNCHANGED"

eligible.loc[
    eligible[
        "_review_star_delta"
    ].gt(0),
    "_review_star_transition",
] = "INCREASED"

eligible.loc[
    eligible[
        "_review_star_delta"
    ].lt(0),
    "_review_star_transition",
] = "DECREASED"

print_counts(
    "REVIEW-STAR CHANGE DIRECTION",
    eligible[
        "_review_star_transition"
    ],
)

print_counts(
    "REVIEW-STAR NUMERIC DELTA",
    eligible[
        "_review_star_delta"
    ],
)

same_star_mask = (
    eligible[
        "_review_star_delta"
    ].eq(0)
)

increased_star_mask = (
    eligible[
        "_review_star_delta"
    ].gt(0)
)

decreased_star_mask = (
    eligible[
        "_review_star_delta"
    ].lt(0)
)

print("\nSAME-STAR ANALYSIS COHORT SIZE")
print("-" * 116)
print(
    f"Same-star eligible records:               "
    f"{same_star_mask.sum():,}"
)
print(
    f"Review-star increases:                    "
    f"{increased_star_mask.sum():,}"
)
print(
    f"Review-star decreases:                    "
    f"{decreased_star_mask.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 10. Exact review-status inventories and transition table
# --------------------------------------------------------------------------------------------------

print_counts(
    "T0 EXACT REVIEW STATUS",
    eligible[
        "_t0_review_status"
    ],
)

print_counts(
    "T1 EXACT REVIEW STATUS",
    eligible[
        "_t1_review_status"
    ],
)

status_transition_table = pd.crosstab(
    eligible[
        "_t0_review_status"
    ],
    eligible[
        "_t1_review_status"
    ],
    margins=True,
    dropna=False,
)

print("\nT0 × T1 EXACT REVIEW-STATUS TRANSITION TABLE")
print("-" * 116)
print(
    status_transition_table.to_string()
)


# --------------------------------------------------------------------------------------------------
# 11. Review-status to star consistency
# --------------------------------------------------------------------------------------------------

t0_status_star_table = pd.crosstab(
    eligible[
        "_t0_review_status"
    ],
    eligible[
        "_t0_star"
    ],
    margins=True,
    dropna=False,
)

t1_status_star_table = pd.crosstab(
    eligible[
        "_t1_review_status"
    ],
    eligible[
        "_t1_star"
    ],
    margins=True,
    dropna=False,
)

print("\nT0 REVIEW STATUS × STAR")
print("-" * 116)
print(
    t0_status_star_table.to_string()
)

print("\nT1 REVIEW STATUS × STAR")
print("-" * 116)
print(
    t1_status_star_table.to_string()
)


# --------------------------------------------------------------------------------------------------
# 12. Expert-panel transitions
# --------------------------------------------------------------------------------------------------

EXPERT_PANEL_STATUS = (
    "reviewed by expert panel"
)

eligible["_t0_expert_panel"] = (
    eligible[
        "_t0_review_status"
    ]
    .astype("string")
    .str.casefold()
    .eq(EXPERT_PANEL_STATUS)
)

eligible["_t1_expert_panel"] = (
    eligible[
        "_t1_review_status"
    ]
    .astype("string")
    .str.casefold()
    .eq(EXPERT_PANEL_STATUS)
)


def assign_expert_panel_transition(row):
    t0_expert = bool(
        row["_t0_expert_panel"]
    )

    t1_expert = bool(
        row["_t1_expert_panel"]
    )

    if not t0_expert and not t1_expert:
        return "NO_EXPERT_PANEL_AT_EITHER_TIME"

    if not t0_expert and t1_expert:
        return "NEW_EXPERT_PANEL_AT_T1"

    if t0_expert and t1_expert:
        return "EXPERT_PANEL_PERSISTED"

    if t0_expert and not t1_expert:
        return "EXPERT_PANEL_NOT_PRESENT_AT_T1"

    return "UNEXPECTED"


eligible["_expert_panel_transition"] = (
    eligible.apply(
        assign_expert_panel_transition,
        axis=1,
    )
)

print_counts(
    "EXPERT-PANEL REVIEW TRANSITIONS",
    eligible[
        "_expert_panel_transition"
    ],
)

expert_panel_by_gene = pd.crosstab(
    eligible["_t0_gene"],
    eligible[
        "_expert_panel_transition"
    ],
    dropna=False,
)

print("\nEXPERT-PANEL TRANSITION BY TARGET GENE")
print("-" * 116)
print(
    expert_panel_by_gene.to_string()
)


# --------------------------------------------------------------------------------------------------
# 13. Verify expert-panel/star consistency
# --------------------------------------------------------------------------------------------------

t0_expert_not_three_star = int(
    (
        eligible[
            "_t0_expert_panel"
        ]
        & eligible[
            "_t0_star"
        ].ne(3)
    ).sum()
)

t1_expert_not_three_star = int(
    (
        eligible[
            "_t1_expert_panel"
        ]
        & eligible[
            "_t1_star"
        ].ne(3)
    ).sum()
)

t0_three_star_not_expert = int(
    (
        eligible[
            "_t0_star"
        ].eq(3)
        & ~eligible[
            "_t0_expert_panel"
        ]
    ).sum()
)

t1_three_star_not_expert = int(
    (
        eligible[
            "_t1_star"
        ].eq(3)
        & ~eligible[
            "_t1_expert_panel"
        ]
    ).sum()
)

print("\nEXPERT-PANEL / THREE-STAR CONSISTENCY")
print("-" * 116)
print(
    f"T0 expert-panel records not rated 3 stars: "
    f"{t0_expert_not_three_star:,}"
)
print(
    f"T1 expert-panel records not rated 3 stars: "
    f"{t1_expert_not_three_star:,}"
)
print(
    f"T0 3-star records not expert-panel reviewed: "
    f"{t0_three_star_not_expert:,}"
)
print(
    f"T1 3-star records not expert-panel reviewed: "
    f"{t1_three_star_not_expert:,}"
)


# --------------------------------------------------------------------------------------------------
# 14. Axis-specific review drift
# --------------------------------------------------------------------------------------------------

no_classification_mask = (
    eligible["_t1_axis"]
    .eq("NO_CLASSIFICATION")
)

non_germline_mask = (
    eligible["_t1_axis"]
    .isin(
        {
            "ONCOGENICITY",
            "SOMATIC_CLINICAL_IMPACT",
        }
    )
)

germline_mask = (
    eligible["_t1_axis"]
    .eq("GERMLINE")
)

print("\nT1 AXIS ACCOUNTING")
print("-" * 116)
print(
    f"Germline-axis records:                    "
    f"{germline_mask.sum():,}"
)
print(
    f"NoClassification records:                "
    f"{no_classification_mask.sum():,}"
)
print(
    f"Non-germline selected-axis records:       "
    f"{non_germline_mask.sum():,}"
)

print_counts(
    "NoClassification — T1 REVIEW STATUS",
    eligible.loc[
        no_classification_mask,
        "_t1_review_status",
    ],
)

print_counts(
    "NoClassification — T1 REVIEW STARS",
    eligible.loc[
        no_classification_mask,
        "_t1_star",
    ],
)

print("\nNON-GERMLINE SELECTED-AXIS RECORDS")
print("-" * 116)

non_germline_display_columns = [
    "t0_rcv_accession",
    "t1_rcv_accession",
    "_t0_gene",
    "_t1_axis",
    "t0_aggregate_classification",
    "_t0_review_status",
    "_t0_star",
    "t1_aggregate_classification",
    "_t1_review_status",
    "_t1_star",
]

if non_germline_mask.any():
    print(
        eligible.loc[
            non_germline_mask,
            non_germline_display_columns,
        ]
        .to_string(index=False)
    )
else:
    print("None")


# --------------------------------------------------------------------------------------------------
# 15. Star-transition accounting by target gene
# --------------------------------------------------------------------------------------------------

star_direction_by_gene = pd.crosstab(
    eligible["_t0_gene"],
    eligible[
        "_review_star_transition"
    ],
    dropna=False,
)

print("\nREVIEW-STAR CHANGE DIRECTION BY TARGET GENE")
print("-" * 116)
print(
    star_direction_by_gene.to_string()
)

same_star_by_baseline_star = pd.crosstab(
    eligible.loc[
        same_star_mask,
        "_t0_gene",
    ],
    eligible.loc[
        same_star_mask,
        "_t0_star",
    ],
    margins=True,
    dropna=False,
)

print("\nSAME-STAR RECORDS BY GENE AND BASELINE STAR")
print("-" * 116)
print(
    same_star_by_baseline_star.to_string()
)


# --------------------------------------------------------------------------------------------------
# 16. Structural checks and final diagnostic decision
# --------------------------------------------------------------------------------------------------

gene_mismatch_count = int(
    eligible["_t0_gene"]
    .ne(
        eligible["_t1_gene"]
    )
    .sum()
)

unexpected_axis_count = int(
    eligible["_t1_axis"]
    .astype("string")
    .str.startswith(
        "UNEXPECTED:",
        na=False,
    )
    .sum()
)

star_transition_total = int(
    same_star_mask.sum()
    + increased_star_mask.sum()
    + decreased_star_mask.sum()
)

critical_failures = {
    "unexpected Stage 3H row count":
        len(df) != EXPECTED_TOTAL_ROWS,

    "unexpected eligible-row count":
        len(eligible) != EXPECTED_ELIGIBLE_ROWS,

    "missing or invalid eligibility":
        (
            missing_eligibility > 0
            or invalid_eligibility > 0
        ),

    "target-gene mismatch":
        gene_mismatch_count > 0,

    "missing T0 review status":
        eligible[
            "_t0_review_status"
        ].isna().any(),

    "missing T1 review status":
        eligible[
            "_t1_review_status"
        ].isna().any(),

    "missing T0 review stars":
        eligible[
            "_t0_star"
        ].isna().any(),

    "missing T1 review stars":
        eligible[
            "_t1_star"
        ].isna().any(),

    "malformed or out-of-range T0 stars":
        (
            t0_malformed_stars > 0
            or t0_invalid_range > 0
        ),

    "malformed or out-of-range T1 stars":
        (
            t1_malformed_stars > 0
            or t1_invalid_range > 0
        ),

    "review-star transitions do not reconcile":
        star_transition_total
        != len(eligible),

    "unexpected T1 axis":
        unexpected_axis_count > 0,

    "unexpected NoClassification count":
        int(
            no_classification_mask.sum()
        )
        != EXPECTED_NO_CLASSIFICATION_ROWS,

    "unexpected non-germline-axis count":
        int(
            non_germline_mask.sum()
        )
        != EXPECTED_NON_GERMLINE_ROWS,

    "T0 expert-panel/star inconsistency":
        (
            t0_expert_not_three_star > 0
            or t0_three_star_not_expert > 0
        ),

    "T1 expert-panel/star inconsistency":
        (
            t1_expert_not_three_star > 0
            or t1_three_star_not_expert > 0
        ),
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\n" + "=" * 116)
print("STAGE 5A STEP 3E DECISION")
print("=" * 116)

if failed_checks:

    print(
        "REVIEW_STAGE5A_REVIEW_STATUS_AND_STAR_DRIFT"
    )

    print("\nChecks requiring review:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Review-status or review-star semantics require "
        "resolution before the outcome policy is frozen."
    )

print(
    "PASS_STAGE5A_REVIEW_STATUS_STAR_AND_EXPERT_PANEL_DRIFT_AUDITED"
)

print()
print(
    "Interpretation:"
)
print(
    "Review-star and expert-panel changes are valid secondary "
    "evidence-drift measures."
)
print(
    "A review-star change alone must not define the primary "
    "future-instability outcome."
)

print()
print(
    "Primary outcome assigned:          NO"
)
print(
    "Review-drift outcome frozen:       NO"
)
print(
    "GES scores loaded:                 NO"
)
print(
    "Temporal performance examined:     NO"
)
print(
    "Files written or modified:         NO"
)

STAGE 5A STEP 3E — REVIEW-STATUS, STAR, AND EXPERT-PANEL DRIFT AUDIT

BASIC ACCOUNTING
--------------------------------------------------------------------------------------------------------------------
Complete Stage 3H rows:                  71,659
Outcome-eligible linked rows:            70,583
Missing eligibility values:              0
Invalid eligibility values:              0

REVIEW-STAR STRUCTURAL CHECKS
--------------------------------------------------------------------------------------------------------------------
T0 missing review-star values:           0
T0 malformed review-star values:         0
T0 review stars outside integer 0-4:     0
T1 missing review-star values:           0
T1 malformed review-star values:         0
T1 review stars outside integer 0-4:     0

T0 REVIEW-STAR DISTRIBUTION
--------------------------------------------------------------------------------------------------------------------
_t0_star
1    49330
2     9223
3     8157
0     3873

T1 REVIE

In [12]:
# ==================================================================================================
# STAGE 5B — STEP 1
# FREEZE THE PRIMARY FUTURE-INSTABILITY OUTCOME POLICY
#
# Purpose:
#   1. Cryptographically reverify the frozen T0, T1, and Stage 3H inputs.
#   2. Freeze the exact classification-group mapping.
#   3. Freeze the T1 classification-axis eligibility policy.
#   4. Freeze the three prespecified primary instability components.
#   5. Freeze outcome-positive, outcome-negative, and nonevaluable assignment precedence.
#   6. Preserve review-star and expert-panel drift as secondary outcomes only.
#
# Important:
#   - This cell freezes rules only.
#   - It does not create any record-level future-instability label.
#   - It does not load Stage 4 GES scores.
#   - It does not calculate event prevalence or temporal performance.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import tempfile

import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Project locations
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

T0_PATH = (
    PROJECT_ROOT
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    PROJECT_ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3H_LINKAGE_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage3_crosswalk"
    / "stage3h_final_t0_t1_linkage_v1.parquet"
)

STAGE5_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
)

STAGE5_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

POLICY_PATH = (
    STAGE5_CONFIG_DIR
    / "stage5_primary_future_instability_outcome_policy_v1.json"
)

POLICY_SHA256_PATH = (
    STAGE5_CONFIG_DIR
    / "stage5_primary_future_instability_outcome_policy_v1.sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected input checksums
# --------------------------------------------------------------------------------------------------

EXPECTED_INPUTS = {
    "t0_accepted_cohort": {
        "path": T0_PATH,
        "sha256": (
            "f6b6760b2ad6e4352e3bdecdeaf89827"
            "e8a514b031abf2373bb17568d999466d"
        ),
        "rows": 71_659,
        "columns": 34,
    },

    "t1_accepted_cohort": {
        "path": T1_PATH,
        "sha256": (
            "5713a11bdbf4804758cc011f9b2f302a"
            "fc91fa1f88c1b178d675c28bb277d37c"
        ),
        "rows": 100_920,
        "columns": 36,
    },

    "stage3h_final_linkage": {
        "path": STAGE3H_LINKAGE_PATH,
        "sha256": (
            "77d0522af5ec3ac0938a03aecdb9ebd2"
            "30d6cafae8e8ded7a8de1fd462cfbe75"
        ),
        "rows": 71_659,
        "columns": 81,
    },
}


# --------------------------------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading the entire file into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def canonical_json_bytes(
    value,
) -> bytes:
    """
    Create deterministic UTF-8 JSON bytes for checksum freezing.
    """

    text = (
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n"
    )

    return text.encode("utf-8")


def write_immutable(
    destination: Path,
    content: bytes,
) -> str:
    """
    Write a new immutable artifact atomically.

    If the file already exists:
      - accept it only when its bytes are exactly identical;
      - otherwise stop rather than overwrite a frozen artifact.
    """

    if destination.exists():

        existing_content = destination.read_bytes()

        if existing_content != content:
            raise RuntimeError(
                "A different frozen artifact already exists:\n"
                f"{destination}\n\n"
                "The existing file was not overwritten."
            )

        return "EXISTING_IDENTICAL_ARTIFACT_VERIFIED"

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=destination.parent,
        prefix=f".{destination.name}.",
        suffix=".tmp",
        delete=False,
    ) as temporary_file:

        temporary_path = Path(
            temporary_file.name
        )

        temporary_file.write(
            content
        )

        temporary_file.flush()
        os.fsync(
            temporary_file.fileno()
        )

    temporary_path.replace(
        destination
    )

    return "NEW_IMMUTABLE_ARTIFACT_WRITTEN"


# --------------------------------------------------------------------------------------------------
# 4. Reverify frozen inputs
# --------------------------------------------------------------------------------------------------

print("=" * 118)
print("STAGE 5B STEP 1 — PRIMARY FUTURE-INSTABILITY OUTCOME POLICY FREEZE")
print("=" * 118)

print("\nFROZEN INPUT VERIFICATION")
print("-" * 118)

verified_inputs = {}

for artifact_name, specification in EXPECTED_INPUTS.items():

    artifact_path = specification["path"]

    if not artifact_path.exists():
        raise FileNotFoundError(
            "Required frozen input was not found:\n"
            f"{artifact_path}"
        )

    observed_sha256 = calculate_sha256(
        artifact_path
    )

    parquet_file = pq.ParquetFile(
        artifact_path
    )

    observed_rows = (
        parquet_file.metadata.num_rows
    )

    observed_columns = (
        parquet_file.metadata.num_columns
    )

    checksum_pass = (
        observed_sha256
        == specification["sha256"]
    )

    rows_pass = (
        observed_rows
        == specification["rows"]
    )

    columns_pass = (
        observed_columns
        == specification["columns"]
    )

    print(f"\n{artifact_name}")
    print(
        f"  SHA-256: "
        f"{'PASS' if checksum_pass else 'FAIL'}"
    )
    print(
        f"  Rows:    "
        f"{observed_rows:,} "
        f"| expected {specification['rows']:,} "
        f"| {'PASS' if rows_pass else 'FAIL'}"
    )
    print(
        f"  Columns: "
        f"{observed_columns:,} "
        f"| expected {specification['columns']:,} "
        f"| {'PASS' if columns_pass else 'FAIL'}"
    )

    if not (
        checksum_pass
        and rows_pass
        and columns_pass
    ):
        raise RuntimeError(
            "Frozen-input verification failed for "
            f"{artifact_name}."
        )

    verified_inputs[artifact_name] = {
        "path": str(
            artifact_path
        ),
        "sha256": observed_sha256,
        "rows": observed_rows,
        "columns": observed_columns,
    }


# --------------------------------------------------------------------------------------------------
# 5. Preserve the original freeze timestamp on a safe rerun
# --------------------------------------------------------------------------------------------------

existing_policy = None

if POLICY_PATH.exists():

    try:
        existing_policy = json.loads(
            POLICY_PATH.read_text(
                encoding="utf-8"
            )
        )
    except Exception as error:
        raise RuntimeError(
            "An existing policy file could not be parsed:\n"
            f"{POLICY_PATH}"
        ) from error

freeze_timestamp = (
    existing_policy.get(
        "frozen_at_utc"
    )
    if existing_policy
    else datetime.now(
        timezone.utc
    ).isoformat()
)


# --------------------------------------------------------------------------------------------------
# 6. Freeze the exact Stage 5 primary-outcome policy
# --------------------------------------------------------------------------------------------------

policy = {
    "policy_id": (
        "GES_STAGE5_PRIMARY_FUTURE_INSTABILITY_OUTCOME_POLICY"
    ),

    "version": "1.0.0",

    "status": (
        "FROZEN_BEFORE_RECORD_LEVEL_OUTCOME_ASSIGNMENT"
    ),

    "frozen_at_utc": freeze_timestamp,

    "study": {
        "title": (
            "GES-RAG: Temporal Validation and Stability-Aware "
            "Context Assembly for Reliable Genomic Question Answering"
        ),

        "experiment": (
            "Experiment 1: Temporal Validation of GES"
        ),

        "primary_unit": (
            "RCV-level variant-condition aggregate"
        ),

        "t0_archive_label": (
            "ClinVar January 2023"
        ),

        "t0_embedded_cutoff": (
            "2022-12-31"
        ),

        "t1_archive_label": (
            "ClinVar January 2026"
        ),

        "t1_embedded_cutoff": (
            "2025-12-27"
        ),

        "primary_genes": [
            "BRCA1",
            "BRCA2",
            "MLH1",
        ],

        "exploratory_gene": (
            "EGFR"
        ),
    },

    "verified_frozen_inputs": (
        verified_inputs
    ),

    "stage3_linkage_boundary": {
        "total_t0_records": 71_659,

        "accepted_linkage_records": 70_583,

        "unresolved_linkage_records": 1_076,

        "accepted_exact_links": 70_413,

        "accepted_nonexact_tier1_one_to_one_links": 170,

        "unresolved_records_must_not_be_called_stable": True,
    },

    "semantic_precheck_status": {
        "stage5a_complete": True,

        "classification_and_axis_inventory_passed": True,

        "no_classification_sentinel_confirmed": True,

        "conflict_field_semantics_confirmed": True,

        "prior_conflict_resolution_patterns_audited": True,

        "review_star_and_expert_panel_drift_audited": True,

        "ges_scores_loaded_during_precheck": False,

        "temporal_performance_examined": False,
    },

    "semantic_precheck_observations": {
        "outcome_eligible_stage3_links": 70_583,

        "t1_germline_axis_records": 67_489,

        "t1_no_classification_records": 3_092,

        "t1_non_germline_selected_axis_records": 2,

        "triad_to_triad_records": 60_376,

        "diagnostic_triad_change_candidates": 1_405,

        "diagnostic_new_conflict_candidates": 4_789,

        "t0_conflict_positive_records": 1_481,

        "conflict_resolved_by_t1_records": 298,

        "material_conflicting_to_triad_candidates": 297,

        "flag_only_same_group_resolution_records": 1,

        "same_review_star_records": 60_375,

        "t0_group_flag_semantic_differences": 4,

        "t1_group_flag_semantic_differences": 27,

        "all_group_flag_differences_explained_by_explicit_conflict_review_status": True,

        "counts_used_to_select_or_tune_rules": False,
    },

    "canonical_classification_group_mapping": {
        "Benign/Likely benign": "BLB",

        "VUS": "VUS",

        "Pathogenic/Likely pathogenic": "PLP",

        "Conflicting": "CONFLICTING",

        "Other": "OTHER",

        "Missing": "MISSING",

        "Mixed": "MIXED",
    },

    "primary_clinical_groups": [
        "BLB",
        "VUS",
        "PLP",
    ],

    "primary_comparable_groups": [
        "BLB",
        "VUS",
        "PLP",
        "CONFLICTING",
    ],

    "classification_axis_policy": {
        "primary_axis": (
            "GermlineClassification"
        ),

        "NoClassification": {
            "primary_outcome_status": (
                "NONEVALUABLE"
            ),

            "reason_code": (
                "NONEVALUABLE_T1_NO_CLINICAL_CLASSIFICATION"
            ),

            "evidence_only_is_not_a_clinical_classification": True,
        },

        "OncogenicityClassification": {
            "primary_outcome_status": (
                "NONEVALUABLE"
            ),

            "reason_code": (
                "NONEVALUABLE_AXIS_INCOMPATIBLE_ONCOGENICITY"
            ),

            "future_use": (
                "Separate exploratory or sensitivity analysis"
            ),
        },

        "SomaticClinicalImpact": {
            "primary_outcome_status": (
                "NONEVALUABLE"
            ),

            "reason_code": (
                "NONEVALUABLE_AXIS_INCOMPATIBLE_SOMATIC_CLINICAL_IMPACT"
            ),

            "future_use": (
                "Separate exploratory or sensitivity analysis"
            ),
        },
    },

    "conflict_semantics": {
        "authoritative_record_level_field": (
            "aggregate_conflict_flag"
        ),

        "semantic_definition": [
            (
                "classification_group is Conflicting"
            ),
            (
                "OR review status explicitly reports "
                "conflicting interpretations/classifications"
            ),
        ],

        "classification_group_alone_is_not_sufficient": True,

        "scv_group_disagreement_is_a_separate_secondary_measure": True,
    },

    "primary_instability_event_components": {
        "component_1_material_classification_group_change": {
            "code": (
                "MATERIAL_CLINICAL_GROUP_CHANGE"
            ),

            "criteria": [
                (
                    "T1 selected classification axis is "
                    "GermlineClassification"
                ),
                (
                    "T0 canonical classification group is one of "
                    "BLB, VUS, or PLP"
                ),
                (
                    "T1 canonical classification group is one of "
                    "BLB, VUS, or PLP"
                ),
                (
                    "T0 and T1 canonical clinical groups differ"
                ),
            ],

            "review_star_change_required": False,
        },

        "component_2_new_unresolved_conflict": {
            "code": (
                "NEW_UNRESOLVED_CONFLICT_AT_T1"
            ),

            "criteria": [
                (
                    "T1 selected classification axis is "
                    "GermlineClassification"
                ),
                (
                    "T0 aggregate_conflict_flag is False"
                ),
                (
                    "T1 aggregate_conflict_flag is True"
                ),
            ],

            "uses_authoritative_conflict_flag": True,
        },

        "component_3_material_prior_conflict_resolution": {
            "code": (
                "PRIOR_CONFLICT_RESOLVED_TO_MATERIAL_GROUP"
            ),

            "criteria": [
                (
                    "T1 selected classification axis is "
                    "GermlineClassification"
                ),
                (
                    "T0 aggregate_conflict_flag is True"
                ),
                (
                    "T1 aggregate_conflict_flag is False"
                ),
                (
                    "T0 canonical classification group is CONFLICTING"
                ),
                (
                    "T1 canonical classification group is one of "
                    "BLB, VUS, or PLP"
                ),
            ],

            "flag_only_resolution_with_same_clinical_group_is_not_primary_instability": True,
        },
    },

    "primary_outcome_assignment_precedence": [
        {
            "order": 1,

            "condition": (
                "Stage 3 linkage is unresolved or censored"
            ),

            "primary_outcome": None,

            "status": (
                "LINKAGE_CENSORED"
            ),
        },

        {
            "order": 2,

            "condition": (
                "T1 selected classification axis is "
                "NoClassification, OncogenicityClassification, "
                "or SomaticClinicalImpact"
            ),

            "primary_outcome": None,

            "status": (
                "PRIMARY_OUTCOME_NONEVALUABLE_AXIS"
            ),
        },

        {
            "order": 3,

            "condition": (
                "At least one prespecified primary instability "
                "event component is True"
            ),

            "primary_outcome": 1,

            "status": (
                "PRIMARY_FUTURE_INSTABILITY_EVENT"
            ),
        },

        {
            "order": 4,

            "condition": (
                "No primary event component is True AND both T0 "
                "and T1 canonical groups are in BLB, VUS, PLP, "
                "or CONFLICTING"
            ),

            "primary_outcome": 0,

            "status": (
                "NO_PRIMARY_FUTURE_INSTABILITY_EVENT"
            ),
        },

        {
            "order": 5,

            "condition": (
                "No primary event component is True, but one or "
                "both classification groups are OTHER, MISSING, "
                "MIXED, or otherwise outside the primary comparable set"
            ),

            "primary_outcome": None,

            "status": (
                "PRIMARY_OUTCOME_NONEVALUABLE_NONPRIMARY_CLASSIFICATION"
            ),
        },
    ],

    "negative_outcome_interpretation": {
        "meaning": (
            "No prespecified primary future-instability event "
            "was observed during the T0-to-T1 interval."
        ),

        "does_not_mean_permanent_clinical_stability": True,

        "does_not_mean_absence_of_all_evidence_drift": True,
    },

    "secondary_outcomes_not_part_of_primary_binary_label": [
        "review-star change",

        "exact review-status change",

        "new expert-panel involvement",

        "loss of expert-panel involvement",

        "classification availability change",

        "selected classification-axis change",

        "SCV disagreement change",

        "submitter-distribution change",

        "persistent conflict",

        "selected-group change while conflict remains unresolved",
    ],

    "leakage_controls": {
        "stage4_ges_scores_loaded": False,

        "stage4_model_coefficients_loaded": False,

        "ges_threshold_used_to_define_outcome": False,

        "outcome_prevalence_used_to_tune_rules": False,

        "temporal_performance_examined": False,

        "review_star_change_used_as_primary_event": False,

        "unresolved_linkage_records_called_stable": False,

        "rule_changes_after_score_join_prohibited": True,
    },

    "next_authorized_action": (
        "Construct the record-level primary and secondary "
        "outcome table using this frozen policy, then checksum "
        "and freeze the outcome artifact before joining GES scores."
    ),

    "software_environment": {
        "python_version": (
            platform.python_version()
        ),

        "policy_serialization": (
            "UTF-8 deterministic JSON with sorted keys"
        ),
    },
}


# --------------------------------------------------------------------------------------------------
# 7. Serialize and checksum the policy
# --------------------------------------------------------------------------------------------------

policy_bytes = canonical_json_bytes(
    policy
)

policy_sha256 = hashlib.sha256(
    policy_bytes
).hexdigest()

write_status = write_immutable(
    POLICY_PATH,
    policy_bytes,
)

sha256_sidecar_content = (
    f"{policy_sha256}  {POLICY_PATH.name}\n"
).encode("utf-8")

sha256_write_status = write_immutable(
    POLICY_SHA256_PATH,
    sha256_sidecar_content,
)


# --------------------------------------------------------------------------------------------------
# 8. Full readback and checksum verification
# --------------------------------------------------------------------------------------------------

readback_bytes = (
    POLICY_PATH.read_bytes()
)

readback_sha256 = hashlib.sha256(
    readback_bytes
).hexdigest()

readback_policy = json.loads(
    readback_bytes.decode("utf-8")
)

sidecar_text = (
    POLICY_SHA256_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

expected_sidecar_text = (
    f"{policy_sha256}  {POLICY_PATH.name}"
)

readback_pass = (
    readback_bytes == policy_bytes
)

checksum_pass = (
    readback_sha256 == policy_sha256
)

sidecar_pass = (
    sidecar_text
    == expected_sidecar_text
)

identity_pass = (
    readback_policy.get(
        "policy_id"
    )
    == policy["policy_id"]
)

version_pass = (
    readback_policy.get(
        "version"
    )
    == "1.0.0"
)

status_pass = (
    readback_policy.get(
        "status"
    )
    == "FROZEN_BEFORE_RECORD_LEVEL_OUTCOME_ASSIGNMENT"
)


# --------------------------------------------------------------------------------------------------
# 9. Final decision
# --------------------------------------------------------------------------------------------------

print("\nPOLICY ARTIFACT")
print("-" * 118)
print(
    f"Policy path:       {POLICY_PATH}"
)
print(
    f"Write status:      {write_status}"
)
print(
    f"SHA-256 path:      {POLICY_SHA256_PATH}"
)
print(
    f"Sidecar status:    {sha256_write_status}"
)
print(
    f"Policy SHA-256:    {policy_sha256}"
)

print("\nREADBACK VALIDATION")
print("-" * 118)
print(
    f"Exact byte readback: "
    f"{'PASS' if readback_pass else 'FAIL'}"
)
print(
    f"SHA-256 verification: "
    f"{'PASS' if checksum_pass else 'FAIL'}"
)
print(
    f"SHA-256 sidecar: "
    f"{'PASS' if sidecar_pass else 'FAIL'}"
)
print(
    f"Policy identity: "
    f"{'PASS' if identity_pass else 'FAIL'}"
)
print(
    f"Policy version: "
    f"{'PASS' if version_pass else 'FAIL'}"
)
print(
    f"Freeze status: "
    f"{'PASS' if status_pass else 'FAIL'}"
)

failed_checks = []

if not readback_pass:
    failed_checks.append(
        "exact policy-byte readback"
    )

if not checksum_pass:
    failed_checks.append(
        "policy SHA-256 verification"
    )

if not sidecar_pass:
    failed_checks.append(
        "SHA-256 sidecar verification"
    )

if not identity_pass:
    failed_checks.append(
        "policy identity verification"
    )

if not version_pass:
    failed_checks.append(
        "policy version verification"
    )

if not status_pass:
    failed_checks.append(
        "policy freeze-status verification"
    )

print("\n" + "=" * 118)
print("STAGE 5B STEP 1 DECISION")
print("=" * 118)

if failed_checks:

    print(
        "FAIL_STAGE5_PRIMARY_OUTCOME_POLICY_FREEZE"
    )

    print("\nFailed checks:")

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "The primary-outcome policy was not safely frozen. "
        "Do not construct record-level outcomes."
    )

print(
    "PASS_STAGE5_PRIMARY_FUTURE_INSTABILITY_OUTCOME_POLICY_FROZEN"
)

print()
print(
    "Primary event definitions frozen:   YES"
)
print(
    "Axis eligibility policy frozen:     YES"
)
print(
    "Nonevaluable rules frozen:          YES"
)
print(
    "Review-star drift kept secondary:   YES"
)
print()
print(
    "Record-level outcomes created:      NO"
)
print(
    "GES scores loaded:                  NO"
)
print(
    "Temporal performance examined:      NO"
)

STAGE 5B STEP 1 — PRIMARY FUTURE-INSTABILITY OUTCOME POLICY FREEZE

FROZEN INPUT VERIFICATION
----------------------------------------------------------------------------------------------------------------------

t0_accepted_cohort
  SHA-256: PASS
  Rows:    71,659 | expected 71,659 | PASS
  Columns: 34 | expected 34 | PASS

t1_accepted_cohort
  SHA-256: PASS
  Rows:    100,920 | expected 100,920 | PASS
  Columns: 36 | expected 36 | PASS

stage3h_final_linkage
  SHA-256: PASS
  Rows:    71,659 | expected 71,659 | PASS
  Columns: 81 | expected 81 | PASS

POLICY ARTIFACT
----------------------------------------------------------------------------------------------------------------------
Policy path:       /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage5_outcomes/stage5_primary_future_instability_outcome_policy_v1.json
Write status:      NEW_IMMUTABLE_ARTIFACT_WRITTEN
SHA-256 path:      /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage5_outcomes/stage5_primary_futu

In [13]:
# ==================================================================================================
# STAGE 5B — STEP 2
# CONSTRUCT AND VALIDATE THE RECORD-LEVEL FUTURE-INSTABILITY OUTCOME TABLE
#
# Purpose:
#   1. Verify the frozen Stage 5 outcome policy and Stage 3H linkage.
#   2. Apply the frozen assignment precedence to all 71,659 T0 RCVs.
#   3. Construct the three prespecified primary-event components.
#   4. Assign:
#        - primary outcome = 1;
#        - primary outcome = 0;
#        - linkage-censored;
#        - axis-nonevaluable;
#        - classification-nonevaluable.
#   5. Construct secondary review, conflict, and expert-panel drift fields.
#   6. Validate complete accounting and write an immutable outcome Parquet
#      plus a construction-QC report.
#
# Scientific boundary:
#   - The frozen policy is not changed or tuned.
#   - No Stage 4 GES score, feature, weak-label, or model artifact is loaded.
#   - No AUPRC, AUROC, calibration, enrichment, or threshold analysis is performed.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import tempfile

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen paths and expected checksums
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

LINKAGE_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage3_crosswalk"
    / "stage3h_final_t0_t1_linkage_v1.parquet"
)

POLICY_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.json"
)

POLICY_SHA256_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.sha256"
)

OUTCOME_DIR = (
    PROJECT_ROOT
    / "data_processed"
    / "stage5_outcomes"
)

QC_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "quality_checks"
    / "stage5_outcomes"
)

OUTCOME_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

QC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTCOME_PATH = (
    OUTCOME_DIR
    / "stage5_primary_future_instability_outcomes_v1.parquet"
)

QC_PATH = (
    QC_DIR
    / "stage5_primary_future_instability_outcome_construction_qc_v1.json"
)

EXPECTED_LINKAGE_SHA256 = (
    "77d0522af5ec3ac0938a03aecdb9ebd2"
    "30d6cafae8e8ded7a8de1fd462cfbe75"
)

EXPECTED_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a"
    "999a1373e5f01d8f57b5812d67a57c4e"
)

EXPECTED_COUNTS = {
    "total_rows": 71_659,
    "accepted_links": 70_583,
    "linkage_censored": 1_076,
    "germline_axis": 67_489,
    "no_classification_axis": 3_092,
    "non_germline_selected_axis": 2,
    "material_group_change_component": 1_405,
    "new_conflict_component": 4_789,
    "material_prior_conflict_resolution_component": 297,
}


# --------------------------------------------------------------------------------------------------
# 2. General helpers
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading the complete file into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def is_missing_scalar(value) -> bool:
    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except Exception:
        pass

    return False


def normalize_boolean(value):
    if is_missing_scalar(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        if value == 1:
            return True

        if value == 0:
            return False

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(
        str(value).strip().casefold(),
        "INVALID",
    )


def normalize_text(value):
    if is_missing_scalar(value):
        return pd.NA

    text = str(value).strip()

    if text.casefold() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return pd.NA

    return text


def parse_single_gene(value):
    if is_missing_scalar(value):
        return pd.NA

    try:
        parsed = (
            value
            if isinstance(value, list)
            else json.loads(str(value))
        )
    except Exception:
        return "<PARSE_ERROR>"

    if not isinstance(parsed, list):
        return "<WRONG_TYPE>"

    genes = sorted(
        {
            str(gene).strip().upper()
            for gene in parsed
            if gene is not None
            and str(gene).strip()
        }
    )

    if len(genes) == 1:
        return genes[0]

    if not genes:
        return "<EMPTY>"

    return "|".join(genes)


def canonical_group(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "benign/likely benign": "BLB",
        "vus": "VUS",
        "pathogenic/likely pathogenic": "PLP",
        "conflicting": "CONFLICTING",
        "other": "OTHER",
        "missing": "MISSING",
        "mixed": "MIXED",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def canonical_axis(value):
    text = normalize_text(value)

    if pd.isna(text):
        return pd.NA

    mapping = {
        "germlineclassification": "GERMLINE",
        "noclassification": "NO_CLASSIFICATION",
        "oncogenicityclassification": "ONCOGENICITY",
        "somaticclinicalimpact": "SOMATIC_CLINICAL_IMPACT",
    }

    return mapping.get(
        text.casefold(),
        f"UNEXPECTED:{text}",
    )


def canonical_json_bytes(value) -> bytes:
    return (
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")


def value_counts_dict(series: pd.Series) -> dict:
    """
    JSON-safe value counts.
    """

    display = (
        series.astype("string")
        .fillna("<MISSING>")
    )

    return {
        str(key): int(value)
        for key, value in (
            display.value_counts(
                dropna=False
            )
            .sort_index()
            .items()
        )
    }


def crosstab_dict(
    row_series: pd.Series,
    column_series: pd.Series,
) -> dict:
    """
    JSON-safe cross-tabulation.
    """

    table = pd.crosstab(
        row_series.astype("string")
        .fillna("<MISSING>"),
        column_series.astype("string")
        .fillna("<MISSING>"),
        dropna=False,
    )

    return {
        str(row_name): {
            str(column_name): int(value)
            for column_name, value in row.items()
        }
        for row_name, row in table.iterrows()
    }


# --------------------------------------------------------------------------------------------------
# 3. Immutable-write helpers
# --------------------------------------------------------------------------------------------------

def write_parquet_immutable(
    frame: pd.DataFrame,
    destination: Path,
) -> str:
    """
    Write a new Parquet artifact atomically.

    If an artifact already exists, it is accepted only if its
    complete dataframe content and column order are identical.
    """

    if destination.exists():

        existing = pd.read_parquet(
            destination
        )

        try:
            pd.testing.assert_frame_equal(
                existing,
                frame,
                check_dtype=False,
                check_like=False,
                check_exact=True,
            )
        except AssertionError as error:
            raise RuntimeError(
                "A different outcome artifact already exists:\n"
                f"{destination}\n\n"
                "The existing artifact was not overwritten."
            ) from error

        return "EXISTING_IDENTICAL_ARTIFACT_VERIFIED"

    file_descriptor, temporary_name = (
        tempfile.mkstemp(
            prefix=f".{destination.name}.",
            suffix=".tmp",
            dir=destination.parent,
        )
    )

    os.close(
        file_descriptor
    )

    temporary_path = Path(
        temporary_name
    )

    try:
        frame.to_parquet(
            temporary_path,
            index=False,
            engine="pyarrow",
            compression="zstd",
        )

        os.replace(
            temporary_path,
            destination,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    return "NEW_IMMUTABLE_ARTIFACT_WRITTEN"


def write_json_immutable(
    destination: Path,
    value: dict,
) -> str:
    """
    Write deterministic JSON atomically and prevent overwriting
    a different artifact.
    """

    content = canonical_json_bytes(
        value
    )

    if destination.exists():

        existing_content = (
            destination.read_bytes()
        )

        if existing_content != content:
            raise RuntimeError(
                "A different QC artifact already exists:\n"
                f"{destination}\n\n"
                "The existing artifact was not overwritten."
            )

        return "EXISTING_IDENTICAL_ARTIFACT_VERIFIED"

    file_descriptor, temporary_name = (
        tempfile.mkstemp(
            prefix=f".{destination.name}.",
            suffix=".tmp",
            dir=destination.parent,
        )
    )

    os.close(
        file_descriptor
    )

    temporary_path = Path(
        temporary_name
    )

    try:
        temporary_path.write_bytes(
            content
        )

        os.replace(
            temporary_path,
            destination,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    return "NEW_IMMUTABLE_ARTIFACT_WRITTEN"


# --------------------------------------------------------------------------------------------------
# 4. Verify the frozen policy and frozen linkage
# --------------------------------------------------------------------------------------------------

print("=" * 122)
print("STAGE 5B STEP 2 — RECORD-LEVEL FUTURE-INSTABILITY OUTCOME CONSTRUCTION")
print("=" * 122)

for required_path in [
    LINKAGE_PATH,
    POLICY_PATH,
    POLICY_SHA256_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Required frozen artifact was not found:\n"
            f"{required_path}"
        )

observed_linkage_sha256 = (
    calculate_sha256(
        LINKAGE_PATH
    )
)

observed_policy_sha256 = (
    calculate_sha256(
        POLICY_PATH
    )
)

sidecar_text = (
    POLICY_SHA256_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

expected_sidecar_text = (
    f"{EXPECTED_POLICY_SHA256}  "
    f"{POLICY_PATH.name}"
)

policy = json.loads(
    POLICY_PATH.read_text(
        encoding="utf-8"
    )
)

print("\nFROZEN ARTIFACT VERIFICATION")
print("-" * 122)
print(
    f"Stage 3H linkage SHA-256: "
    f"{'PASS' if observed_linkage_sha256 == EXPECTED_LINKAGE_SHA256 else 'FAIL'}"
)
print(
    f"Stage 5 policy SHA-256:   "
    f"{'PASS' if observed_policy_sha256 == EXPECTED_POLICY_SHA256 else 'FAIL'}"
)
print(
    f"Policy SHA sidecar:       "
    f"{'PASS' if sidecar_text == expected_sidecar_text else 'FAIL'}"
)
print(
    f"Policy version:           "
    f"{policy.get('version')}"
)
print(
    f"Policy status:            "
    f"{policy.get('status')}"
)

if observed_linkage_sha256 != EXPECTED_LINKAGE_SHA256:
    raise RuntimeError(
        "Stage 3H linkage checksum verification failed."
    )

if observed_policy_sha256 != EXPECTED_POLICY_SHA256:
    raise RuntimeError(
        "Stage 5 outcome-policy checksum verification failed."
    )

if sidecar_text != expected_sidecar_text:
    raise RuntimeError(
        "Stage 5 outcome-policy sidecar verification failed."
    )

if policy.get("version") != "1.0.0":
    raise RuntimeError(
        "Unexpected Stage 5 policy version."
    )

if (
    policy.get("status")
    != "FROZEN_BEFORE_RECORD_LEVEL_OUTCOME_ASSIGNMENT"
):
    raise RuntimeError(
        "The Stage 5 policy does not have the required frozen status."
    )


# --------------------------------------------------------------------------------------------------
# 5. Load only linkage and outcome-construction fields
# --------------------------------------------------------------------------------------------------

linkage_schema = set(
    pq.ParquetFile(
        LINKAGE_PATH
    ).schema.names
)

required_columns = [
    "t0_rcv_accession",
    "linked_t1_rcv_accession",
    "t1_rcv_accession",

    "linkage_status",
    "linkage_method",
    "linkage_decision_category",
    "censoring_disposition",
    "temporal_outcome_eligible",
    "future_instability_outcome_created",
    "future_instability_label",
    "stage3_freeze_version",

    "t0_target_genes_json",
    "t0_aggregate_classification",
    "t0_aggregate_classification_group",
    "t0_aggregate_review_status",
    "t0_aggregate_review_stars",
    "t0_aggregate_conflict_flag",
    "t0_scv_group_disagreement_flag",

    "t1_target_genes_json",
    "t1_aggregate_classification",
    "t1_aggregate_classification_group",
    "t1_aggregate_review_status",
    "t1_aggregate_review_stars",
    "t1_aggregate_conflict_flag",
    "t1_scv_group_disagreement_flag",
    "t1_aggregate_classification_axis",
]

optional_identifier_columns = [
    "t0_rcv_version",
    "t1_rcv_version",
    "t0_variation_id",
    "t1_variation_id",
    "t0_vcv_accession",
    "t1_vcv_accession",
    "t0_vcv_version",
    "t1_vcv_version",
]

missing_required_columns = sorted(
    set(required_columns).difference(
        linkage_schema
    )
)

if missing_required_columns:
    raise RuntimeError(
        "Required Stage 5 outcome fields are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_required_columns
        )
    )

available_optional_columns = [
    column
    for column in optional_identifier_columns
    if column in linkage_schema
]

loaded_columns = (
    required_columns
    + available_optional_columns
)

forbidden_loaded_columns = [
    column
    for column in loaded_columns
    if (
        "ges" in column.casefold()
        or "p_stable" in column.casefold()
        or "weak_label" in column.casefold()
    )
]

if forbidden_loaded_columns:
    raise RuntimeError(
        "A prohibited Stage 4/GES field was selected:\n"
        + "\n".join(
            forbidden_loaded_columns
        )
    )

linkage = pd.read_parquet(
    LINKAGE_PATH,
    columns=loaded_columns,
)


# --------------------------------------------------------------------------------------------------
# 6. Normalize control, gene, classification, axis, conflict, and review fields
# --------------------------------------------------------------------------------------------------

eligibility_raw = (
    linkage[
        "temporal_outcome_eligible"
    ]
    .apply(normalize_boolean)
)

source_outcome_created_raw = (
    linkage[
        "future_instability_outcome_created"
    ]
    .apply(normalize_boolean)
)

invalid_eligibility = int(
    eligibility_raw.eq("INVALID").sum()
)

missing_eligibility = int(
    eligibility_raw.isna().sum()
)

invalid_source_created = int(
    source_outcome_created_raw.eq(
        "INVALID"
    ).sum()
)

missing_source_created = int(
    source_outcome_created_raw.isna().sum()
)

linkage["_eligible"] = (
    eligibility_raw
    .replace(
        "INVALID",
        pd.NA,
    )
    .astype("boolean")
)

linkage["_source_outcome_created"] = (
    source_outcome_created_raw
    .replace(
        "INVALID",
        pd.NA,
    )
    .astype("boolean")
)

linkage["_t0_gene"] = (
    linkage[
        "t0_target_genes_json"
    ]
    .apply(parse_single_gene)
    .astype("string")
)

linkage["_t1_gene"] = (
    linkage[
        "t1_target_genes_json"
    ]
    .apply(parse_single_gene)
    .astype("string")
)

linkage["_t0_group"] = (
    linkage[
        "t0_aggregate_classification_group"
    ]
    .apply(canonical_group)
    .astype("string")
)

linkage["_t1_group"] = (
    linkage[
        "t1_aggregate_classification_group"
    ]
    .apply(canonical_group)
    .astype("string")
)

linkage["_t1_axis"] = (
    linkage[
        "t1_aggregate_classification_axis"
    ]
    .apply(canonical_axis)
    .astype("string")
)

for timepoint in [
    "t0",
    "t1",
]:

    normalized_conflict = (
        linkage[
            f"{timepoint}_aggregate_conflict_flag"
        ]
        .apply(normalize_boolean)
    )

    normalized_disagreement = (
        linkage[
            f"{timepoint}_scv_group_disagreement_flag"
        ]
        .apply(normalize_boolean)
    )

    linkage[
        f"_{timepoint}_conflict"
    ] = (
        normalized_conflict
        .replace(
            "INVALID",
            pd.NA,
        )
        .astype("boolean")
    )

    linkage[
        f"_{timepoint}_scv_disagreement"
    ] = (
        normalized_disagreement
        .replace(
            "INVALID",
            pd.NA,
        )
        .astype("boolean")
    )

    linkage[
        f"_{timepoint}_review_status"
    ] = (
        linkage[
            f"{timepoint}_aggregate_review_status"
        ]
        .apply(normalize_text)
        .astype("string")
    )

    linkage[
        f"_{timepoint}_star"
    ] = (
        pd.to_numeric(
            linkage[
                f"{timepoint}_aggregate_review_stars"
            ],
            errors="coerce",
        )
        .astype("Int64")
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate the source scientific boundary before assigning outcomes
# --------------------------------------------------------------------------------------------------

source_created_true = int(
    linkage[
        "_source_outcome_created"
    ]
    .eq(True)
    .sum()
)

source_future_labels_nonmissing = int(
    linkage[
        "future_instability_label"
    ]
    .notna()
    .sum()
)

accepted_mask = (
    linkage["_eligible"]
    .eq(True)
    .fillna(False)
)

linkage_censored_mask = (
    linkage["_eligible"]
    .eq(False)
    .fillna(False)
)

gene_mismatch_mask = (
    accepted_mask
    & linkage["_t0_gene"]
    .ne(
        linkage["_t1_gene"]
    )
    .fillna(True)
)

unexpected_group_mask = (
    accepted_mask
    & (
        linkage["_t0_group"]
        .str.startswith(
            "UNEXPECTED:",
            na=False,
        )
        |
        linkage["_t1_group"]
        .str.startswith(
            "UNEXPECTED:",
            na=False,
        )
    )
)

unexpected_axis_mask = (
    accepted_mask
    & linkage["_t1_axis"]
    .str.startswith(
        "UNEXPECTED:",
        na=False,
    )
)

accepted_missing_conflict = int(
    (
        accepted_mask
        & (
            linkage[
                "_t0_conflict"
            ].isna()
            |
            linkage[
                "_t1_conflict"
            ].isna()
        )
    ).sum()
)

if (
    invalid_eligibility > 0
    or missing_eligibility > 0
    or invalid_source_created > 0
    or missing_source_created > 0
    or source_created_true > 0
    or source_future_labels_nonmissing > 0
    or gene_mismatch_mask.any()
    or unexpected_group_mask.any()
    or unexpected_axis_mask.any()
    or accepted_missing_conflict > 0
):
    raise RuntimeError(
        "The frozen linkage failed the pre-assignment scientific-boundary check."
    )


# --------------------------------------------------------------------------------------------------
# 8. Construct the frozen policy masks
# --------------------------------------------------------------------------------------------------

CLINICAL_TRIAD = {
    "BLB",
    "VUS",
    "PLP",
}

PRIMARY_COMPARABLE_GROUPS = {
    "BLB",
    "VUS",
    "PLP",
    "CONFLICTING",
}

germline_mask = (
    accepted_mask
    & linkage["_t1_axis"]
    .eq("GERMLINE")
    .fillna(False)
)

no_classification_mask = (
    accepted_mask
    & linkage["_t1_axis"]
    .eq("NO_CLASSIFICATION")
    .fillna(False)
)

oncogenicity_mask = (
    accepted_mask
    & linkage["_t1_axis"]
    .eq("ONCOGENICITY")
    .fillna(False)
)

somatic_impact_mask = (
    accepted_mask
    & linkage["_t1_axis"]
    .eq("SOMATIC_CLINICAL_IMPACT")
    .fillna(False)
)

axis_nonevaluable_mask = (
    no_classification_mask
    | oncogenicity_mask
    | somatic_impact_mask
)

t0_triad_mask = (
    linkage["_t0_group"]
    .isin(CLINICAL_TRIAD)
    .fillna(False)
)

t1_triad_mask = (
    linkage["_t1_group"]
    .isin(CLINICAL_TRIAD)
    .fillna(False)
)

t0_comparable_mask = (
    linkage["_t0_group"]
    .isin(
        PRIMARY_COMPARABLE_GROUPS
    )
    .fillna(False)
)

t1_comparable_mask = (
    linkage["_t1_group"]
    .isin(
        PRIMARY_COMPARABLE_GROUPS
    )
    .fillna(False)
)


# --------------------------------------------------------------------------------------------------
# 9. Construct the three prespecified primary-event components
# --------------------------------------------------------------------------------------------------

component_material_group_change = (
    germline_mask
    & t0_triad_mask
    & t1_triad_mask
    & linkage["_t0_group"]
    .ne(
        linkage["_t1_group"]
    )
    .fillna(False)
)

component_new_conflict = (
    germline_mask
    & linkage["_t0_conflict"]
    .eq(False)
    .fillna(False)
    & linkage["_t1_conflict"]
    .eq(True)
    .fillna(False)
)

component_material_conflict_resolution = (
    germline_mask
    & linkage["_t0_conflict"]
    .eq(True)
    .fillna(False)
    & linkage["_t1_conflict"]
    .eq(False)
    .fillna(False)
    & linkage["_t0_group"]
    .eq("CONFLICTING")
    .fillna(False)
    & t1_triad_mask
)

primary_positive_mask = (
    component_material_group_change
    | component_new_conflict
    | component_material_conflict_resolution
)

primary_negative_mask = (
    germline_mask
    & ~primary_positive_mask
    & t0_comparable_mask
    & t1_comparable_mask
)

classification_nonevaluable_mask = (
    germline_mask
    & ~primary_positive_mask
    & ~(
        t0_comparable_mask
        & t1_comparable_mask
    )
)

primary_evaluable_mask = (
    primary_positive_mask
    | primary_negative_mask
)


# --------------------------------------------------------------------------------------------------
# 10. Build the record-level outcome dataframe
# --------------------------------------------------------------------------------------------------

outcome = pd.DataFrame(
    {
        "t0_rcv_accession":
            linkage["t0_rcv_accession"]
            .astype("string"),

        "linked_t1_rcv_accession":
            linkage["linked_t1_rcv_accession"]
            .astype("string"),

        "t1_rcv_accession":
            linkage["t1_rcv_accession"]
            .astype("string"),

        "target_gene":
            linkage["_t0_gene"],

        "linkage_status":
            linkage["linkage_status"]
            .astype("string"),

        "linkage_method":
            linkage["linkage_method"]
            .astype("string"),

        "linkage_decision_category":
            linkage[
                "linkage_decision_category"
            ]
            .astype("string"),

        "censoring_disposition":
            linkage[
                "censoring_disposition"
            ]
            .astype("string"),

        "stage3_temporal_outcome_eligible":
            linkage["_eligible"],

        "t0_aggregate_classification":
            linkage[
                "t0_aggregate_classification"
            ]
            .astype("string"),

        "t0_canonical_classification_group":
            linkage["_t0_group"],

        "t0_aggregate_review_status":
            linkage[
                "_t0_review_status"
            ],

        "t0_aggregate_review_stars":
            linkage["_t0_star"],

        "t0_aggregate_conflict_flag":
            linkage["_t0_conflict"],

        "t0_scv_group_disagreement_flag":
            linkage[
                "_t0_scv_disagreement"
            ],

        "t1_aggregate_classification_axis":
            linkage[
                "t1_aggregate_classification_axis"
            ]
            .astype("string"),

        "t1_canonical_classification_axis":
            linkage["_t1_axis"],

        "t1_aggregate_classification":
            linkage[
                "t1_aggregate_classification"
            ]
            .astype("string"),

        "t1_canonical_classification_group":
            linkage["_t1_group"],

        "t1_aggregate_review_status":
            linkage[
                "_t1_review_status"
            ],

        "t1_aggregate_review_stars":
            linkage["_t1_star"],

        "t1_aggregate_conflict_flag":
            linkage["_t1_conflict"],

        "t1_scv_group_disagreement_flag":
            linkage[
                "_t1_scv_disagreement"
            ],
    }
)

for optional_column in available_optional_columns:
    outcome[optional_column] = (
        linkage[optional_column]
    )


# --------------------------------------------------------------------------------------------------
# 11. Primary component fields
# --------------------------------------------------------------------------------------------------

component_columns = [
    "event_material_clinical_group_change",
    "event_new_unresolved_conflict_at_t1",
    "event_prior_conflict_resolved_to_material_group",
]

for column in component_columns:
    outcome[column] = pd.Series(
        pd.NA,
        index=outcome.index,
        dtype="boolean",
    )

outcome.loc[
    germline_mask,
    "event_material_clinical_group_change",
] = (
    component_material_group_change.loc[
        germline_mask
    ]
    .astype(bool)
    .to_numpy()
)

outcome.loc[
    germline_mask,
    "event_new_unresolved_conflict_at_t1",
] = (
    component_new_conflict.loc[
        germline_mask
    ]
    .astype(bool)
    .to_numpy()
)

outcome.loc[
    germline_mask,
    "event_prior_conflict_resolved_to_material_group",
] = (
    component_material_conflict_resolution.loc[
        germline_mask
    ]
    .astype(bool)
    .to_numpy()
)

outcome[
    "primary_event_component_count"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="Int8",
)

outcome.loc[
    germline_mask,
    "primary_event_component_count",
] = (
    component_material_group_change.loc[
        germline_mask
    ].astype("int8")
    +
    component_new_conflict.loc[
        germline_mask
    ].astype("int8")
    +
    component_material_conflict_resolution.loc[
        germline_mask
    ].astype("int8")
).astype("int8").to_numpy()


# --------------------------------------------------------------------------------------------------
# 12. Fixed-order primary event combination
# --------------------------------------------------------------------------------------------------

outcome[
    "primary_event_combination"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

event_code_order = [
    (
        component_material_group_change,
        "MATERIAL_CLINICAL_GROUP_CHANGE",
    ),
    (
        component_new_conflict,
        "NEW_UNRESOLVED_CONFLICT_AT_T1",
    ),
    (
        component_material_conflict_resolution,
        "PRIOR_CONFLICT_RESOLVED_TO_MATERIAL_GROUP",
    ),
]

for row_index in outcome.index[
    germline_mask
]:

    event_codes = [
        event_code
        for event_mask, event_code
        in event_code_order
        if bool(
            event_mask.loc[
                row_index
            ]
        )
    ]

    outcome.at[
        row_index,
        "primary_event_combination",
    ] = (
        "|".join(event_codes)
        if event_codes
        else "NO_PRIMARY_EVENT"
    )


# --------------------------------------------------------------------------------------------------
# 13. Primary outcome assignment using the frozen precedence
# --------------------------------------------------------------------------------------------------

outcome[
    "primary_outcome_evaluable"
] = pd.Series(
    False,
    index=outcome.index,
    dtype="boolean",
)

outcome.loc[
    primary_evaluable_mask,
    "primary_outcome_evaluable",
] = True

outcome[
    "primary_future_instability"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="Int8",
)

outcome.loc[
    primary_positive_mask,
    "primary_future_instability",
] = 1

outcome.loc[
    primary_negative_mask,
    "primary_future_instability",
] = 0

outcome[
    "primary_outcome_status"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

outcome[
    "primary_outcome_reason_code"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

# Precedence 1 — unresolved/censored linkage
outcome.loc[
    linkage_censored_mask,
    "primary_outcome_status",
] = "LINKAGE_CENSORED"

outcome.loc[
    linkage_censored_mask,
    "primary_outcome_reason_code",
] = (
    linkage.loc[
        linkage_censored_mask,
        "censoring_disposition",
    ]
    .astype("string")
    .to_numpy()
)

# Precedence 2 — incompatible or unavailable T1 classification axis
outcome.loc[
    no_classification_mask,
    "primary_outcome_status",
] = "PRIMARY_OUTCOME_NONEVALUABLE_AXIS"

outcome.loc[
    no_classification_mask,
    "primary_outcome_reason_code",
] = "NONEVALUABLE_T1_NO_CLINICAL_CLASSIFICATION"

outcome.loc[
    oncogenicity_mask,
    "primary_outcome_status",
] = "PRIMARY_OUTCOME_NONEVALUABLE_AXIS"

outcome.loc[
    oncogenicity_mask,
    "primary_outcome_reason_code",
] = "NONEVALUABLE_AXIS_INCOMPATIBLE_ONCOGENICITY"

outcome.loc[
    somatic_impact_mask,
    "primary_outcome_status",
] = "PRIMARY_OUTCOME_NONEVALUABLE_AXIS"

outcome.loc[
    somatic_impact_mask,
    "primary_outcome_reason_code",
] = "NONEVALUABLE_AXIS_INCOMPATIBLE_SOMATIC_CLINICAL_IMPACT"

# Precedence 3 — at least one primary event
outcome.loc[
    primary_positive_mask,
    "primary_outcome_status",
] = "PRIMARY_FUTURE_INSTABILITY_EVENT"

outcome.loc[
    primary_positive_mask,
    "primary_outcome_reason_code",
] = "AT_LEAST_ONE_PRESPECIFIED_PRIMARY_EVENT"

# Precedence 4 — no event among comparable groups
outcome.loc[
    primary_negative_mask,
    "primary_outcome_status",
] = "NO_PRIMARY_FUTURE_INSTABILITY_EVENT"

outcome.loc[
    primary_negative_mask,
    "primary_outcome_reason_code",
] = "NO_PRIMARY_EVENT_AMONG_COMPARABLE_GROUPS"

# Precedence 5 — no event but nonprimary classification state
outcome.loc[
    classification_nonevaluable_mask,
    "primary_outcome_status",
] = "PRIMARY_OUTCOME_NONEVALUABLE_NONPRIMARY_CLASSIFICATION"

outcome.loc[
    classification_nonevaluable_mask,
    "primary_outcome_reason_code",
] = "NONEVALUABLE_NONPRIMARY_CLASSIFICATION_GROUP"


# --------------------------------------------------------------------------------------------------
# 14. Classification transition direction
# --------------------------------------------------------------------------------------------------

outcome[
    "material_group_change_direction"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

outcome.loc[
    component_material_group_change,
    "material_group_change_direction",
] = (
    linkage.loc[
        component_material_group_change,
        "_t0_group",
    ]
    + "_TO_"
    + linkage.loc[
        component_material_group_change,
        "_t1_group",
    ]
).to_numpy()

outcome[
    "prior_conflict_resolution_final_group"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

outcome.loc[
    component_material_conflict_resolution,
    "prior_conflict_resolution_final_group",
] = (
    linkage.loc[
        component_material_conflict_resolution,
        "_t1_group",
    ]
    .to_numpy()
)


# --------------------------------------------------------------------------------------------------
# 15. Secondary conflict transitions
# --------------------------------------------------------------------------------------------------

outcome[
    "secondary_conflict_transition"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

no_conflict_both_mask = (
    germline_mask
    & linkage["_t0_conflict"]
    .eq(False)
    .fillna(False)
    & linkage["_t1_conflict"]
    .eq(False)
    .fillna(False)
)

new_conflict_mask = (
    germline_mask
    & linkage["_t0_conflict"]
    .eq(False)
    .fillna(False)
    & linkage["_t1_conflict"]
    .eq(True)
    .fillna(False)
)

persistent_conflict_mask = (
    germline_mask
    & linkage["_t0_conflict"]
    .eq(True)
    .fillna(False)
    & linkage["_t1_conflict"]
    .eq(True)
    .fillna(False)
)

resolved_conflict_mask = (
    germline_mask
    & linkage["_t0_conflict"]
    .eq(True)
    .fillna(False)
    & linkage["_t1_conflict"]
    .eq(False)
    .fillna(False)
)

outcome.loc[
    no_conflict_both_mask,
    "secondary_conflict_transition",
] = "NO_CONFLICT_AT_EITHER_TIME"

outcome.loc[
    new_conflict_mask,
    "secondary_conflict_transition",
] = "NEW_CONFLICT_AT_T1"

outcome.loc[
    persistent_conflict_mask,
    "secondary_conflict_transition",
] = "CONFLICT_PERSISTED"

outcome.loc[
    resolved_conflict_mask,
    "secondary_conflict_transition",
] = "CONFLICT_RESOLVED_BY_T1"


# --------------------------------------------------------------------------------------------------
# 16. Secondary review-star and review-status drift
# --------------------------------------------------------------------------------------------------

outcome[
    "secondary_review_star_delta"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="Int8",
)

outcome.loc[
    accepted_mask,
    "secondary_review_star_delta",
] = (
    linkage.loc[
        accepted_mask,
        "_t1_star",
    ]
    - linkage.loc[
        accepted_mask,
        "_t0_star",
    ]
).astype("Int8").to_numpy()

outcome[
    "secondary_review_star_transition"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

star_delta = (
    outcome[
        "secondary_review_star_delta"
    ]
)

outcome.loc[
    accepted_mask
    & star_delta.eq(0).fillna(False),
    "secondary_review_star_transition",
] = "UNCHANGED"

outcome.loc[
    accepted_mask
    & star_delta.gt(0).fillna(False),
    "secondary_review_star_transition",
] = "INCREASED"

outcome.loc[
    accepted_mask
    & star_delta.lt(0).fillna(False),
    "secondary_review_star_transition",
] = "DECREASED"

outcome[
    "secondary_review_status_changed"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="boolean",
)

outcome.loc[
    accepted_mask,
    "secondary_review_status_changed",
] = (
    linkage.loc[
        accepted_mask,
        "_t0_review_status",
    ]
    .ne(
        linkage.loc[
            accepted_mask,
            "_t1_review_status",
        ]
    )
    .fillna(False)
    .astype(bool)
    .to_numpy()
)


# --------------------------------------------------------------------------------------------------
# 17. Secondary expert-panel transitions
# --------------------------------------------------------------------------------------------------

EXPERT_PANEL_STATUS = (
    "reviewed by expert panel"
)

t0_expert_panel = (
    linkage[
        "_t0_review_status"
    ]
    .str.casefold()
    .eq(EXPERT_PANEL_STATUS)
    .fillna(False)
)

t1_expert_panel = (
    linkage[
        "_t1_review_status"
    ]
    .str.casefold()
    .eq(EXPERT_PANEL_STATUS)
    .fillna(False)
)

outcome[
    "secondary_expert_panel_transition"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

outcome.loc[
    accepted_mask
    & ~t0_expert_panel
    & ~t1_expert_panel,
    "secondary_expert_panel_transition",
] = "NO_EXPERT_PANEL_AT_EITHER_TIME"

outcome.loc[
    accepted_mask
    & ~t0_expert_panel
    & t1_expert_panel,
    "secondary_expert_panel_transition",
] = "NEW_EXPERT_PANEL_AT_T1"

outcome.loc[
    accepted_mask
    & t0_expert_panel
    & t1_expert_panel,
    "secondary_expert_panel_transition",
] = "EXPERT_PANEL_PERSISTED"

outcome.loc[
    accepted_mask
    & t0_expert_panel
    & ~t1_expert_panel,
    "secondary_expert_panel_transition",
] = "EXPERT_PANEL_NOT_PRESENT_AT_T1"


# --------------------------------------------------------------------------------------------------
# 18. Additional secondary fields and policy provenance
# --------------------------------------------------------------------------------------------------

outcome[
    "secondary_selected_group_transition"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="string",
)

outcome.loc[
    germline_mask,
    "secondary_selected_group_transition",
] = (
    linkage.loc[
        germline_mask,
        "_t0_group",
    ]
    + "_TO_"
    + linkage.loc[
        germline_mask,
        "_t1_group",
    ]
).to_numpy()

outcome[
    "secondary_scv_disagreement_changed"
] = pd.Series(
    pd.NA,
    index=outcome.index,
    dtype="boolean",
)

outcome.loc[
    accepted_mask,
    "secondary_scv_disagreement_changed",
] = (
    linkage.loc[
        accepted_mask,
        "_t0_scv_disagreement",
    ]
    .ne(
        linkage.loc[
            accepted_mask,
            "_t1_scv_disagreement",
        ]
    )
    .fillna(False)
    .astype(bool)
    .to_numpy()
)

outcome[
    "outcome_policy_version"
] = "1.0.0"

outcome[
    "outcome_policy_sha256"
] = EXPECTED_POLICY_SHA256

outcome[
    "source_stage3_freeze_version"
] = (
    linkage[
        "stage3_freeze_version"
    ]
    .astype("string")
)

outcome[
    "record_level_outcome_assignment_created"
] = pd.Series(
    True,
    index=outcome.index,
    dtype="boolean",
)


# --------------------------------------------------------------------------------------------------
# 19. Complete validation
# --------------------------------------------------------------------------------------------------

status_unassigned = int(
    outcome[
        "primary_outcome_status"
    ]
    .isna()
    .sum()
)

reason_unassigned = int(
    outcome[
        "primary_outcome_reason_code"
    ]
    .isna()
    .sum()
)

duplicate_t0_rows = int(
    outcome[
        "t0_rcv_accession"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)

missing_t0_keys = int(
    outcome[
        "t0_rcv_accession"
    ]
    .isna()
    .sum()
)

positive_count = int(
    outcome[
        "primary_future_instability"
    ]
    .eq(1)
    .sum()
)

negative_count = int(
    outcome[
        "primary_future_instability"
    ]
    .eq(0)
    .sum()
)

evaluable_count = int(
    outcome[
        "primary_outcome_evaluable"
    ]
    .eq(True)
    .sum()
)

nonevaluable_count = int(
    outcome[
        "primary_outcome_evaluable"
    ]
    .eq(False)
    .sum()
)

null_primary_labels = int(
    outcome[
        "primary_future_instability"
    ]
    .isna()
    .sum()
)

component_1_count = int(
    component_material_group_change.sum()
)

component_2_count = int(
    component_new_conflict.sum()
)

component_3_count = int(
    component_material_conflict_resolution.sum()
)

assignment_masks = pd.DataFrame(
    {
        "linkage_censored":
            linkage_censored_mask,

        "axis_nonevaluable":
            axis_nonevaluable_mask,

        "primary_positive":
            primary_positive_mask,

        "primary_negative":
            primary_negative_mask,

        "classification_nonevaluable":
            classification_nonevaluable_mask,
    }
)

assignment_count_per_row = (
    assignment_masks
    .astype("int8")
    .sum(axis=1)
)

rows_without_one_assignment = int(
    assignment_count_per_row
    .ne(1)
    .sum()
)

positive_without_component = int(
    (
        primary_positive_mask
        & outcome[
            "primary_event_component_count"
        ]
        .fillna(0)
        .eq(0)
    ).sum()
)

negative_with_component = int(
    (
        primary_negative_mask
        & outcome[
            "primary_event_component_count"
        ]
        .fillna(0)
        .gt(0)
    ).sum()
)

nonevaluable_with_binary_label = int(
    (
        ~primary_evaluable_mask
        & outcome[
            "primary_future_instability"
        ]
        .notna()
    ).sum()
)

evaluable_without_binary_label = int(
    (
        primary_evaluable_mask
        & outcome[
            "primary_future_instability"
        ]
        .isna()
    ).sum()
)

component_overlap_table = (
    outcome.loc[
        germline_mask,
        component_columns,
    ]
    .astype("Int8")
    .value_counts(
        dropna=False
    )
    .rename("record_count")
    .reset_index()
)

critical_failures = {
    "unexpected total row count":
        len(outcome)
        != EXPECTED_COUNTS[
            "total_rows"
        ],

    "missing T0 keys":
        missing_t0_keys > 0,

    "duplicate T0 keys":
        duplicate_t0_rows > 0,

    "unexpected accepted-link count":
        int(accepted_mask.sum())
        != EXPECTED_COUNTS[
            "accepted_links"
        ],

    "unexpected linkage-censored count":
        int(
            linkage_censored_mask.sum()
        )
        != EXPECTED_COUNTS[
            "linkage_censored"
        ],

    "unexpected germline-axis count":
        int(germline_mask.sum())
        != EXPECTED_COUNTS[
            "germline_axis"
        ],

    "unexpected NoClassification count":
        int(
            no_classification_mask.sum()
        )
        != EXPECTED_COUNTS[
            "no_classification_axis"
        ],

    "unexpected non-germline-axis count":
        int(
            oncogenicity_mask.sum()
            + somatic_impact_mask.sum()
        )
        != EXPECTED_COUNTS[
            "non_germline_selected_axis"
        ],

    "unexpected material-group-change count":
        component_1_count
        != EXPECTED_COUNTS[
            "material_group_change_component"
        ],

    "unexpected new-conflict count":
        component_2_count
        != EXPECTED_COUNTS[
            "new_conflict_component"
        ],

    "unexpected material-conflict-resolution count":
        component_3_count
        != EXPECTED_COUNTS[
            "material_prior_conflict_resolution_component"
        ],

    "rows do not have exactly one assignment":
        rows_without_one_assignment > 0,

    "unassigned outcome status":
        status_unassigned > 0,

    "unassigned outcome reason":
        reason_unassigned > 0,

    "evaluable accounting mismatch":
        (
            positive_count
            + negative_count
        )
        != evaluable_count,

    "nonevaluable accounting mismatch":
        null_primary_labels
        != nonevaluable_count,

    "positive record without an event component":
        positive_without_component > 0,

    "negative record with an event component":
        negative_with_component > 0,

    "nonevaluable record has a binary label":
        nonevaluable_with_binary_label > 0,

    "evaluable record lacks a binary label":
        evaluable_without_binary_label > 0,

    "GES or Stage 4 fields were loaded":
        len(
            forbidden_loaded_columns
        ) > 0,
}

failed_checks = [
    name
    for name, failed
    in critical_failures.items()
    if failed
]


# --------------------------------------------------------------------------------------------------
# 20. Print complete outcome accounting before writing
# --------------------------------------------------------------------------------------------------

print("\nPRIMARY OUTCOME STATUS DISTRIBUTION")
print("-" * 122)
print(
    outcome[
        "primary_outcome_status"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print("\nPRIMARY BINARY OUTCOME")
print("-" * 122)
print(
    f"Evaluable records:          "
    f"{evaluable_count:,}"
)
print(
    f"Primary outcome positive:   "
    f"{positive_count:,}"
)
print(
    f"Primary outcome negative:   "
    f"{negative_count:,}"
)
print(
    f"Nonevaluable/censored:       "
    f"{nonevaluable_count:,}"
)

if evaluable_count > 0:
    print(
        f"Observed event prevalence:  "
        f"{positive_count / evaluable_count:.6%}"
    )

print("\nPRIMARY EVENT COMPONENT COUNTS")
print("-" * 122)
print(
    f"Material clinical-group change:             "
    f"{component_1_count:,}"
)
print(
    f"New unresolved conflict at T1:              "
    f"{component_2_count:,}"
)
print(
    f"Prior conflict resolved to material group:  "
    f"{component_3_count:,}"
)

print("\nPRIMARY EVENT COMPONENT OVERLAP")
print("-" * 122)
print(
    component_overlap_table.to_string(
        index=False
    )
)

print("\nOUTCOME STATUS BY TARGET GENE")
print("-" * 122)
print(
    pd.crosstab(
        outcome[
            "target_gene"
        ],
        outcome[
            "primary_outcome_status"
        ],
        dropna=False,
    ).to_string()
)

print("\nPRIMARY EVENT COMBINATION DISTRIBUTION")
print("-" * 122)
print(
    outcome[
        "primary_event_combination"
    ]
    .fillna("<NONEVALUABLE_OR_CENSORED>")
    .value_counts(
        dropna=False
    )
    .to_string()
)

print("\nVALIDATION RESULTS")
print("-" * 122)
print(
    f"Rows without exactly one assignment:       "
    f"{rows_without_one_assignment:,}"
)
print(
    f"Unassigned outcome statuses:               "
    f"{status_unassigned:,}"
)
print(
    f"Unassigned outcome reasons:                "
    f"{reason_unassigned:,}"
)
print(
    f"Positive records without a component:      "
    f"{positive_without_component:,}"
)
print(
    f"Negative records with a component:         "
    f"{negative_with_component:,}"
)
print(
    f"Nonevaluable records with binary labels:   "
    f"{nonevaluable_with_binary_label:,}"
)
print(
    f"Evaluable records without binary labels:   "
    f"{evaluable_without_binary_label:,}"
)

if failed_checks:

    print("\nFAILED CHECKS")
    print("-" * 122)

    for check in failed_checks:
        print(" -", check)

    raise RuntimeError(
        "Record-level outcome construction failed validation. "
        "No outcome artifact was written."
    )


# --------------------------------------------------------------------------------------------------
# 21. Write and revalidate the immutable outcome Parquet
# --------------------------------------------------------------------------------------------------

outcome_write_status = (
    write_parquet_immutable(
        outcome,
        OUTCOME_PATH,
    )
)

outcome_sha256 = (
    calculate_sha256(
        OUTCOME_PATH
    )
)

outcome_parquet = pq.ParquetFile(
    OUTCOME_PATH
)

outcome_readback = pd.read_parquet(
    OUTCOME_PATH
)

pd.testing.assert_frame_equal(
    outcome_readback,
    outcome,
    check_dtype=False,
    check_like=False,
    check_exact=True,
)

if (
    outcome_parquet.metadata.num_rows
    != EXPECTED_COUNTS[
        "total_rows"
    ]
):
    raise RuntimeError(
        "Outcome Parquet readback row count failed."
    )


# --------------------------------------------------------------------------------------------------
# 22. Create construction-QC report
# --------------------------------------------------------------------------------------------------

existing_qc = None

if QC_PATH.exists():

    try:
        existing_qc = json.loads(
            QC_PATH.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        existing_qc = None

created_at_utc = (
    existing_qc.get(
        "created_at_utc"
    )
    if existing_qc
    else datetime.now(
        timezone.utc
    ).isoformat()
)

qc_report = {
    "report_id": (
        "GES_STAGE5_PRIMARY_OUTCOME_CONSTRUCTION_QC"
    ),

    "version": "1.0.0",

    "created_at_utc": (
        created_at_utc
    ),

    "decision": (
        "PASS_RECORD_LEVEL_OUTCOMES_CONSTRUCTED_AND_VALIDATED"
    ),

    "frozen_policy": {
        "path": str(
            POLICY_PATH
        ),

        "sha256": (
            EXPECTED_POLICY_SHA256
        ),

        "version": (
            policy["version"]
        ),
    },

    "source_linkage": {
        "path": str(
            LINKAGE_PATH
        ),

        "sha256": (
            observed_linkage_sha256
        ),

        "rows": int(
            len(linkage)
        ),
    },

    "outcome_artifact": {
        "path": str(
            OUTCOME_PATH
        ),

        "sha256": (
            outcome_sha256
        ),

        "rows": int(
            outcome_parquet.metadata.num_rows
        ),

        "columns": int(
            outcome_parquet.metadata.num_columns
        ),

        "row_groups": int(
            outcome_parquet.metadata.num_row_groups
        ),
    },

    "primary_outcome_accounting": {
        "evaluable_records": (
            evaluable_count
        ),

        "positive_records": (
            positive_count
        ),

        "negative_records": (
            negative_count
        ),

        "nonevaluable_or_censored_records": (
            nonevaluable_count
        ),

        "event_prevalence_among_evaluable": (
            positive_count
            / evaluable_count
            if evaluable_count
            else None
        ),

        "status_distribution": (
            value_counts_dict(
                outcome[
                    "primary_outcome_status"
                ]
            )
        ),

        "reason_distribution": (
            value_counts_dict(
                outcome[
                    "primary_outcome_reason_code"
                ]
            )
        ),
    },

    "primary_event_components": {
        "material_clinical_group_change": (
            component_1_count
        ),

        "new_unresolved_conflict_at_t1": (
            component_2_count
        ),

        "prior_conflict_resolved_to_material_group": (
            component_3_count
        ),

        "event_combination_distribution": (
            value_counts_dict(
                outcome[
                    "primary_event_combination"
                ]
            )
        ),
    },

    "secondary_drift": {
        "review_star_transition": (
            value_counts_dict(
                outcome[
                    "secondary_review_star_transition"
                ]
            )
        ),

        "conflict_transition": (
            value_counts_dict(
                outcome[
                    "secondary_conflict_transition"
                ]
            )
        ),

        "expert_panel_transition": (
            value_counts_dict(
                outcome[
                    "secondary_expert_panel_transition"
                ]
            )
        ),
    },

    "outcome_status_by_gene": (
        crosstab_dict(
            outcome[
                "target_gene"
            ],
            outcome[
                "primary_outcome_status"
            ],
        )
    ),

    "validation": {
        "missing_t0_keys": (
            missing_t0_keys
        ),

        "duplicate_t0_rows": (
            duplicate_t0_rows
        ),

        "rows_without_exactly_one_assignment": (
            rows_without_one_assignment
        ),

        "unassigned_outcome_status": (
            status_unassigned
        ),

        "unassigned_outcome_reason": (
            reason_unassigned
        ),

        "positive_without_component": (
            positive_without_component
        ),

        "negative_with_component": (
            negative_with_component
        ),

        "nonevaluable_with_binary_label": (
            nonevaluable_with_binary_label
        ),

        "evaluable_without_binary_label": (
            evaluable_without_binary_label
        ),

        "source_outcomes_present_before_construction": (
            source_created_true
        ),

        "source_future_labels_present_before_construction": (
            source_future_labels_nonmissing
        ),

        "stage4_or_ges_fields_loaded": (
            forbidden_loaded_columns
        ),

        "all_critical_checks_passed": True,
    },

    "scientific_boundary": {
        "ges_scores_loaded": False,

        "temporal_performance_examined": False,

        "outcome_rules_modified_after_policy_freeze": False,

        "unresolved_records_called_stable": False,

        "review_star_change_used_as_primary_event": False,
    },

    "next_authorized_action": (
        "Checksum-freeze the outcome artifact and QC report "
        "through a Stage 5 outcome freeze manifest before "
        "joining any GES score or comparator."
    ),
}

qc_write_status = (
    write_json_immutable(
        QC_PATH,
        qc_report,
    )
)

qc_sha256 = (
    calculate_sha256(
        QC_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 23. Final decision
# --------------------------------------------------------------------------------------------------

print("\nOUTPUT ARTIFACTS")
print("-" * 122)
print(
    f"Outcome path:        "
    f"{OUTCOME_PATH}"
)
print(
    f"Outcome write:       "
    f"{outcome_write_status}"
)
print(
    f"Outcome rows:        "
    f"{outcome_parquet.metadata.num_rows:,}"
)
print(
    f"Outcome columns:     "
    f"{outcome_parquet.metadata.num_columns:,}"
)
print(
    f"Outcome SHA-256:     "
    f"{outcome_sha256}"
)

print()
print(
    f"QC path:             "
    f"{QC_PATH}"
)
print(
    f"QC write:            "
    f"{qc_write_status}"
)
print(
    f"QC SHA-256:          "
    f"{qc_sha256}"
)

print("\n" + "=" * 122)
print("STAGE 5B STEP 2 DECISION")
print("=" * 122)

print(
    "PASS_STAGE5_RECORD_LEVEL_FUTURE_INSTABILITY_OUTCOMES_CONSTRUCTED"
)

print()
print(
    "Frozen policy applied without modification: YES"
)
print(
    "All T0 records accounted for:              YES"
)
print(
    "Unresolved links called stable:            NO"
)
print(
    "Incompatible axes called stable:           NO"
)
print(
    "Review-star change used as primary event:  NO"
)
print()
print(
    "GES scores loaded:                         NO"
)
print(
    "Temporal performance examined:             NO"
)
print(
    "Outcome freeze manifest created:           NO — next step"
)

STAGE 5B STEP 2 — RECORD-LEVEL FUTURE-INSTABILITY OUTCOME CONSTRUCTION

FROZEN ARTIFACT VERIFICATION
--------------------------------------------------------------------------------------------------------------------------
Stage 3H linkage SHA-256: PASS
Stage 5 policy SHA-256:   PASS
Policy SHA sidecar:       PASS
Policy version:           1.0.0
Policy status:            FROZEN_BEFORE_RECORD_LEVEL_OUTCOME_ASSIGNMENT

PRIMARY OUTCOME STATUS DISTRIBUTION
--------------------------------------------------------------------------------------------------------------------------
primary_outcome_status
NO_PRIMARY_FUTURE_INSTABILITY_EVENT                       60151
PRIMARY_FUTURE_INSTABILITY_EVENT                           6485
PRIMARY_OUTCOME_NONEVALUABLE_AXIS                          3094
LINKAGE_CENSORED                                           1076
PRIMARY_OUTCOME_NONEVALUABLE_NONPRIMARY_CLASSIFICATION      853

PRIMARY BINARY OUTCOME
----------------------------------------------------

In [14]:
# ==================================================================================================
# STAGE 5B — STEP 3
# CHECKSUM-FREEZE THE FUTURE-INSTABILITY OUTCOME ARTIFACT
#
# Purpose:
#   1. Verify the frozen policy, linkage, outcome Parquet, and construction-QC report.
#   2. Independently reproduce all critical outcome counts from the Parquet.
#   3. Confirm that no GES score, probability, weak label, or Stage 4 model output
#      is present in the outcome artifact.
#   4. Create immutable SHA-256 sidecars for the outcome and QC artifacts.
#   5. Create and checksum-freeze the Stage 5 outcome manifest.
#
# Scientific boundary:
#   - No Stage 4 GES score table is opened.
#   - No score-outcome join is performed.
#   - No temporal predictive-performance metric is calculated.
#   - No outcome rule is modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import tempfile

import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen artifact paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

LINKAGE_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage3_crosswalk"
    / "stage3h_final_t0_t1_linkage_v1.parquet"
)

POLICY_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.json"
)

POLICY_SHA256_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.sha256"
)

OUTCOME_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcomes_v1.parquet"
)

OUTCOME_SHA256_PATH = (
    OUTCOME_PATH.with_name(
        OUTCOME_PATH.name + ".sha256"
    )
)

QC_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "quality_checks"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcome_construction_qc_v1.json"
)

QC_SHA256_PATH = (
    QC_PATH.with_name(
        QC_PATH.name + ".sha256"
    )
)

MANIFEST_DIR = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    MANIFEST_DIR
    / "stage5_future_instability_outcome_freeze_manifest_v1.json"
)

MANIFEST_SHA256_PATH = (
    MANIFEST_DIR
    / "stage5_future_instability_outcome_freeze_manifest_v1.sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected checksums
# --------------------------------------------------------------------------------------------------

EXPECTED_LINKAGE_SHA256 = (
    "77d0522af5ec3ac0938a03aecdb9ebd2"
    "30d6cafae8e8ded7a8de1fd462cfbe75"
)

EXPECTED_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a"
    "999a1373e5f01d8f57b5812d67a57c4e"
)

EXPECTED_OUTCOME_SHA256 = (
    "c5508f5a8518160eef50482fd2c425dc"
    "4dcd9cf8a2fe04856e46760de60efbc8"
)

EXPECTED_QC_SHA256 = (
    "93753c0b4eed103a56aa81606850a935"
    "9b5107ee7aabf49fbf81c29f7c63c16a"
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen expected dimensions and outcome counts
# --------------------------------------------------------------------------------------------------

EXPECTED_OUTCOME_ROWS = 71_659
EXPECTED_OUTCOME_COLUMNS = 53

EXPECTED_EVALUABLE = 66_636
EXPECTED_POSITIVE = 6_485
EXPECTED_NEGATIVE = 60_151
EXPECTED_NONEVALUABLE_OR_CENSORED = 5_023

EXPECTED_COMPONENT_COUNTS = {
    "event_material_clinical_group_change": 1_405,
    "event_new_unresolved_conflict_at_t1": 4_789,
    "event_prior_conflict_resolved_to_material_group": 297,
}

EXPECTED_STATUS_COUNTS = {
    "NO_PRIMARY_FUTURE_INSTABILITY_EVENT": 60_151,
    "PRIMARY_FUTURE_INSTABILITY_EVENT": 6_485,
    "PRIMARY_OUTCOME_NONEVALUABLE_AXIS": 3_094,
    "LINKAGE_CENSORED": 1_076,
    "PRIMARY_OUTCOME_NONEVALUABLE_NONPRIMARY_CLASSIFICATION": 853,
}

EXPECTED_EVENT_COMBINATION_COUNTS = {
    "NO_PRIMARY_EVENT": 61_004,
    "NEW_UNRESOLVED_CONFLICT_AT_T1": 4_783,
    "MATERIAL_CLINICAL_GROUP_CHANGE": 1_399,
    "PRIOR_CONFLICT_RESOLVED_TO_MATERIAL_GROUP": 297,
    (
        "MATERIAL_CLINICAL_GROUP_CHANGE"
        "|NEW_UNRESOLVED_CONFLICT_AT_T1"
    ): 6,
}

EXPECTED_COMPONENT_OVERLAP = {
    (0, 0, 0): 61_004,
    (0, 1, 0): 4_783,
    (1, 0, 0): 1_399,
    (0, 0, 1): 297,
    (1, 1, 0): 6,
}


# --------------------------------------------------------------------------------------------------
# 4. Helpers
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading the entire file into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def canonical_json_bytes(
    value: dict,
) -> bytes:
    """
    Create deterministic UTF-8 JSON bytes.
    """

    return (
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")


def write_bytes_immutable(
    destination: Path,
    content: bytes,
) -> str:
    """
    Atomically write an immutable artifact.

    Existing files are accepted only when their bytes are
    exactly identical to the requested content.
    """

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination.exists():

        existing_content = (
            destination.read_bytes()
        )

        if existing_content != content:
            raise RuntimeError(
                "A different frozen artifact already exists:\n"
                f"{destination}\n\n"
                "The existing file was not overwritten."
            )

        return "EXISTING_IDENTICAL_ARTIFACT_VERIFIED"

    file_descriptor, temporary_name = (
        tempfile.mkstemp(
            prefix=f".{destination.name}.",
            suffix=".tmp",
            dir=destination.parent,
        )
    )

    os.close(
        file_descriptor
    )

    temporary_path = Path(
        temporary_name
    )

    try:
        temporary_path.write_bytes(
            content
        )

        os.replace(
            temporary_path,
            destination,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    return "NEW_IMMUTABLE_ARTIFACT_WRITTEN"


def make_sha256_sidecar(
    file_path: Path,
    observed_sha256: str,
) -> bytes:
    return (
        f"{observed_sha256}  {file_path.name}\n"
    ).encode("utf-8")


def integer_value_counts(
    series: pd.Series,
) -> dict:
    return {
        str(key): int(value)
        for key, value in (
            series.astype("string")
            .fillna("<MISSING>")
            .value_counts(
                dropna=False
            )
            .items()
        )
    }


# --------------------------------------------------------------------------------------------------
# 5. Verify all required artifacts exist
# --------------------------------------------------------------------------------------------------

required_paths = [
    LINKAGE_PATH,
    POLICY_PATH,
    POLICY_SHA256_PATH,
    OUTCOME_PATH,
    QC_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            "Required Stage 5 freeze input was not found:\n"
            f"{required_path}"
        )


# --------------------------------------------------------------------------------------------------
# 6. Cryptographic verification
# --------------------------------------------------------------------------------------------------

observed_linkage_sha256 = (
    calculate_sha256(
        LINKAGE_PATH
    )
)

observed_policy_sha256 = (
    calculate_sha256(
        POLICY_PATH
    )
)

observed_outcome_sha256 = (
    calculate_sha256(
        OUTCOME_PATH
    )
)

observed_qc_sha256 = (
    calculate_sha256(
        QC_PATH
    )
)

policy_sidecar_text = (
    POLICY_SHA256_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

expected_policy_sidecar_text = (
    f"{EXPECTED_POLICY_SHA256}  "
    f"{POLICY_PATH.name}"
)

print("=" * 124)
print("STAGE 5B STEP 3 — FUTURE-INSTABILITY OUTCOME CHECKSUM FREEZE")
print("=" * 124)

print("\nCRYPTOGRAPHIC VERIFICATION")
print("-" * 124)
print(
    f"Stage 3H linkage SHA-256: "
    f"{'PASS' if observed_linkage_sha256 == EXPECTED_LINKAGE_SHA256 else 'FAIL'}"
)
print(
    f"Stage 5 policy SHA-256:   "
    f"{'PASS' if observed_policy_sha256 == EXPECTED_POLICY_SHA256 else 'FAIL'}"
)
print(
    f"Policy SHA sidecar:       "
    f"{'PASS' if policy_sidecar_text == expected_policy_sidecar_text else 'FAIL'}"
)
print(
    f"Outcome Parquet SHA-256:  "
    f"{'PASS' if observed_outcome_sha256 == EXPECTED_OUTCOME_SHA256 else 'FAIL'}"
)
print(
    f"Construction QC SHA-256:  "
    f"{'PASS' if observed_qc_sha256 == EXPECTED_QC_SHA256 else 'FAIL'}"
)


# --------------------------------------------------------------------------------------------------
# 7. Load and verify policy and construction QC
# --------------------------------------------------------------------------------------------------

policy = json.loads(
    POLICY_PATH.read_text(
        encoding="utf-8"
    )
)

qc_report = json.loads(
    QC_PATH.read_text(
        encoding="utf-8"
    )
)

policy_identity_pass = (
    policy.get("policy_id")
    == "GES_STAGE5_PRIMARY_FUTURE_INSTABILITY_OUTCOME_POLICY"
)

policy_version_pass = (
    policy.get("version")
    == "1.0.0"
)

policy_status_pass = (
    policy.get("status")
    == "FROZEN_BEFORE_RECORD_LEVEL_OUTCOME_ASSIGNMENT"
)

qc_identity_pass = (
    qc_report.get("report_id")
    == "GES_STAGE5_PRIMARY_OUTCOME_CONSTRUCTION_QC"
)

qc_decision_pass = (
    qc_report.get("decision")
    == "PASS_RECORD_LEVEL_OUTCOMES_CONSTRUCTED_AND_VALIDATED"
)

qc_policy_sha_pass = (
    qc_report.get(
        "frozen_policy",
        {},
    ).get("sha256")
    == EXPECTED_POLICY_SHA256
)

qc_linkage_sha_pass = (
    qc_report.get(
        "source_linkage",
        {},
    ).get("sha256")
    == EXPECTED_LINKAGE_SHA256
)

qc_outcome_sha_pass = (
    qc_report.get(
        "outcome_artifact",
        {},
    ).get("sha256")
    == EXPECTED_OUTCOME_SHA256
)

qc_critical_checks_pass = bool(
    qc_report.get(
        "validation",
        {},
    ).get(
        "all_critical_checks_passed",
        False,
    )
)

qc_no_ges_loaded_pass = (
    qc_report.get(
        "scientific_boundary",
        {},
    ).get(
        "ges_scores_loaded"
    )
    is False
)

qc_no_performance_pass = (
    qc_report.get(
        "scientific_boundary",
        {},
    ).get(
        "temporal_performance_examined"
    )
    is False
)

print("\nPOLICY AND QC VERIFICATION")
print("-" * 124)
print(
    f"Policy identity:              "
    f"{'PASS' if policy_identity_pass else 'FAIL'}"
)
print(
    f"Policy version:               "
    f"{'PASS' if policy_version_pass else 'FAIL'}"
)
print(
    f"Policy frozen status:         "
    f"{'PASS' if policy_status_pass else 'FAIL'}"
)
print(
    f"QC identity:                  "
    f"{'PASS' if qc_identity_pass else 'FAIL'}"
)
print(
    f"QC construction decision:     "
    f"{'PASS' if qc_decision_pass else 'FAIL'}"
)
print(
    f"QC policy checksum lineage:   "
    f"{'PASS' if qc_policy_sha_pass else 'FAIL'}"
)
print(
    f"QC linkage checksum lineage:  "
    f"{'PASS' if qc_linkage_sha_pass else 'FAIL'}"
)
print(
    f"QC outcome checksum lineage:  "
    f"{'PASS' if qc_outcome_sha_pass else 'FAIL'}"
)
print(
    f"QC critical checks:           "
    f"{'PASS' if qc_critical_checks_pass else 'FAIL'}"
)
print(
    f"QC confirms no GES load:      "
    f"{'PASS' if qc_no_ges_loaded_pass else 'FAIL'}"
)
print(
    f"QC confirms no performance:   "
    f"{'PASS' if qc_no_performance_pass else 'FAIL'}"
)


# --------------------------------------------------------------------------------------------------
# 8. Verify outcome Parquet metadata and prohibit score fields
# --------------------------------------------------------------------------------------------------

outcome_parquet = pq.ParquetFile(
    OUTCOME_PATH
)

observed_outcome_rows = (
    outcome_parquet.metadata.num_rows
)

observed_outcome_columns = (
    outcome_parquet.metadata.num_columns
)

observed_row_groups = (
    outcome_parquet.metadata.num_row_groups
)

outcome_schema_columns = list(
    outcome_parquet.schema.names
)

PROHIBITED_COLUMN_TOKENS = [
    "p_stable",
    "weak_label",
    "ges_score",
    "ges_probability",
    "predicted_stability",
    "predicted_instability",
    "model_probability",
    "model_prediction",
]

prohibited_outcome_columns = [
    column
    for column in outcome_schema_columns
    if any(
        token in column.casefold()
        for token in PROHIBITED_COLUMN_TOKENS
    )
]

print("\nOUTCOME PARQUET STRUCTURE")
print("-" * 124)
print(
    f"Rows:                       "
    f"{observed_outcome_rows:,}"
)
print(
    f"Columns:                    "
    f"{observed_outcome_columns:,}"
)
print(
    f"Row groups:                 "
    f"{observed_row_groups:,}"
)
print(
    f"Prohibited score columns:   "
    f"{len(prohibited_outcome_columns):,}"
)

if prohibited_outcome_columns:
    for column in prohibited_outcome_columns:
        print(f"  - {column}")


# --------------------------------------------------------------------------------------------------
# 9. Load only the fields needed for independent outcome verification
# --------------------------------------------------------------------------------------------------

verification_columns = [
    "t0_rcv_accession",
    "target_gene",

    "primary_outcome_evaluable",
    "primary_future_instability",
    "primary_outcome_status",
    "primary_outcome_reason_code",

    "event_material_clinical_group_change",
    "event_new_unresolved_conflict_at_t1",
    "event_prior_conflict_resolved_to_material_group",

    "primary_event_component_count",
    "primary_event_combination",

    "outcome_policy_version",
    "outcome_policy_sha256",
    "record_level_outcome_assignment_created",
]

outcome = pd.read_parquet(
    OUTCOME_PATH,
    columns=verification_columns,
)


# --------------------------------------------------------------------------------------------------
# 10. Independently reproduce primary accounting
# --------------------------------------------------------------------------------------------------

duplicate_t0_keys = int(
    outcome[
        "t0_rcv_accession"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)

missing_t0_keys = int(
    outcome[
        "t0_rcv_accession"
    ]
    .isna()
    .sum()
)

evaluable_count = int(
    outcome[
        "primary_outcome_evaluable"
    ]
    .eq(True)
    .sum()
)

positive_count = int(
    outcome[
        "primary_future_instability"
    ]
    .eq(1)
    .sum()
)

negative_count = int(
    outcome[
        "primary_future_instability"
    ]
    .eq(0)
    .sum()
)

null_binary_count = int(
    outcome[
        "primary_future_instability"
    ]
    .isna()
    .sum()
)

status_counts = integer_value_counts(
    outcome[
        "primary_outcome_status"
    ]
)

event_combination_counts = {
    key: value
    for key, value in integer_value_counts(
        outcome.loc[
            outcome[
                "primary_event_combination"
            ].notna(),
            "primary_event_combination",
        ]
    ).items()
}


# --------------------------------------------------------------------------------------------------
# 11. Independently reproduce event-component counts and overlap
# --------------------------------------------------------------------------------------------------

component_columns = [
    "event_material_clinical_group_change",
    "event_new_unresolved_conflict_at_t1",
    "event_prior_conflict_resolved_to_material_group",
]

observed_component_counts = {
    column: int(
        outcome[column]
        .eq(True)
        .sum()
    )
    for column in component_columns
}

component_overlap_frame = (
    outcome.loc[
        outcome[
            "primary_event_combination"
        ].notna(),
        component_columns,
    ]
    .astype("boolean")
    .fillna(False)
    .astype("int8")
)

observed_component_overlap = {
    tuple(
        int(item)
        for item in index
    ): int(count)
    for index, count in (
        component_overlap_frame
        .value_counts(
            dropna=False
        )
        .items()
    )
}

reconstructed_component_count = (
    outcome[
        component_columns
    ]
    .astype("boolean")
    .fillna(False)
    .astype("int8")
    .sum(axis=1)
)

component_count_mismatches = int(
    (
        outcome[
            "primary_event_component_count"
        ]
        .notna()
        &
        outcome[
            "primary_event_component_count"
        ]
        .astype("Int64")
        .ne(
            reconstructed_component_count
            .astype("Int64")
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 12. Verify policy provenance embedded in every row
# --------------------------------------------------------------------------------------------------

observed_policy_versions = sorted(
    outcome[
        "outcome_policy_version"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

observed_policy_hashes = sorted(
    outcome[
        "outcome_policy_sha256"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

assignment_created_false_or_missing = int(
    (
        outcome[
            "record_level_outcome_assignment_created"
        ]
        .ne(True)
        .fillna(True)
    ).sum()
)

status_missing = int(
    outcome[
        "primary_outcome_status"
    ]
    .isna()
    .sum()
)

reason_missing = int(
    outcome[
        "primary_outcome_reason_code"
    ]
    .isna()
    .sum()
)


# --------------------------------------------------------------------------------------------------
# 13. Print independently reconstructed accounting
# --------------------------------------------------------------------------------------------------

print("\nINDEPENDENT OUTCOME ACCOUNTING")
print("-" * 124)
print(
    f"Complete records:                "
    f"{len(outcome):,}"
)
print(
    f"Evaluable records:               "
    f"{evaluable_count:,}"
)
print(
    f"Positive records:                "
    f"{positive_count:,}"
)
print(
    f"Negative records:                "
    f"{negative_count:,}"
)
print(
    f"Nonevaluable/censored records:   "
    f"{null_binary_count:,}"
)

if evaluable_count:
    print(
        f"Event prevalence:                "
        f"{positive_count / evaluable_count:.6%}"
    )

print("\nINDEPENDENT COMPONENT COUNTS")
print("-" * 124)

for column, observed_count in (
    observed_component_counts.items()
):
    print(
        f"{column:<58} "
        f"{observed_count:>7,}"
    )

print("\nINDEPENDENT STATUS DISTRIBUTION")
print("-" * 124)

for status, count in (
    status_counts.items()
):
    print(
        f"{status:<70} "
        f"{count:>7,}"
    )

print("\nINDEPENDENT EVENT-COMBINATION DISTRIBUTION")
print("-" * 124)

for combination, count in (
    event_combination_counts.items()
):
    print(
        f"{combination:<90} "
        f"{count:>7,}"
    )


# --------------------------------------------------------------------------------------------------
# 14. Critical freeze checks
# --------------------------------------------------------------------------------------------------

critical_checks = {
    "linkage checksum":
        observed_linkage_sha256
        == EXPECTED_LINKAGE_SHA256,

    "policy checksum":
        observed_policy_sha256
        == EXPECTED_POLICY_SHA256,

    "policy sidecar":
        policy_sidecar_text
        == expected_policy_sidecar_text,

    "outcome checksum":
        observed_outcome_sha256
        == EXPECTED_OUTCOME_SHA256,

    "QC checksum":
        observed_qc_sha256
        == EXPECTED_QC_SHA256,

    "policy identity":
        policy_identity_pass,

    "policy version":
        policy_version_pass,

    "policy frozen status":
        policy_status_pass,

    "QC identity":
        qc_identity_pass,

    "QC construction decision":
        qc_decision_pass,

    "QC policy lineage":
        qc_policy_sha_pass,

    "QC linkage lineage":
        qc_linkage_sha_pass,

    "QC outcome lineage":
        qc_outcome_sha_pass,

    "QC critical validation":
        qc_critical_checks_pass,

    "QC no GES load":
        qc_no_ges_loaded_pass,

    "QC no temporal performance":
        qc_no_performance_pass,

    "outcome row count":
        observed_outcome_rows
        == EXPECTED_OUTCOME_ROWS,

    "outcome column count":
        observed_outcome_columns
        == EXPECTED_OUTCOME_COLUMNS,

    "no prohibited score columns":
        len(prohibited_outcome_columns) == 0,

    "no missing T0 keys":
        missing_t0_keys == 0,

    "no duplicate T0 keys":
        duplicate_t0_keys == 0,

    "expected evaluable count":
        evaluable_count
        == EXPECTED_EVALUABLE,

    "expected positive count":
        positive_count
        == EXPECTED_POSITIVE,

    "expected negative count":
        negative_count
        == EXPECTED_NEGATIVE,

    "expected nonevaluable/censored count":
        null_binary_count
        == EXPECTED_NONEVALUABLE_OR_CENSORED,

    "binary/evaluable reconciliation":
        (
            positive_count
            + negative_count
        )
        == evaluable_count,

    "status distribution":
        status_counts
        == EXPECTED_STATUS_COUNTS,

    "component counts":
        observed_component_counts
        == EXPECTED_COMPONENT_COUNTS,

    "event-combination distribution":
        event_combination_counts
        == EXPECTED_EVENT_COMBINATION_COUNTS,

    "component overlap distribution":
        observed_component_overlap
        == EXPECTED_COMPONENT_OVERLAP,

    "component-count reconstruction":
        component_count_mismatches == 0,

    "policy version embedded in every row":
        observed_policy_versions
        == ["1.0.0"],

    "policy checksum embedded in every row":
        observed_policy_hashes
        == [EXPECTED_POLICY_SHA256],

    "assignment-created field true":
        assignment_created_false_or_missing == 0,

    "no missing outcome status":
        status_missing == 0,

    "no missing outcome reason":
        reason_missing == 0,
}

failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]

print("\nFREEZE VALIDATION")
print("-" * 124)

for check_name, passed in (
    critical_checks.items()
):
    print(
        f"{check_name:<68} "
        f"{'PASS' if passed else 'FAIL'}"
    )

if failed_checks:

    print("\nFAILED CHECKS")
    print("-" * 124)

    for failed_check in failed_checks:
        print(f" - {failed_check}")

    raise RuntimeError(
        "Stage 5 outcome freeze validation failed. "
        "No freeze manifest or new sidecars were written."
    )


# --------------------------------------------------------------------------------------------------
# 15. Create immutable outcome and QC SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

outcome_sidecar_status = (
    write_bytes_immutable(
        OUTCOME_SHA256_PATH,
        make_sha256_sidecar(
            OUTCOME_PATH,
            observed_outcome_sha256,
        ),
    )
)

qc_sidecar_status = (
    write_bytes_immutable(
        QC_SHA256_PATH,
        make_sha256_sidecar(
            QC_PATH,
            observed_qc_sha256,
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. Preserve timestamp on an identical safe rerun
# --------------------------------------------------------------------------------------------------

existing_manifest = None

if MANIFEST_PATH.exists():

    try:
        existing_manifest = json.loads(
            MANIFEST_PATH.read_text(
                encoding="utf-8"
            )
        )
    except Exception as error:
        raise RuntimeError(
            "An existing Stage 5 freeze manifest "
            "could not be parsed:\n"
            f"{MANIFEST_PATH}"
        ) from error

frozen_at_utc = (
    existing_manifest.get(
        "frozen_at_utc"
    )
    if existing_manifest
    else datetime.now(
        timezone.utc
    ).isoformat()
)


# --------------------------------------------------------------------------------------------------
# 17. Construct Stage 5 freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_manifest = {
    "manifest_id": (
        "GES_STAGE5_FUTURE_INSTABILITY_OUTCOME_FREEZE_MANIFEST"
    ),

    "version": "1.0.0",

    "status": (
        "STAGE5_OUTCOMES_ACCEPTED_AND_FROZEN"
    ),

    "frozen_at_utc": (
        frozen_at_utc
    ),

    "study_unit": (
        "RCV-level variant-condition aggregate"
    ),

    "frozen_dependencies": {
        "stage3h_final_linkage": {
            "path": str(
                LINKAGE_PATH
            ),

            "sha256": (
                observed_linkage_sha256
            ),
        },

        "stage5_outcome_policy": {
            "path": str(
                POLICY_PATH
            ),

            "sha256": (
                observed_policy_sha256
            ),

            "sha256_sidecar_path": str(
                POLICY_SHA256_PATH
            ),

            "version": (
                policy["version"]
            ),

            "status": (
                policy["status"]
            ),
        },
    },

    "frozen_stage5_artifacts": {
        "primary_future_instability_outcome_table": {
            "path": str(
                OUTCOME_PATH
            ),

            "sha256": (
                observed_outcome_sha256
            ),

            "sha256_sidecar_path": str(
                OUTCOME_SHA256_PATH
            ),

            "rows": (
                observed_outcome_rows
            ),

            "columns": (
                observed_outcome_columns
            ),

            "row_groups": (
                observed_row_groups
            ),
        },

        "outcome_construction_qc": {
            "path": str(
                QC_PATH
            ),

            "sha256": (
                observed_qc_sha256
            ),

            "sha256_sidecar_path": str(
                QC_SHA256_PATH
            ),

            "decision": (
                qc_report["decision"]
            ),
        },
    },

    "primary_outcome_accounting": {
        "total_t0_records": (
            EXPECTED_OUTCOME_ROWS
        ),

        "evaluable_records": (
            evaluable_count
        ),

        "positive_records": (
            positive_count
        ),

        "negative_records": (
            negative_count
        ),

        "nonevaluable_or_censored_records": (
            null_binary_count
        ),

        "event_prevalence_among_evaluable": (
            positive_count
            / evaluable_count
        ),

        "primary_outcome_status_distribution": (
            status_counts
        ),
    },

    "primary_event_components": {
        "material_clinical_group_change": (
            observed_component_counts[
                "event_material_clinical_group_change"
            ]
        ),

        "new_unresolved_conflict_at_t1": (
            observed_component_counts[
                "event_new_unresolved_conflict_at_t1"
            ]
        ),

        "prior_conflict_resolved_to_material_group": (
            observed_component_counts[
                "event_prior_conflict_resolved_to_material_group"
            ]
        ),

        "event_combination_distribution": (
            event_combination_counts
        ),
    },

    "scientific_boundary": {
        "outcome_policy_frozen_before_score_join": True,

        "stage4_ges_scores_loaded_during_outcome_construction": False,

        "stage4_ges_scores_loaded_during_outcome_freeze": False,

        "temporal_performance_examined": False,

        "outcome_rules_modified_after_policy_freeze": False,

        "unresolved_links_called_stable": False,

        "incompatible_axes_called_stable": False,

        "review_star_change_used_as_primary_event": False,
    },

    "freeze_validation": {
        "all_critical_checks_passed": True,

        "missing_t0_keys": (
            missing_t0_keys
        ),

        "duplicate_t0_keys": (
            duplicate_t0_keys
        ),

        "component_count_mismatches": (
            component_count_mismatches
        ),

        "missing_outcome_statuses": (
            status_missing
        ),

        "missing_outcome_reasons": (
            reason_missing
        ),

        "prohibited_score_columns": (
            prohibited_outcome_columns
        ),
    },

    "next_authorized_action": (
        "Begin locked temporal validation by verifying and "
        "joining the frozen Stage 4 T0 score table to this "
        "checksum-frozen Stage 5 outcome table. Outcome definitions "
        "and labels must not be changed after the score join."
    ),

    "serialization": (
        "UTF-8 deterministic JSON with sorted keys"
    ),
}


# --------------------------------------------------------------------------------------------------
# 18. Write and checksum-freeze the manifest
# --------------------------------------------------------------------------------------------------

manifest_bytes = canonical_json_bytes(
    freeze_manifest
)

manifest_sha256 = hashlib.sha256(
    manifest_bytes
).hexdigest()

manifest_write_status = (
    write_bytes_immutable(
        MANIFEST_PATH,
        manifest_bytes,
    )
)

manifest_sidecar_status = (
    write_bytes_immutable(
        MANIFEST_SHA256_PATH,
        make_sha256_sidecar(
            MANIFEST_PATH,
            manifest_sha256,
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 19. Readback verification
# --------------------------------------------------------------------------------------------------

manifest_readback_bytes = (
    MANIFEST_PATH.read_bytes()
)

manifest_readback_sha256 = (
    calculate_sha256(
        MANIFEST_PATH
    )
)

manifest_sidecar_text = (
    MANIFEST_SHA256_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

expected_manifest_sidecar_text = (
    f"{manifest_sha256}  "
    f"{MANIFEST_PATH.name}"
)

manifest_readback = json.loads(
    manifest_readback_bytes.decode(
        "utf-8"
    )
)

manifest_bytes_pass = (
    manifest_readback_bytes
    == manifest_bytes
)

manifest_checksum_pass = (
    manifest_readback_sha256
    == manifest_sha256
)

manifest_sidecar_pass = (
    manifest_sidecar_text
    == expected_manifest_sidecar_text
)

manifest_identity_pass = (
    manifest_readback.get(
        "manifest_id"
    )
    == (
        "GES_STAGE5_FUTURE_INSTABILITY_"
        "OUTCOME_FREEZE_MANIFEST"
    )
)

manifest_status_pass = (
    manifest_readback.get(
        "status"
    )
    == "STAGE5_OUTCOMES_ACCEPTED_AND_FROZEN"
)

if not all(
    [
        manifest_bytes_pass,
        manifest_checksum_pass,
        manifest_sidecar_pass,
        manifest_identity_pass,
        manifest_status_pass,
    ]
):
    raise RuntimeError(
        "Stage 5 freeze-manifest readback verification failed."
    )


# --------------------------------------------------------------------------------------------------
# 20. Final output
# --------------------------------------------------------------------------------------------------

print("\nFROZEN STAGE 5 ARTIFACTS")
print("-" * 124)

print(
    f"Outcome Parquet:     "
    f"{OUTCOME_PATH}"
)
print(
    f"  SHA-256:           "
    f"{observed_outcome_sha256}"
)
print(
    f"  Sidecar:           "
    f"{OUTCOME_SHA256_PATH}"
)
print(
    f"  Sidecar status:    "
    f"{outcome_sidecar_status}"
)

print()
print(
    f"Construction QC:     "
    f"{QC_PATH}"
)
print(
    f"  SHA-256:           "
    f"{observed_qc_sha256}"
)
print(
    f"  Sidecar:           "
    f"{QC_SHA256_PATH}"
)
print(
    f"  Sidecar status:    "
    f"{qc_sidecar_status}"
)

print()
print(
    f"Freeze manifest:     "
    f"{MANIFEST_PATH}"
)
print(
    f"  Write status:      "
    f"{manifest_write_status}"
)
print(
    f"  SHA-256:           "
    f"{manifest_sha256}"
)
print(
    f"  Sidecar:           "
    f"{MANIFEST_SHA256_PATH}"
)
print(
    f"  Sidecar status:    "
    f"{manifest_sidecar_status}"
)

print("\nMANIFEST READBACK")
print("-" * 124)
print(
    f"Exact byte readback: "
    f"{'PASS' if manifest_bytes_pass else 'FAIL'}"
)
print(
    f"SHA-256 verification: "
    f"{'PASS' if manifest_checksum_pass else 'FAIL'}"
)
print(
    f"SHA-256 sidecar: "
    f"{'PASS' if manifest_sidecar_pass else 'FAIL'}"
)
print(
    f"Manifest identity: "
    f"{'PASS' if manifest_identity_pass else 'FAIL'}"
)
print(
    f"Manifest status: "
    f"{'PASS' if manifest_status_pass else 'FAIL'}"
)

print("\n" + "=" * 124)
print("STAGE 5B STEP 3 DECISION")
print("=" * 124)

print(
    "PASS_STAGE5_FUTURE_INSTABILITY_OUTCOMES_ACCEPTED_AND_FROZEN"
)

print()
print(
    "Outcome policy frozen:                 YES"
)
print(
    "Record-level outcomes frozen:          YES"
)
print(
    "Outcome QC frozen:                     YES"
)
print(
    "Outcome freeze manifest created:       YES"
)
print(
    "All 71,659 T0 records accounted for:   YES"
)
print()
print(
    "GES scores loaded during Stage 5:      NO"
)
print(
    "Temporal performance examined:         NO"
)
print(
    "Stage 5 outcome definitions editable:  NO"
)
print()
print(
    "NEXT AUTHORIZED WORK:"
)
print(
    "Stage 6 — locked temporal validation using the frozen "
    "Stage 4 scores and the frozen Stage 5 outcomes."
)

STAGE 5B STEP 3 — FUTURE-INSTABILITY OUTCOME CHECKSUM FREEZE

CRYPTOGRAPHIC VERIFICATION
----------------------------------------------------------------------------------------------------------------------------
Stage 3H linkage SHA-256: PASS
Stage 5 policy SHA-256:   PASS
Policy SHA sidecar:       PASS
Outcome Parquet SHA-256:  PASS
Construction QC SHA-256:  PASS

POLICY AND QC VERIFICATION
----------------------------------------------------------------------------------------------------------------------------
Policy identity:              PASS
Policy version:               PASS
Policy frozen status:         PASS
QC identity:                  PASS
QC construction decision:     PASS
QC policy checksum lineage:   PASS
QC linkage checksum lineage:  PASS
QC outcome checksum lineage:  PASS
QC critical checks:           PASS
QC confirms no GES load:      PASS
QC confirms no performance:   PASS

OUTCOME PARQUET STRUCTURE
------------------------------------------------------------------